# <span style="color:PINK"> RELATIONSHIPS WHITH TARGET </span> #

## <span style="color:PINK">PACKAGES USED</span> ##

In [13]:
from pathlib import Path

import base64
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy import stats

from sklearn.metrics import (
    mutual_info_score,
    normalized_mutual_info_score
)
from IPython.display import display

import plotly.graph_objects as go

from sklearn.preprocessing import OneHotEncoder

## <span style="color:PINK"> BINARY ENCODING </span> ##

In [3]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "02_relationships_with_target"
)

ASSOCIATION_GROUP = (
    "target_association"
)

FEATURE_GROUP = (
    "binary_encoding"
)


GENDER_FEATURE = (
    "SEND_GENDER_BE"
)

YEAR_FEATURE = (
    "TRANS_YEAR_BE"
)

TARGET_FEATURE = (
    "TARGET_OMEGA"
)


BINARY_FEATURES = [
    GENDER_FEATURE,
    YEAR_FEATURE
]


REQUIRED_FEATURES = (
    BINARY_FEATURES
    +
    [
        TARGET_FEATURE
    ]
)


TARGET_POSITIVE_VALUE = 1

ALPHA = 0.05

STANDARDIZED_RESIDUAL_THRESHOLD = 2.0

PNG_DPI = 300


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / ASSOCIATION_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_binary_encoding_target_association.html"
)


GENDER_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "binary_target_send_gender_fraud_rate.png"
)


YEAR_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "binary_target_trans_year_fraud_rate.png"
)


FEATURE_PLOT_PATHS = {

    GENDER_FEATURE:
        GENDER_FRAUD_RATE_PATH,

    YEAR_FEATURE:
        YEAR_FRAUD_RATE_PATH
}


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD REQUIRED FEATURES
# ============================================================

dataset_binary_target = pd.read_parquet(
    DATASET_PATH,
    columns=REQUIRED_FEATURES
)


total_observations = int(
    len(
        dataset_binary_target
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. VALIDATE REQUIRED FEATURES
# ============================================================

missing_features = [
    feature
    for feature in REQUIRED_FEATURES
    if feature not in dataset_binary_target.columns
]


if missing_features:

    raise KeyError(
        "Missing required features: "
        + ", ".join(
            missing_features
        )
    )


# ============================================================
# 09. PREPARE COMMON COMPLETE SAMPLE
#
# All binary-target analyses use the same observations.
#
# The original parquet dataset is never modified.
# ============================================================

analysis_data = (
    dataset_binary_target[
        REQUIRED_FEATURES
    ]
    .dropna()
    .copy()
)


analysis_observations = int(
    len(
        analysis_data
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 10. VALIDATE TARGET
# ============================================================

target_values = (
    analysis_data[
        TARGET_FEATURE
    ]
    .drop_duplicates()
    .tolist()
)


if len(
    target_values
) != 2:

    raise ValueError(
        f"{TARGET_FEATURE} must contain exactly two classes. "
        f"Observed values: {target_values}"
    )


# ============================================================
# 11. IDENTIFY POSITIVE TARGET CLASS
#
# TARGET_POSITIVE_VALUE = 1 represents fraud.
#
# String and numeric representations are both supported.
# ============================================================

target_positive_value = None


for value in target_values:

    if (
        value == TARGET_POSITIVE_VALUE
        or
        str(
            value
        )
        == str(
            TARGET_POSITIVE_VALUE
        )
    ):

        target_positive_value = (
            value
        )

        break


if target_positive_value is None:

    raise ValueError(
        f"Fraud target value {TARGET_POSITIVE_VALUE} was not found. "
        f"Observed target values: {target_values}"
    )


target_negative_values = [
    value
    for value in target_values
    if value != target_positive_value
]


if len(
    target_negative_values
) != 1:

    raise ValueError(
        "Unable to determine the negative target class."
    )


target_negative_value = (
    target_negative_values[
        0
    ]
)


# ============================================================
# 12. VALIDATE BINARY FEATURES
# ============================================================

binary_feature_overview_records = []


for feature in BINARY_FEATURES:

    feature_values = (
        analysis_data[
            feature
        ]
        .drop_duplicates()
        .tolist()
    )


    if len(
        feature_values
    ) != 2:

        raise ValueError(
            f"{feature} must contain exactly two categories. "
            f"Observed values: {feature_values}"
        )


    binary_feature_overview_records.append({

        "FEATURE":
            feature,

        "DATA_TYPE":
            str(
                analysis_data[
                    feature
                ].dtype
            ),

        "CATEGORY_1":
            str(
                feature_values[
                    0
                ]
            ),

        "CATEGORY_2":
            str(
                feature_values[
                    1
                ]
            ),

        "UNIQUE_CATEGORIES":
            int(
                len(
                    feature_values
                )
            )
    })


binary_feature_overview_table = pd.DataFrame(
    binary_feature_overview_records
)


# ============================================================
# 13. TARGET CLASS OVERVIEW
# ============================================================

target_class_counts = (
    analysis_data[
        TARGET_FEATURE
    ]
    .value_counts()
    .reindex(
        [
            target_negative_value,
            target_positive_value
        ],
        fill_value=0
    )
)


target_class_percentages = (
    target_class_counts
    / analysis_observations
    * 100
)


target_overview_table = pd.DataFrame({

    "TARGET_VALUE": [
        target_negative_value,
        target_positive_value
    ],

    "TARGET_INTERPRETATION": [
        "Non-fraud",
        "Fraud"
    ],

    "COUNT": [
        int(
            target_class_counts.loc[
                target_negative_value
            ]
        ),

        int(
            target_class_counts.loc[
                target_positive_value
            ]
        )
    ],

    "PERCENTAGE": [
        float(
            target_class_percentages.loc[
                target_negative_value
            ]
        ),

        float(
            target_class_percentages.loc[
                target_positive_value
            ]
        )
    ]
})


overall_fraud_rate = float(
    (
        analysis_data[
            TARGET_FEATURE
        ]
        == target_positive_value
    )
    .mean()
)


overall_fraud_percentage = (
    overall_fraud_rate
    * 100
)


# ============================================================
# 14. ASSOCIATION STRENGTH INTERPRETATION
#
# Descriptive exploratory guidelines only.
# ============================================================

def interpret_association_strength(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        return (
            "Weak"
        )


    elif absolute_value < 0.50:

        return (
            "Moderate"
        )


    elif absolute_value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


# ============================================================
# 15. CATEGORY SORTING FUNCTION
# ============================================================

def sort_binary_categories(
    values
):

    try:

        return sorted(
            values
        )


    except TypeError:

        return sorted(
            values,
            key=lambda value:
                str(
                    value
                )
        )


# ============================================================
# 16. IMAGE TO BASE64 FUNCTION
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 17. STORAGE FOR FEATURE RESULTS
# ============================================================

feature_results = {}

association_summary_records = []


# ============================================================
# 18. ANALYZE EACH BINARY FEATURE AGAINST TARGET
# ============================================================

for feature in BINARY_FEATURES:

    # --------------------------------------------------------
    # CATEGORY ORDER
    #
    # First category:
    # reference category = 0
    #
    # Second category:
    # exposure/comparison category = 1
    #
    # Examples:
    #
    # F -> 0
    # M -> 1
    #
    # 2019 -> 0
    # 2020 -> 1
    # --------------------------------------------------------

    feature_categories = sort_binary_categories(
        analysis_data[
            feature
        ]
        .drop_duplicates()
        .tolist()
    )


    reference_category = (
        feature_categories[
            0
        ]
    )


    exposure_category = (
        feature_categories[
            1
        ]
    )


    # --------------------------------------------------------
    # CONTINGENCY TABLE
    # --------------------------------------------------------

    contingency_table = pd.crosstab(

        analysis_data[
            feature
        ],

        analysis_data[
            TARGET_FEATURE
        ]
    )


    contingency_table = (
        contingency_table
        .reindex(
            index=[
                reference_category,
                exposure_category
            ],
            columns=[
                target_negative_value,
                target_positive_value
            ],
            fill_value=0
        )
    )


    contingency_table.columns = [
        "NON_FRAUD",
        "FRAUD"
    ]


    # --------------------------------------------------------
    # FRAUD RATE BY CATEGORY
    # --------------------------------------------------------

    category_totals = (
        contingency_table
        .sum(
            axis=1
        )
    )


    fraud_rates = (
        contingency_table[
            "FRAUD"
        ]
        /
        category_totals
    )


    fraud_rate_table = pd.DataFrame({

        "CATEGORY":
            contingency_table.index,

        "TOTAL":
            category_totals.astype(
                int
            ).to_numpy(),

        "NON_FRAUD":
            contingency_table[
                "NON_FRAUD"
            ]
            .astype(
                int
            )
            .to_numpy(),

        "FRAUD":
            contingency_table[
                "FRAUD"
            ]
            .astype(
                int
            )
            .to_numpy(),

        "FRAUD_RATE":
            fraud_rates.to_numpy(),

        "FRAUD_PERCENTAGE":
            (
                fraud_rates
                * 100
            )
            .to_numpy()
    })


    # --------------------------------------------------------
    # CHI-SQUARE TEST
    #
    # H0:
    # Feature and TARGET_OMEGA are independent.
    #
    # H1:
    # An association exists.
    # --------------------------------------------------------

    observed_matrix = (
        contingency_table
        .to_numpy(
            dtype="float64"
        )
    )


    (
        chi_square_statistic,
        chi_square_p_value,
        chi_square_degrees_of_freedom,
        expected_matrix
    ) = stats.chi2_contingency(
        observed_matrix,
        correction=False
    )


    chi_square_statistic = float(
        chi_square_statistic
    )


    chi_square_p_value = float(
        chi_square_p_value
    )


    chi_square_degrees_of_freedom = int(
        chi_square_degrees_of_freedom
    )


    expected_matrix = np.asarray(
        expected_matrix,
        dtype="float64"
    )


    # --------------------------------------------------------
    # EXPECTED FREQUENCY DIAGNOSTICS
    # --------------------------------------------------------

    minimum_expected_frequency = float(
        np.min(
            expected_matrix
        )
    )


    cells_expected_below_5 = int(
        np.sum(
            expected_matrix < 5
        )
    )


    # --------------------------------------------------------
    # TEMPORARY BINARY CODING FOR SIGNED PHI
    #
    # Reference category = 0
    # Exposure category  = 1
    #
    # Non-fraud = 0
    # Fraud     = 1
    #
    # This transformation exists only in memory.
    # --------------------------------------------------------

    feature_binary = (
        analysis_data[
            feature
        ]
        .map({

            reference_category:
                0,

            exposure_category:
                1
        })
        .to_numpy(
            dtype="float64"
        )
    )


    target_binary = (
        analysis_data[
            TARGET_FEATURE
        ]
        .map({

            target_negative_value:
                0,

            target_positive_value:
                1
        })
        .to_numpy(
            dtype="float64"
        )
    )


    phi_coefficient = float(
        np.corrcoef(
            feature_binary,
            target_binary
        )[
            0,
            1
        ]
    )


    phi_strength = (
        interpret_association_strength(
            phi_coefficient
        )
    )


    # --------------------------------------------------------
    # CRAMER'S V
    #
    # For a 2 × 2 table using the same uncorrected
    # chi-square statistic:
    #
    # Cramer's V = |Phi|
    # --------------------------------------------------------

    number_rows = int(
        observed_matrix.shape[
            0
        ]
    )


    number_columns = int(
        observed_matrix.shape[
            1
        ]
    )


    cramers_dimension = min(
        number_rows - 1,
        number_columns - 1
    )


    cramers_v = float(
        np.sqrt(
            chi_square_statistic
            /
            (
                analysis_observations
                *
                cramers_dimension
            )
        )
    )


    cramers_v_strength = (
        interpret_association_strength(
            cramers_v
        )
    )


    phi_cramers_difference = abs(
        abs(
            phi_coefficient
        )
        -
        cramers_v
    )


    # --------------------------------------------------------
    # 2 × 2 CELL DEFINITIONS
    #
    #                  Non-fraud    Fraud
    #
    # Reference             d          c
    # Exposure              b          a
    #
    # OR = (a*d) / (b*c)
    # --------------------------------------------------------

    d = float(
        contingency_table.loc[
            reference_category,
            "NON_FRAUD"
        ]
    )


    c = float(
        contingency_table.loc[
            reference_category,
            "FRAUD"
        ]
    )


    b = float(
        contingency_table.loc[
            exposure_category,
            "NON_FRAUD"
        ]
    )


    a = float(
        contingency_table.loc[
            exposure_category,
            "FRAUD"
        ]
    )


    # --------------------------------------------------------
    # HALDANE-ANSCOMBE CORRECTION
    #
    # Add 0.5 to all four cells only if at least
    # one cell is zero.
    # --------------------------------------------------------

    zero_cell_correction_used = bool(
        min(
            a,
            b,
            c,
            d
        )
        == 0
    )


    if zero_cell_correction_used:

        a_or = (
            a
            + 0.5
        )

        b_or = (
            b
            + 0.5
        )

        c_or = (
            c
            + 0.5
        )

        d_or = (
            d
            + 0.5
        )


    else:

        a_or = a
        b_or = b
        c_or = c
        d_or = d


    # --------------------------------------------------------
    # ODDS RATIO
    # --------------------------------------------------------

    odds_ratio = float(
        (
            a_or
            * d_or
        )
        /
        (
            b_or
            * c_or
        )
    )


    log_odds_ratio_standard_error = float(
        np.sqrt(
            1.0
            / a_or

            +

            1.0
            / b_or

            +

            1.0
            / c_or

            +

            1.0
            / d_or
        )
    )


    odds_ratio_ci_lower = float(
        np.exp(
            np.log(
                odds_ratio
            )
            -
            1.96
            * log_odds_ratio_standard_error
        )
    )


    odds_ratio_ci_upper = float(
        np.exp(
            np.log(
                odds_ratio
            )
            +
            1.96
            * log_odds_ratio_standard_error
        )
    )


    # --------------------------------------------------------
    # FRAUD RISK
    # --------------------------------------------------------

    reference_fraud_risk = (
        c
        /
        (
            c
            + d
        )
    )


    exposure_fraud_risk = (
        a
        /
        (
            a
            + b
        )
    )


    if reference_fraud_risk > 0:

        risk_ratio = float(
            exposure_fraud_risk
            /
            reference_fraud_risk
        )


    else:

        risk_ratio = np.nan


    risk_difference = float(
        exposure_fraud_risk
        -
        reference_fraud_risk
    )


    risk_difference_percentage_points = (
        risk_difference
        * 100
    )


    # --------------------------------------------------------
    # ODDS RATIO INTERPRETATION
    # --------------------------------------------------------

    if odds_ratio > 1:

        odds_ratio_direction = (
            f"{exposure_category} has higher fraud odds "
            f"than {reference_category}."
        )


    elif odds_ratio < 1:

        odds_ratio_direction = (
            f"{exposure_category} has lower fraud odds "
            f"than {reference_category}."
        )


    else:

        odds_ratio_direction = (
            f"{exposure_category} and {reference_category} "
            f"have equal estimated fraud odds."
        )


    # --------------------------------------------------------
    # ADJUSTED STANDARDIZED RESIDUALS
    # --------------------------------------------------------

    row_totals = (
        observed_matrix
        .sum(
            axis=1,
            keepdims=True
        )
    )


    column_totals = (
        observed_matrix
        .sum(
            axis=0,
            keepdims=True
        )
    )


    grand_total = float(
        observed_matrix.sum()
    )


    row_proportions = (
        row_totals
        /
        grand_total
    )


    column_proportions = (
        column_totals
        /
        grand_total
    )


    residual_denominator = np.sqrt(

        expected_matrix

        *

        (
            1
            - row_proportions
        )

        *

        (
            1
            - column_proportions
        )
    )


    adjusted_residual_matrix = np.divide(

        observed_matrix
        - expected_matrix,

        residual_denominator,

        out=np.full_like(
            observed_matrix,
            np.nan,
            dtype="float64"
        ),

        where=(
            residual_denominator > 0
        )
    )


    adjusted_residuals_table = pd.DataFrame(

        adjusted_residual_matrix,

        index=[
            reference_category,
            exposure_category
        ],

        columns=[
            "NON_FRAUD",
            "FRAUD"
        ]
    )


    flagged_residual_cells = int(
        np.sum(
            np.abs(
                adjusted_residual_matrix
            )
            >= STANDARDIZED_RESIDUAL_THRESHOLD
        )
    )


    # --------------------------------------------------------
    # CHI-SQUARE DECISION
    # --------------------------------------------------------

    if chi_square_p_value < ALPHA:

        chi_square_decision = (
            "Reject the null hypothesis"
        )


        chi_square_interpretation = (
            f"Statistical evidence of association exists "
            f"between {feature} and {TARGET_FEATURE}."
        )


    else:

        chi_square_decision = (
            "Do not reject the null hypothesis"
        )


        chi_square_interpretation = (
            f"No sufficient statistical evidence of "
            f"association was identified between {feature} "
            f"and {TARGET_FEATURE}."
        )


    # --------------------------------------------------------
    # FRAUD RATE PLOT
    # --------------------------------------------------------

    plot_path = (
        FEATURE_PLOT_PATHS[
            feature
        ]
    )


    fig, ax = plt.subplots(
        figsize=(
            9,
            6
        )
    )


    ax.bar(
        fraud_rate_table[
            "CATEGORY"
        ]
        .astype(
            str
        ),
        fraud_rate_table[
            "FRAUD_PERCENTAGE"
        ]
    )


    ax.axhline(
        overall_fraud_percentage,
        linestyle="--",
        linewidth=1.5,
        label=(
            "Overall fraud rate"
        )
    )


    ax.set_xlabel(
        feature
    )


    ax.set_ylabel(
        "Fraud rate (%)"
    )


    ax.set_title(
        f"Fraud rate by {feature}"
    )


    ax.legend()


    ax.grid(
        axis="y",
        alpha=0.20
    )


    fig.tight_layout()


    fig.savefig(
        plot_path,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    # --------------------------------------------------------
    # SAVE ALL RESULTS FOR HTML AND NOTEBOOK OUTPUT
    # --------------------------------------------------------

    feature_results[
        feature
    ] = {

        "categories":
            feature_categories,

        "reference_category":
            reference_category,

        "exposure_category":
            exposure_category,

        "contingency_table":
            contingency_table,

        "fraud_rate_table":
            fraud_rate_table,

        "chi_square_statistic":
            chi_square_statistic,

        "chi_square_p_value":
            chi_square_p_value,

        "chi_square_degrees_of_freedom":
            chi_square_degrees_of_freedom,

        "chi_square_decision":
            chi_square_decision,

        "chi_square_interpretation":
            chi_square_interpretation,

        "minimum_expected_frequency":
            minimum_expected_frequency,

        "cells_expected_below_5":
            cells_expected_below_5,

        "phi_coefficient":
            phi_coefficient,

        "phi_strength":
            phi_strength,

        "cramers_v":
            cramers_v,

        "cramers_v_strength":
            cramers_v_strength,

        "phi_cramers_difference":
            phi_cramers_difference,

        "odds_ratio":
            odds_ratio,

        "odds_ratio_ci_lower":
            odds_ratio_ci_lower,

        "odds_ratio_ci_upper":
            odds_ratio_ci_upper,

        "odds_ratio_direction":
            odds_ratio_direction,

        "zero_cell_correction_used":
            zero_cell_correction_used,

        "reference_fraud_risk":
            reference_fraud_risk,

        "exposure_fraud_risk":
            exposure_fraud_risk,

        "risk_ratio":
            risk_ratio,

        "risk_difference":
            risk_difference,

        "risk_difference_percentage_points":
            risk_difference_percentage_points,

        "adjusted_residuals_table":
            adjusted_residuals_table,

        "flagged_residual_cells":
            flagged_residual_cells,

        "plot_path":
            plot_path
    }


    # --------------------------------------------------------
    # GLOBAL ASSOCIATION SUMMARY
    # --------------------------------------------------------

    association_summary_records.append({

        "FEATURE":
            feature,

        "REFERENCE_CATEGORY":
            str(
                reference_category
            ),

        "EXPOSURE_CATEGORY":
            str(
                exposure_category
            ),

        "PHI":
            phi_coefficient,

        "PHI_STRENGTH":
            phi_strength,

        "CRAMERS_V":
            cramers_v,

        "CRAMERS_V_STRENGTH":
            cramers_v_strength,

        "CHI_SQUARE_P_VALUE":
            chi_square_p_value,

        "ODDS_RATIO":
            odds_ratio,

        "OR_CI_95_LOWER":
            odds_ratio_ci_lower,

        "OR_CI_95_UPPER":
            odds_ratio_ci_upper,

        "RISK_RATIO":
            risk_ratio,

        "RISK_DIFFERENCE_PERCENTAGE_POINTS":
            risk_difference_percentage_points
    })


# ============================================================
# 19. GLOBAL ASSOCIATION SUMMARY TABLE
# ============================================================

association_summary_table = pd.DataFrame(
    association_summary_records
)


association_summary_table = (
    association_summary_table
    .sort_values(
        by="CRAMERS_V",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 20. PREPARE TARGET OVERVIEW HTML
# ============================================================

target_overview_html = (
    target_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PERCENTAGE":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


binary_feature_overview_html = (
    binary_feature_overview_table
    .to_html(
        index=False,
        border=0
    )
)


association_summary_html = (
    association_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PHI":
                lambda value:
                    f"{value:.6f}",

            "CRAMERS_V":
                lambda value:
                    f"{value:.6f}",

            "CHI_SQUARE_P_VALUE":
                lambda value:
                    f"{value:.12g}",

            "ODDS_RATIO":
                lambda value:
                    f"{value:.6f}",

            "OR_CI_95_LOWER":
                lambda value:
                    f"{value:.6f}",

            "OR_CI_95_UPPER":
                lambda value:
                    f"{value:.6f}",

            "RISK_RATIO":
                lambda value:
                    (
                        "NaN"
                        if pd.isna(
                            value
                        )
                        else f"{value:.6f}"
                    ),

            "RISK_DIFFERENCE_PERCENTAGE_POINTS":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


# ============================================================
# 21. PREPARE FEATURE-SPECIFIC HTML
# ============================================================

feature_html_sections = []


for section_number, feature in enumerate(
    BINARY_FEATURES,
    start=3
):

    result = (
        feature_results[
            feature
        ]
    )


    contingency_html = (
        result[
            "contingency_table"
        ]
        .to_html(
            border=0
        )
    )


    fraud_rate_html = (
        result[
            "fraud_rate_table"
        ]
        .to_html(
            index=False,
            border=0,
            formatters={

                "FRAUD_RATE":
                    lambda value:
                        f"{value:.8f}",

                "FRAUD_PERCENTAGE":
                    lambda value:
                        f"{value:.6f}"
            }
        )
    )


    residuals_html = (
        result[
            "adjusted_residuals_table"
        ]
        .to_html(
            border=0,
            float_format=lambda value:
                f"{value:.6f}"
        )
    )


    plot_base64 = (
        image_to_base64(
            result[
                "plot_path"
            ]
        )
    )


    if result[
        "zero_cell_correction_used"
    ]:

        zero_correction_note = """

        <p>

        At least one cell contained zero observations.
        A Haldane-Anscombe correction of 0.5 was added
        to all four cells for Odds Ratio estimation.

        </p>

        """


    else:

        zero_correction_note = """

        <p>

        No zero-cell correction was required for
        Odds Ratio estimation.

        </p>

        """


    feature_html = f"""

    <h2>
    {section_number}. {feature} × {TARGET_FEATURE}
    </h2>


    <h3>
    Contingency table
    </h3>


    <div class="table-container">

    {contingency_html}

    </div>


    <h3>
    Fraud rate by category
    </h3>


    <div class="table-container">

    {fraud_rate_html}

    </div>


    <div class="chart">

    <img
        src="data:image/png;base64,{plot_base64}"
        alt="Fraud rate by {feature}"
    >

    </div>


    <div class="note">

    The dashed horizontal line represents the overall
    fraud rate of the complete analysis sample:

    <strong>
    {overall_fraud_percentage:.6f}%
    </strong>.

    </div>


    <h3>
    Chi-square test of independence
    </h3>


    <p>

    <strong>Null hypothesis:</strong>

    {feature} and {TARGET_FEATURE} are statistically
    independent.

    </p>


    <p>

    <strong>Alternative hypothesis:</strong>

    An association exists between {feature} and
    {TARGET_FEATURE}.

    </p>


    <p class="result">

    Chi-square statistic:
    {result["chi_square_statistic"]:.6f}

    <br>

    Degrees of freedom:
    {result["chi_square_degrees_of_freedom"]}

    <br>

    p-value:
    {result["chi_square_p_value"]:.12g}

    <br>

    Decision:
    {result["chi_square_decision"]}

    </p>


    <div class="note">

    {result["chi_square_interpretation"]}

    <br><br>

    Minimum expected frequency:

    <strong>
    {result["minimum_expected_frequency"]:.6f}
    </strong>

    <br>

    Cells with expected frequency below 5:

    <strong>
    {result["cells_expected_below_5"]}
    </strong>

    <br><br>

    Because the dataset contains a very large number of
    observations, statistical significance should not be
    interpreted without an effect-size measure.

    </div>


    <h3>
    Phi coefficient
    </h3>


    <p class="result">

    Phi:
    {result["phi_coefficient"]:.6f}

    <br>

    Descriptive strength:
    {result["phi_strength"]}

    </p>


    <div class="note">

    Temporary coding used only to determine the direction
    of Phi:

    <br><br>

    {result["reference_category"]} = 0

    <br>

    {result["exposure_category"]} = 1

    <br>

    Non-fraud = 0

    <br>

    Fraud = 1

    <br><br>

    A positive Phi indicates that the exposure category
    tends to occur with fraud more often relative to the
    reference category.

    A negative Phi indicates the opposite direction.

    The sign depends on the temporary coding orientation.

    </div>


    <h3>
    Cramer's V
    </h3>


    <p class="result">

    Cramer's V:
    {result["cramers_v"]:.6f}

    <br>

    Descriptive strength:
    {result["cramers_v_strength"]}

    </p>


    <div class="note">

    For a 2 × 2 table using the same uncorrected
    chi-square statistic:

    <br><br>

    <strong>
    Cramer's V = |Phi|
    </strong>

    <br><br>

    Observed absolute difference:

    <strong>
    {result["phi_cramers_difference"]:.12f}
    </strong>

    </div>


    <h3>
    Odds Ratio
    </h3>


    <p>

    Comparison:

    <strong>
    {result["exposure_category"]}
    versus
    {result["reference_category"]}
    </strong>

    </p>


    <p class="result">

    Odds Ratio:
    {result["odds_ratio"]:.6f}

    <br>

    95% CI:
    [
    {result["odds_ratio_ci_lower"]:.6f},
    {result["odds_ratio_ci_upper"]:.6f}
    ]

    </p>


    <div class="note">

    {result["odds_ratio_direction"]}

    <br><br>

    An Odds Ratio equal to 1 indicates equal estimated
    odds of fraud.

    Values above 1 indicate higher fraud odds in the
    exposure category.

    Values below 1 indicate lower fraud odds in the
    exposure category.

    </div>


    {zero_correction_note}


    <h3>
    Relative and absolute fraud risk
    </h3>


    <p class="result">

    Fraud risk in reference category
    ({result["reference_category"]}):

    {result["reference_fraud_risk"] * 100:.6f}%

    <br>

    Fraud risk in exposure category
    ({result["exposure_category"]}):

    {result["exposure_fraud_risk"] * 100:.6f}%

    <br>

    Risk Ratio:

    {
        "NaN"
        if pd.isna(
            result["risk_ratio"]
        )
        else f'{result["risk_ratio"]:.6f}'
    }

    <br>

    Risk difference:

    {result["risk_difference_percentage_points"]:.6f}
    percentage points

    </p>


    <h3>
    Adjusted standardized residuals
    </h3>


    <div class="table-container">

    {residuals_html}

    </div>


    <div class="note">

    Positive residuals indicate combinations occurring
    more frequently than expected under independence.

    Negative residuals indicate combinations occurring
    less frequently than expected.

    <br><br>

    The exploratory reference is:

    <strong>
    |adjusted standardized residual| >=
    {STANDARDIZED_RESIDUAL_THRESHOLD:.1f}
    </strong>.

    <br><br>

    Flagged cells:

    <strong>
    {result["flagged_residual_cells"]}
    </strong>.

    This criterion is descriptive only and is not used
    to remove observations.

    </div>

    """


    feature_html_sections.append(
        feature_html
    )


feature_html_content = "\n".join(
    feature_html_sections
)


# ============================================================
# 22. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Binary Encoding - Target Association
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1500px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 13px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 8px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Target Association —
Binary Encoding
</h1>


<p>

Binary explanatory features:

</p>


<ul>

<li>{GENDER_FEATURE}</li>
<li>{YEAR_FEATURE}</li>

</ul>


<p>

Target:

<strong>
{TARGET_FEATURE}
</strong>

</p>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. TARGET OVERVIEW
========================================================= -->


<h2>
1. Target overview
</h2>


<div class="table-container">

{target_overview_html}

</div>


<p class="result">

Overall fraud rate:
{overall_fraud_percentage:.6f}%

</p>


<div class="note">

TARGET_OMEGA is used only as an outcome variable for
exploratory association analysis.

It must not be included as an explanatory input variable
when constructing PCA, t-SNE or GMM representations.

</div>


<!-- ========================================================
     2. BINARY FEATURE OVERVIEW
========================================================= -->


<h2>
2. Binary feature overview
</h2>


<div class="table-container">

{binary_feature_overview_html}

</div>


<div class="note">

The original values are preserved in the parquet dataset.

Temporary binary coding is used only when required to
calculate signed measures such as Phi.

No permanent transformation is performed during this
exploratory analysis.

</div>


{feature_html_content}


<!-- ========================================================
     5. COMPARATIVE SUMMARY
========================================================= -->


<h2>
5. Comparative target-association summary
</h2>


<div class="table-container">

{association_summary_html}

</div>


<div class="note">

Cramer's V and the absolute value of Phi summarize the
magnitude of binary association with fraud.

Odds Ratio, Risk Ratio and Risk Difference provide
additional information about the direction and practical
magnitude of fraud occurrence between categories.

<br><br>

Because this dataset is very large and fraud is expected
to be substantially less frequent than non-fraud,
p-values should not be interpreted as measures of
practical importance.

Effect sizes and class-specific fraud rates are therefore
essential for interpretation.

</div>


<!-- ========================================================
     6. MODELING IMPLICATIONS
========================================================= -->


<h2>
6. Potential modeling implications
</h2>


<div class="note">

<strong>Target exclusion:</strong>

<br><br>

TARGET_OMEGA must remain outside the feature matrix used
to construct PCA, t-SNE and GMM.

It may later be used to interpret or externally evaluate
the unsupervised structure obtained from those methods.

</div>


<div class="note">

<strong>Binary feature association:</strong>

<br><br>

A statistically significant relationship with fraud does
not automatically justify removing, transforming or
prioritizing a feature before unsupervised modeling.

The evidence identified here should be documented and
reviewed during the final pre-modeling audit.

</div>


<div class="note">

<strong>Large sample size:</strong>

<br><br>

With approximately {analysis_observations:,} complete
observations, very small effects can become statistically
significant.

Therefore, Phi, Cramer's V, Odds Ratio, fraud-rate
differences and other effect measures should receive
greater interpretive emphasis than the p-value alone.

</div>


<!-- ========================================================
     7. SUMMARY
========================================================= -->


<h2>
7. Summary
</h2>


<p class="result">

Binary features analyzed:
{len(BINARY_FEATURES)}

</p>


<p class="result">

Target:
{TARGET_FEATURE}

</p>


<p class="result">

Overall fraud rate:
{overall_fraud_percentage:.6f}%

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The binary target-association analysis compares fraud and
non-fraud behavior across SEND_GENDER_BE and TRANS_YEAR_BE.

<br><br>

Chi-square tests evaluate statistical independence.

Signed Phi coefficients indicate association direction
under explicitly documented temporary coding.

Cramer's V describes association magnitude independently
of coding direction.

<br><br>

Odds Ratios compare fraud odds between the two categories
of each feature.

Risk Ratios and Risk Differences provide complementary
measures based directly on fraud probabilities.

Adjusted standardized residuals identify the individual
cells that differ most from independence expectations.

<br><br>

No feature is removed or transformed during this EDA.

All findings should be revisited during the final
pre-modeling audit before PCA, t-SNE and GMM.

</div>


</body>

</html>
"""


# ============================================================
# 23. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 24. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "BINARY ENCODING - TARGET ASSOCIATION"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 25. DISPLAY TARGET OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TARGET OVERVIEW"
)


print(
    "=" * 100
)


display(
    target_overview_table
)


print(
    "\nOverall fraud rate:",
    f"{overall_fraud_percentage:.6f}%"
)


# ============================================================
# 26. DISPLAY BINARY FEATURE OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "BINARY FEATURE OVERVIEW"
)


print(
    "=" * 100
)


display(
    binary_feature_overview_table
)


# ============================================================
# 27. DISPLAY FEATURE-SPECIFIC RESULTS
# ============================================================

for feature in BINARY_FEATURES:

    result = (
        feature_results[
            feature
        ]
    )


    print(
        "\n"
        + "=" * 100
    )


    print(
        f"{feature} × {TARGET_FEATURE}"
    )


    print(
        "=" * 100
    )


    print(
        "\nContingency table:"
    )


    display(
        result[
            "contingency_table"
        ]
    )


    print(
        "\nFraud rate by category:"
    )


    display(
        result[
            "fraud_rate_table"
        ]
    )


    print(
        "\nChi-square statistic:",
        f'{result["chi_square_statistic"]:.6f}'
    )


    print(
        "Degrees of freedom:",
        result[
            "chi_square_degrees_of_freedom"
        ]
    )


    print(
        "p-value:",
        f'{result["chi_square_p_value"]:.12g}'
    )


    print(
        "Decision:",
        result[
            "chi_square_decision"
        ]
    )


    print(
        "\nPhi:",
        f'{result["phi_coefficient"]:.6f}'
    )


    print(
        "Phi strength:",
        result[
            "phi_strength"
        ]
    )


    print(
        "\nCramer's V:",
        f'{result["cramers_v"]:.6f}'
    )


    print(
        "Cramer's V strength:",
        result[
            "cramers_v_strength"
        ]
    )


    print(
        "\nOdds Ratio:",
        f'{result["odds_ratio"]:.6f}'
    )


    print(
        "95% CI:",
        (
            f'[{result["odds_ratio_ci_lower"]:.6f}, '
            f'{result["odds_ratio_ci_upper"]:.6f}]'
        )
    )


    print(
        "Risk Ratio:",
        (
            "NaN"
            if pd.isna(
                result[
                    "risk_ratio"
                ]
            )
            else f'{result["risk_ratio"]:.6f}'
        )
    )


    print(
        "Risk difference:",
        (
            f'{result["risk_difference_percentage_points"]:.6f} '
            "percentage points"
        )
    )


    print(
        "\nAdjusted standardized residuals:"
    )


    display(
        result[
            "adjusted_residuals_table"
        ]
    )


# ============================================================
# 28. DISPLAY COMPARATIVE SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "COMPARATIVE TARGET-ASSOCIATION SUMMARY"
)


print(
    "=" * 100
)


display(
    association_summary_table
)


# ============================================================
# 29. RELEASE MEMORY
# ============================================================

del dataset_binary_target
del analysis_data

gc.collect()


# ============================================================
# 30. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    GENDER_FRAUD_RATE_PATH
)


print(
    YEAR_FRAUD_RATE_PATH
)


BINARY ENCODING - TARGET ASSOCIATION

Total dataset observations: 1852394
Complete observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

TARGET OVERVIEW


,TARGET_VALUE,TARGET_INTERPRETATION,COUNT,PERCENTAGE
0,0,Non-fraud,1842743,99.478999
1,1,Fraud,9651,0.521001



Overall fraud rate: 0.521001%

BINARY FEATURE OVERVIEW


,FEATURE,DATA_TYPE,CATEGORY_1,CATEGORY_2,UNIQUE_CATEGORIES
0,SEND_GENDER_BE,category,F,M,2
1,TRANS_YEAR_BE,int16,2019,2020,2



SEND_GENDER_BE × TARGET_OMEGA

Contingency table:


...


Chi-square statistic: 67.168707
Degrees of freedom: 1
p-value: 2.49239002583e-16
Decision: Reject the null hypothesis

Phi: -0.006022
Phi strength: Very weak or negligible

Cramer's V: 0.006022
Cramer's V strength: Very weak or negligible

Odds Ratio: 0.845648
95% CI: [0.812375, 0.880283]
Risk Ratio: 0.846385
Risk difference: -0.086703 percentage points

Adjusted standardized residuals:


,NON_FRAUD,FRAUD
2019,-8.195652,8.195652
2020,8.195652,-8.195652



COMPARATIVE TARGET-ASSOCIATION SUMMARY


,FEATURE,REFERENCE_CATEGORY,EXPOSURE_CATEGORY,PHI,PHI_STRENGTH,CRAMERS_V,CRAMERS_V_STRENGTH,CHI_SQUARE_P_VALUE,ODDS_RATIO,OR_CI_95_LOWER,OR_CI_95_UPPER,RISK_RATIO,RISK_DIFFERENCE_PERCENTAGE_POINTS
0,TRANS_YEAR_BE,2019,2020,-0.006022,Very weak or negligible,0.006022,Very weak or negligible,2.492390e-16,0.845648,0.812375,0.880283,0.846385,-0.086703
1,SEND_GENDER_BE,F,M,0.005844,Very weak or negligible,0.005844,Very weak or negligible,1.816963e-15,1.176079,1.129951,1.224091,1.175081,0.084525



ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/binary_encoding

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/binary_encoding/analysis_binary_encoding_target_association.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/binary_encoding/binary_target_send_gender_fraud_rate.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/binary_encoding/binary_target_trans_year_fraud_rate.png


## <span style="color:PINK"> CONTINUOS GEOGRAPHIC </span> ##

In [6]:

# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "02_relationships_with_target"
)

ASSOCIATION_GROUP = (
    "target_association"
)

FEATURE_GROUP = (
    "continuous_geographic"
)


SEND_LAT_FEATURE = (
    "SEND_LAT_REGISTER"
)

SEND_LONG_FEATURE = (
    "SEND_LONG_REGISTER"
)

RECEIVE_LAT_FEATURE = (
    "RECEIVE_LAT"
)

RECEIVE_LONG_FEATURE = (
    "RECEIVE_LONG"
)

TARGET_FEATURE = (
    "TARGET_OMEGA"
)


DISTANCE_FEATURE = (
    "SENDER_RECEIVER_DISTANCE_KM"
)


GEOGRAPHIC_FEATURES = [
    SEND_LAT_FEATURE,
    SEND_LONG_FEATURE,
    RECEIVE_LAT_FEATURE,
    RECEIVE_LONG_FEATURE
]


ANALYSIS_FEATURES = (
    GEOGRAPHIC_FEATURES
    +
    [
        DISTANCE_FEATURE
    ]
)


REQUIRED_FEATURES = (
    GEOGRAPHIC_FEATURES
    +
    [
        TARGET_FEATURE
    ]
)


TARGET_POSITIVE_VALUE = 1

ALPHA = 0.05

EARTH_RADIUS_KM = 6371.0088

MAP_RANDOM_SEED = 42

MAP_SAMPLE_PER_CLASS = 10000

DISTANCE_QUANTILE_BINS = 10

PNG_DPI = 300


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / ASSOCIATION_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_continuous_geographic_target_association.html"
)


SENDER_MAP_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_sender_target_map.html"
)


RECEIVER_MAP_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_receiver_target_map.html"
)


DISTANCE_DISTRIBUTION_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_distance_by_target.png"
)


DISTANCE_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_distance_fraud_rate_by_quantile.png"
)


RANK_BISERIAL_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_target_rank_biserial.png"
)


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD REQUIRED FEATURES
# ============================================================

dataset_geographic_target = pd.read_parquet(
    DATASET_PATH,
    columns=REQUIRED_FEATURES
)


total_observations = int(
    len(
        dataset_geographic_target
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. VALIDATE REQUIRED FEATURES
# ============================================================

missing_features = [
    feature
    for feature in REQUIRED_FEATURES
    if feature not in dataset_geographic_target.columns
]


if missing_features:

    raise KeyError(
        "Missing required features: "
        + ", ".join(
            missing_features
        )
    )


# ============================================================
# 09. VALIDATE NUMERICAL GEOGRAPHIC FEATURES
#
# The parquet dataset itself is not modified.
#
# A temporary numerical copy is created for analysis.
# ============================================================

working_data = (
    dataset_geographic_target
    .copy()
)


geographic_validation_records = []


for feature in GEOGRAPHIC_FEATURES:

    source_series = (
        dataset_geographic_target[
            feature
        ]
    )


    numeric_series = pd.to_numeric(
        source_series,
        errors="coerce"
    )


    invalid_non_numeric_mask = (
        source_series.notna()
        &
        numeric_series.isna()
    )


    invalid_non_numeric_count = int(
        invalid_non_numeric_mask.sum()
    )


    if invalid_non_numeric_count > 0:

        invalid_examples = (
            source_series.loc[
                invalid_non_numeric_mask
            ]
            .drop_duplicates()
            .head(
                10
            )
            .tolist()
        )


        raise TypeError(
            f"{feature} contains non-numeric values. "
            f"Invalid observations: {invalid_non_numeric_count}. "
            f"Examples: {invalid_examples}"
        )


    working_data[
        feature
    ] = (
        numeric_series.astype(
            "float64"
        )
    )


    feature_values = (
        numeric_series
        .dropna()
        .to_numpy(
            dtype="float64"
        )
    )


    non_finite_count = int(
        np.sum(
            ~np.isfinite(
                feature_values
            )
        )
    )


    geographic_validation_records.append({

        "FEATURE":
            feature,

        "SOURCE_DATA_TYPE":
            str(
                source_series.dtype
            ),

        "MISSING_VALUES":
            int(
                source_series.isna().sum()
            ),

        "NON_NUMERIC_VALUES":
            invalid_non_numeric_count,

        "NON_FINITE_VALUES":
            non_finite_count
    })


geographic_validation_table = pd.DataFrame(
    geographic_validation_records
)


# ============================================================
# 10. PREPARE COMMON COMPLETE FINITE SAMPLE
# ============================================================

complete_data = (
    working_data[
        REQUIRED_FEATURES
    ]
    .dropna()
    .copy()
)


finite_mask = np.isfinite(
    complete_data[
        GEOGRAPHIC_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


analysis_data = (
    complete_data.loc[
        finite_mask,
        REQUIRED_FEATURES
    ]
    .copy()
)


analysis_observations = int(
    len(
        analysis_data
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite geographic observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 11. VALIDATE GEOGRAPHIC RANGES
# ============================================================

latitude_features = [
    SEND_LAT_FEATURE,
    RECEIVE_LAT_FEATURE
]


longitude_features = [
    SEND_LONG_FEATURE,
    RECEIVE_LONG_FEATURE
]


invalid_range_records = []


for feature in latitude_features:

    invalid_count = int(
        (
            (analysis_data[feature] < -90)
            |
            (analysis_data[feature] > 90)
        )
        .sum()
    )


    invalid_range_records.append({

        "FEATURE":
            feature,

        "VALID_RANGE":
            "[-90, 90]",

        "OBSERVED_MIN":
            float(
                analysis_data[
                    feature
                ].min()
            ),

        "OBSERVED_MAX":
            float(
                analysis_data[
                    feature
                ].max()
            ),

        "INVALID_RANGE_VALUES":
            invalid_count
    })


for feature in longitude_features:

    invalid_count = int(
        (
            (analysis_data[feature] < -180)
            |
            (analysis_data[feature] > 180)
        )
        .sum()
    )


    invalid_range_records.append({

        "FEATURE":
            feature,

        "VALID_RANGE":
            "[-180, 180]",

        "OBSERVED_MIN":
            float(
                analysis_data[
                    feature
                ].min()
            ),

        "OBSERVED_MAX":
            float(
                analysis_data[
                    feature
                ].max()
            ),

        "INVALID_RANGE_VALUES":
            invalid_count
    })


geographic_range_table = pd.DataFrame(
    invalid_range_records
)


total_invalid_range_values = int(
    geographic_range_table[
        "INVALID_RANGE_VALUES"
    ]
    .sum()
)


if total_invalid_range_values > 0:

    display(
        geographic_range_table
    )


    raise ValueError(
        "Invalid geographic coordinate ranges were detected."
    )


# ============================================================
# 12. VALIDATE TARGET
# ============================================================

target_values = (
    analysis_data[
        TARGET_FEATURE
    ]
    .drop_duplicates()
    .tolist()
)


if len(
    target_values
) != 2:

    raise ValueError(
        f"{TARGET_FEATURE} must contain exactly two classes. "
        f"Observed values: {target_values}"
    )


target_positive_value = None


for value in target_values:

    if (
        value == TARGET_POSITIVE_VALUE
        or
        str(
            value
        )
        == str(
            TARGET_POSITIVE_VALUE
        )
    ):

        target_positive_value = (
            value
        )

        break


if target_positive_value is None:

    raise ValueError(
        f"Fraud target value {TARGET_POSITIVE_VALUE} was not found. "
        f"Observed values: {target_values}"
    )


target_negative_values = [
    value
    for value in target_values
    if value != target_positive_value
]


if len(
    target_negative_values
) != 1:

    raise ValueError(
        "Unable to determine the non-fraud target class."
    )


target_negative_value = (
    target_negative_values[
        0
    ]
)


# ============================================================
# 13. CREATE TEMPORARY BINARY TARGET
#
# Non-fraud = 0
# Fraud     = 1
#
# Temporary only.
# ============================================================

analysis_data[
    "_TARGET_BINARY"
] = (
    analysis_data[
        TARGET_FEATURE
    ]
    .map({

        target_negative_value:
            0,

        target_positive_value:
            1
    })
    .astype(
        "int8"
    )
)


# ============================================================
# 14. TARGET OVERVIEW
# ============================================================

target_counts = (
    analysis_data[
        "_TARGET_BINARY"
    ]
    .value_counts()
    .reindex(
        [
            0,
            1
        ],
        fill_value=0
    )
)


non_fraud_count = int(
    target_counts.loc[
        0
    ]
)


fraud_count = int(
    target_counts.loc[
        1
    ]
)


non_fraud_percentage = (
    non_fraud_count
    / analysis_observations
    * 100
)


fraud_percentage = (
    fraud_count
    / analysis_observations
    * 100
)


target_overview_table = pd.DataFrame({

    "TARGET_CLASS": [
        "Non-fraud",
        "Fraud"
    ],

    "TARGET_VALUE": [
        target_negative_value,
        target_positive_value
    ],

    "COUNT": [
        non_fraud_count,
        fraud_count
    ],

    "PERCENTAGE": [
        non_fraud_percentage,
        fraud_percentage
    ]
})


# ============================================================
# 15. HAVERSINE FUNCTION
# ============================================================

def haversine_distance_km(
    latitude_1,
    longitude_1,
    latitude_2,
    longitude_2
):

    latitude_1_rad = np.radians(
        latitude_1
    )


    longitude_1_rad = np.radians(
        longitude_1
    )


    latitude_2_rad = np.radians(
        latitude_2
    )


    longitude_2_rad = np.radians(
        longitude_2
    )


    delta_latitude = (
        latitude_2_rad
        - latitude_1_rad
    )


    delta_longitude = (
        longitude_2_rad
        - longitude_1_rad
    )


    haversine_a = (

        np.sin(
            delta_latitude
            / 2
        ) ** 2

        +

        np.cos(
            latitude_1_rad
        )

        *

        np.cos(
            latitude_2_rad
        )

        *

        np.sin(
            delta_longitude
            / 2
        ) ** 2
    )


    haversine_a = np.clip(
        haversine_a,
        0,
        1
    )


    angular_distance = (
        2
        *
        np.arcsin(
            np.sqrt(
                haversine_a
            )
        )
    )


    return (
        EARTH_RADIUS_KM
        *
        angular_distance
    )


# ============================================================
# 16. CREATE TEMPORARY SENDER-RECEIVER DISTANCE
#
# This derived variable exists only in memory.
#
# The parquet dataset is not changed.
# ============================================================

analysis_data[
    DISTANCE_FEATURE
] = haversine_distance_km(

    analysis_data[
        SEND_LAT_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    ),

    analysis_data[
        SEND_LONG_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    ),

    analysis_data[
        RECEIVE_LAT_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    ),

    analysis_data[
        RECEIVE_LONG_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


# ============================================================
# 17. EFFECT SIZE INTERPRETATION
#
# Descriptive exploratory guidelines only.
# ============================================================

def interpret_effect_strength(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        return (
            "Weak"
        )


    elif absolute_value < 0.50:

        return (
            "Moderate"
        )


    elif absolute_value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


# ============================================================
# 18. DESCRIPTIVE STATISTICS BY TARGET
# ============================================================

descriptive_records = []


for feature in ANALYSIS_FEATURES:

    for target_code, target_label in [
        (
            0,
            "Non-fraud"
        ),
        (
            1,
            "Fraud"
        )
    ]:

        values = (
            analysis_data.loc[
                analysis_data[
                    "_TARGET_BINARY"
                ]
                == target_code,
                feature
            ]
            .to_numpy(
                dtype="float64"
            )
        )


        q01 = float(
            np.quantile(
                values,
                0.01
            )
        )


        q25 = float(
            np.quantile(
                values,
                0.25
            )
        )


        q50 = float(
            np.quantile(
                values,
                0.50
            )
        )


        q75 = float(
            np.quantile(
                values,
                0.75
            )
        )


        q99 = float(
            np.quantile(
                values,
                0.99
            )
        )


        descriptive_records.append({

            "FEATURE":
                feature,

            "TARGET_CLASS":
                target_label,

            "COUNT":
                int(
                    len(
                        values
                    )
                ),

            "MEAN":
                float(
                    np.mean(
                        values
                    )
                ),

            "STANDARD_DEVIATION":
                float(
                    np.std(
                        values,
                        ddof=1
                    )
                ),

            "MIN":
                float(
                    np.min(
                        values
                    )
                ),

            "P01":
                q01,

            "P25":
                q25,

            "MEDIAN":
                q50,

            "P75":
                q75,

            "P99":
                q99,

            "MAX":
                float(
                    np.max(
                        values
                    )
                ),

            "IQR":
                float(
                    q75
                    - q25
                ),

            "SKEWNESS":
                float(
                    stats.skew(
                        values,
                        bias=False
                    )
                ),

            "EXCESS_KURTOSIS":
                float(
                    stats.kurtosis(
                        values,
                        fisher=True,
                        bias=False
                    )
                )
        })


descriptive_statistics_table = pd.DataFrame(
    descriptive_records
)


# ============================================================
# 19. GEOGRAPHIC CENTROID SUMMARY
#
# Arithmetic mean latitude and longitude are used only
# as descriptive coordinate centers.
# ============================================================

centroid_records = []


location_definitions = {

    "Sender": (
        SEND_LAT_FEATURE,
        SEND_LONG_FEATURE
    ),

    "Receiver": (
        RECEIVE_LAT_FEATURE,
        RECEIVE_LONG_FEATURE
    )
}


for (
    location_name,
    (
        latitude_feature,
        longitude_feature
    )
) in location_definitions.items():

    for target_code, target_label in [
        (
            0,
            "Non-fraud"
        ),
        (
            1,
            "Fraud"
        )
    ]:

        class_data = (
            analysis_data.loc[
                analysis_data[
                    "_TARGET_BINARY"
                ]
                == target_code
            ]
        )


        centroid_records.append({

            "LOCATION":
                location_name,

            "TARGET_CLASS":
                target_label,

            "MEAN_LATITUDE":
                float(
                    class_data[
                        latitude_feature
                    ]
                    .mean()
                ),

            "MEAN_LONGITUDE":
                float(
                    class_data[
                        longitude_feature
                    ]
                    .mean()
                ),

            "MEDIAN_LATITUDE":
                float(
                    class_data[
                        latitude_feature
                    ]
                    .median()
                ),

            "MEDIAN_LONGITUDE":
                float(
                    class_data[
                        longitude_feature
                    ]
                    .median()
                )
        })


centroid_table = pd.DataFrame(
    centroid_records
)


# ============================================================
# 20. CLASS CENTROID SEPARATION
# ============================================================

centroid_separation_records = []


for location_name in [
    "Sender",
    "Receiver"
]:

    location_centroids = (
        centroid_table[
            centroid_table[
                "LOCATION"
            ]
            == location_name
        ]
        .set_index(
            "TARGET_CLASS"
        )
    )


    separation_km = float(
        haversine_distance_km(

            location_centroids.loc[
                "Non-fraud",
                "MEAN_LATITUDE"
            ],

            location_centroids.loc[
                "Non-fraud",
                "MEAN_LONGITUDE"
            ],

            location_centroids.loc[
                "Fraud",
                "MEAN_LATITUDE"
            ],

            location_centroids.loc[
                "Fraud",
                "MEAN_LONGITUDE"
            ]
        )
    )


    centroid_separation_records.append({

        "LOCATION":
            location_name,

        "FRAUD_VS_NON_FRAUD_MEAN_COORDINATE_DISTANCE_KM":
            separation_km
    })


centroid_separation_table = pd.DataFrame(
    centroid_separation_records
)


# ============================================================
# 21. ASSOCIATION TESTS
#
# For every geographic variable:
#
# - Welch t-test
# - Mann-Whitney U
# - Point-biserial correlation
# - Rank-biserial correlation
#
# Mann-Whitney and rank-biserial are especially useful
# because geographic distributions may be non-normal,
# multimodal and highly asymmetric.
# ============================================================

association_records = []


target_binary_array = (
    analysis_data[
        "_TARGET_BINARY"
    ]
    .to_numpy(
        dtype="int8"
    )
)


for feature in ANALYSIS_FEATURES:

    non_fraud_values = (
        analysis_data.loc[
            analysis_data[
                "_TARGET_BINARY"
            ]
            == 0,
            feature
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    fraud_values = (
        analysis_data.loc[
            analysis_data[
                "_TARGET_BINARY"
            ]
            == 1,
            feature
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    # --------------------------------------------------------
    # MEAN AND MEDIAN DIFFERENCES
    # Fraud - Non-fraud
    # --------------------------------------------------------

    mean_difference = float(
        np.mean(
            fraud_values
        )
        -
        np.mean(
            non_fraud_values
        )
    )


    median_difference = float(
        np.median(
            fraud_values
        )
        -
        np.median(
            non_fraud_values
        )
    )


    # --------------------------------------------------------
    # WELCH T-TEST
    # --------------------------------------------------------

    welch_result = stats.ttest_ind(

        fraud_values,

        non_fraud_values,

        equal_var=False
    )


    welch_statistic = float(
        welch_result.statistic
    )


    welch_p_value = float(
        welch_result.pvalue
    )


    # --------------------------------------------------------
    # MANN-WHITNEY U
    #
    # Fraud group is x.
    #
    # Therefore positive rank-biserial values indicate
    # larger values among fraud observations.
    # --------------------------------------------------------

    mann_whitney_result = stats.mannwhitneyu(

        fraud_values,

        non_fraud_values,

        alternative="two-sided",

        method="asymptotic"
    )


    mann_whitney_u = float(
        mann_whitney_result.statistic
    )


    mann_whitney_p_value = float(
        mann_whitney_result.pvalue
    )


    n_fraud = float(
        len(
            fraud_values
        )
    )


    n_non_fraud = float(
        len(
            non_fraud_values
        )
    )


    common_language_probability = float(
        mann_whitney_u
        /
        (
            n_fraud
            *
            n_non_fraud
        )
    )


    rank_biserial = float(
        2
        *
        common_language_probability
        -
        1
    )


    rank_biserial_strength = (
        interpret_effect_strength(
            rank_biserial
        )
    )


    # --------------------------------------------------------
    # POINT-BISERIAL CORRELATION
    # --------------------------------------------------------

    feature_values = (
        analysis_data[
            feature
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    point_biserial_result = stats.pointbiserialr(

        target_binary_array,

        feature_values
    )


    point_biserial = float(
        point_biserial_result.statistic
    )


    point_biserial_p_value = float(
        point_biserial_result.pvalue
    )


    point_biserial_strength = (
        interpret_effect_strength(
            point_biserial
        )
    )


    association_records.append({

        "FEATURE":
            feature,

        "MEAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
            mean_difference,

        "MEDIAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
            median_difference,

        "WELCH_T":
            welch_statistic,

        "WELCH_P_VALUE":
            welch_p_value,

        "MANN_WHITNEY_U":
            mann_whitney_u,

        "MANN_WHITNEY_P_VALUE":
            mann_whitney_p_value,

        "POINT_BISERIAL":
            point_biserial,

        "POINT_BISERIAL_P_VALUE":
            point_biserial_p_value,

        "POINT_BISERIAL_STRENGTH":
            point_biserial_strength,

        "RANK_BISERIAL":
            rank_biserial,

        "RANK_BISERIAL_STRENGTH":
            rank_biserial_strength,

        "COMMON_LANGUAGE_PROBABILITY":
            common_language_probability
    })


association_table = pd.DataFrame(
    association_records
)


# ============================================================
# 22. ASSOCIATION RANKING
#
# Rank-biserial is used as the main robust effect measure.
# ============================================================

association_ranking_table = (
    association_table[
        [
            "FEATURE",
            "POINT_BISERIAL",
            "POINT_BISERIAL_STRENGTH",
            "RANK_BISERIAL",
            "RANK_BISERIAL_STRENGTH",
            "COMMON_LANGUAGE_PROBABILITY"
        ]
    ]
    .copy()
)


association_ranking_table[
    "ABS_RANK_BISERIAL"
] = (
    association_ranking_table[
        "RANK_BISERIAL"
    ]
    .abs()
)


association_ranking_table = (
    association_ranking_table
    .sort_values(
        by="ABS_RANK_BISERIAL",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 23. DISTANCE QUANTILE ANALYSIS
#
# Quantile bins are used only for descriptive visualization
# of possible non-linear relationships between distance
# and fraud rate.
# ============================================================

distance_quantiles = pd.qcut(

    analysis_data[
        DISTANCE_FEATURE
    ],

    q=DISTANCE_QUANTILE_BINS,

    duplicates="drop"
)


distance_quantile_data = pd.DataFrame({

    "DISTANCE_BIN":
        distance_quantiles,

    "DISTANCE_KM":
        analysis_data[
            DISTANCE_FEATURE
        ],

    "TARGET":
        analysis_data[
            "_TARGET_BINARY"
        ]
})


distance_quantile_summary = (
    distance_quantile_data
    .groupby(
        "DISTANCE_BIN",
        observed=True
    )
    .agg(

        COUNT=(
            "TARGET",
            "size"
        ),

        FRAUD_COUNT=(
            "TARGET",
            "sum"
        ),

        DISTANCE_MIN_KM=(
            "DISTANCE_KM",
            "min"
        ),

        DISTANCE_MEDIAN_KM=(
            "DISTANCE_KM",
            "median"
        ),

        DISTANCE_MAX_KM=(
            "DISTANCE_KM",
            "max"
        )
    )
    .reset_index()
)


distance_quantile_summary[
    "FRAUD_RATE"
] = (
    distance_quantile_summary[
        "FRAUD_COUNT"
    ]
    /
    distance_quantile_summary[
        "COUNT"
    ]
)


distance_quantile_summary[
    "FRAUD_PERCENTAGE"
] = (
    distance_quantile_summary[
        "FRAUD_RATE"
    ]
    * 100
)


distance_quantile_summary[
    "QUANTILE_GROUP"
] = (
    np.arange(
        1,
        len(
            distance_quantile_summary
        )
        + 1
    )
)


distance_quantile_summary[
    "DISTANCE_BIN"
] = (
    distance_quantile_summary[
        "DISTANCE_BIN"
    ]
    .astype(
        str
    )
)


# ============================================================
# 24. CREATE BALANCED MAP SAMPLE
#
# Sampling is used ONLY for visualization.
#
# Statistical analyses use all complete finite observations.
#
# Equal class sizes are used in maps so that the much larger
# non-fraud class does not visually hide the fraud class.
#
# Therefore, map density does NOT represent class prevalence.
# ============================================================

fraud_map_data = (
    analysis_data[
        analysis_data[
            "_TARGET_BINARY"
        ]
        == 1
    ]
)


non_fraud_map_data = (
    analysis_data[
        analysis_data[
            "_TARGET_BINARY"
        ]
        == 0
    ]
)


map_sample_size = min(
    MAP_SAMPLE_PER_CLASS,
    len(
        fraud_map_data
    ),
    len(
        non_fraud_map_data
    )
)


fraud_map_sample = (
    fraud_map_data
    .sample(
        n=map_sample_size,
        random_state=MAP_RANDOM_SEED
    )
    .copy()
)


non_fraud_map_sample = (
    non_fraud_map_data
    .sample(
        n=map_sample_size,
        random_state=MAP_RANDOM_SEED
    )
    .copy()
)


# ============================================================
# 25. CREATE INTERACTIVE SENDER MAP
# ============================================================

sender_map = go.Figure()


sender_map.add_trace(

    go.Scattergeo(

        lon=non_fraud_map_sample[
            SEND_LONG_FEATURE
        ],

        lat=non_fraud_map_sample[
            SEND_LAT_FEATURE
        ],

        mode="markers",

        name="Non-fraud",

        marker=dict(
            size=4,
            opacity=0.45
        )
    )
)


sender_map.add_trace(

    go.Scattergeo(

        lon=fraud_map_sample[
            SEND_LONG_FEATURE
        ],

        lat=fraud_map_sample[
            SEND_LAT_FEATURE
        ],

        mode="markers",

        name="Fraud",

        marker=dict(
            size=4,
            opacity=0.55
        )
    )
)


sender_map.update_geos(

    scope="usa",

    projection_type="albers usa",

    showland=True,

    showcountries=True,

    showsubunits=True
)


sender_map.update_layout(

    title=(
        "Sender locations by target class "
        "- balanced visualization sample"
    ),

    legend_title="Target class"
)


sender_map.write_html(

    SENDER_MAP_PATH,

    include_plotlyjs=True,

    full_html=True
)


# ============================================================
# 26. CREATE INTERACTIVE RECEIVER MAP
# ============================================================

receiver_map = go.Figure()


receiver_map.add_trace(

    go.Scattergeo(

        lon=non_fraud_map_sample[
            RECEIVE_LONG_FEATURE
        ],

        lat=non_fraud_map_sample[
            RECEIVE_LAT_FEATURE
        ],

        mode="markers",

        name="Non-fraud",

        marker=dict(
            size=4,
            opacity=0.45
        )
    )
)


receiver_map.add_trace(

    go.Scattergeo(

        lon=fraud_map_sample[
            RECEIVE_LONG_FEATURE
        ],

        lat=fraud_map_sample[
            RECEIVE_LAT_FEATURE
        ],

        mode="markers",

        name="Fraud",

        marker=dict(
            size=4,
            opacity=0.55
        )
    )
)


receiver_map.update_geos(

    scope="usa",

    projection_type="albers usa",

    showland=True,

    showcountries=True,

    showsubunits=True
)


receiver_map.update_layout(

    title=(
        "Receiver locations by target class "
        "- balanced visualization sample"
    ),

    legend_title="Target class"
)


receiver_map.write_html(

    RECEIVER_MAP_PATH,

    include_plotlyjs=True,

    full_html=True
)


# ============================================================
# 27. DISTANCE DISTRIBUTION PLOT
#
# The x-axis is limited to the 99.5th percentile only
# for visualization.
#
# Statistical tests use the complete distance distribution.
# ============================================================

non_fraud_distance = (
    analysis_data.loc[
        analysis_data[
            "_TARGET_BINARY"
        ]
        == 0,
        DISTANCE_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


fraud_distance = (
    analysis_data.loc[
        analysis_data[
            "_TARGET_BINARY"
        ]
        == 1,
        DISTANCE_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


distance_visual_limit = float(
    np.quantile(
        analysis_data[
            DISTANCE_FEATURE
        ],
        0.995
    )
)


distance_bins = np.linspace(
    0,
    distance_visual_limit,
    100
)


fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)


ax.hist(
    non_fraud_distance[
        non_fraud_distance
        <= distance_visual_limit
    ],
    bins=distance_bins,
    density=True,
    histtype="step",
    linewidth=1.5,
    label="Non-fraud"
)


ax.hist(
    fraud_distance[
        fraud_distance
        <= distance_visual_limit
    ],
    bins=distance_bins,
    density=True,
    histtype="step",
    linewidth=1.5,
    label="Fraud"
)


ax.set_xlabel(
    "Sender-receiver Haversine distance (km)"
)


ax.set_ylabel(
    "Density"
)


ax.set_title(
    "Sender-receiver distance distribution by target"
)


ax.legend()


ax.grid(
    alpha=0.20
)


fig.tight_layout()


fig.savefig(
    DISTANCE_DISTRIBUTION_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 28. FRAUD RATE BY DISTANCE QUANTILE PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)


ax.plot(

    distance_quantile_summary[
        "QUANTILE_GROUP"
    ],

    distance_quantile_summary[
        "FRAUD_PERCENTAGE"
    ],

    marker="o"
)


ax.axhline(

    fraud_percentage,

    linestyle="--",

    linewidth=1.5,

    label="Overall fraud rate"
)


ax.set_xlabel(
    "Sender-receiver distance quantile"
)


ax.set_ylabel(
    "Fraud rate (%)"
)


ax.set_title(
    "Fraud rate across sender-receiver distance quantiles"
)


ax.set_xticks(
    distance_quantile_summary[
        "QUANTILE_GROUP"
    ]
)


ax.legend()


ax.grid(
    alpha=0.20
)


fig.tight_layout()


fig.savefig(
    DISTANCE_FRAUD_RATE_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 29. RANK-BISERIAL EFFECT PLOT
# ============================================================

ranking_plot_table = (
    association_ranking_table
    .sort_values(
        by="RANK_BISERIAL",
        ascending=True
    )
)


fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)


ax.barh(

    ranking_plot_table[
        "FEATURE"
    ],

    ranking_plot_table[
        "RANK_BISERIAL"
    ]
)


ax.axvline(
    0,
    linewidth=1
)


ax.set_xlabel(
    "Rank-biserial correlation"
)


ax.set_ylabel(
    "Feature"
)


ax.set_title(
    "Geographic association with fraud - rank-biserial effect"
)


ax.grid(
    axis="x",
    alpha=0.20
)


fig.tight_layout()


fig.savefig(
    RANK_BISERIAL_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 30. IMAGE TO BASE64 FUNCTION
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 31. CONVERT PNG FIGURES TO BASE64
# ============================================================

distance_distribution_base64 = (
    image_to_base64(
        DISTANCE_DISTRIBUTION_PATH
    )
)


distance_fraud_rate_base64 = (
    image_to_base64(
        DISTANCE_FRAUD_RATE_PATH
    )
)


rank_biserial_base64 = (
    image_to_base64(
        RANK_BISERIAL_PATH
    )
)


# ============================================================
# 32. PREPARE HTML TABLES
# ============================================================

target_overview_html = (
    target_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PERCENTAGE":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


geographic_validation_html = (
    geographic_validation_table
    .to_html(
        index=False,
        border=0
    )
)


geographic_range_html = (
    geographic_range_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "OBSERVED_MIN":
                lambda value:
                    f"{value:.6f}",

            "OBSERVED_MAX":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


descriptive_statistics_html = (
    descriptive_statistics_table
    .to_html(
        index=False,
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


centroid_html = (
    centroid_table
    .to_html(
        index=False,
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


centroid_separation_html = (
    centroid_separation_table
    .to_html(
        index=False,
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


association_html = (
    association_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MEAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
                lambda value:
                    f"{value:.6f}",

            "MEDIAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
                lambda value:
                    f"{value:.6f}",

            "WELCH_T":
                lambda value:
                    f"{value:.6f}",

            "WELCH_P_VALUE":
                lambda value:
                    f"{value:.12g}",

            "MANN_WHITNEY_U":
                lambda value:
                    f"{value:.6f}",

            "MANN_WHITNEY_P_VALUE":
                lambda value:
                    f"{value:.12g}",

            "POINT_BISERIAL":
                lambda value:
                    f"{value:.6f}",

            "POINT_BISERIAL_P_VALUE":
                lambda value:
                    f"{value:.12g}",

            "RANK_BISERIAL":
                lambda value:
                    f"{value:.6f}",

            "COMMON_LANGUAGE_PROBABILITY":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


association_ranking_html = (
    association_ranking_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "POINT_BISERIAL":
                lambda value:
                    f"{value:.6f}",

            "RANK_BISERIAL":
                lambda value:
                    f"{value:.6f}",

            "COMMON_LANGUAGE_PROBABILITY":
                lambda value:
                    f"{value:.6f}",

            "ABS_RANK_BISERIAL":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


distance_quantile_html = (
    distance_quantile_summary
    .to_html(
        index=False,
        border=0,
        formatters={

            "DISTANCE_MIN_KM":
                lambda value:
                    f"{value:.6f}",

            "DISTANCE_MEDIAN_KM":
                lambda value:
                    f"{value:.6f}",

            "DISTANCE_MAX_KM":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE":
                lambda value:
                    f"{value:.8f}",

            "FRAUD_PERCENTAGE":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


# ============================================================
# 33. IDENTIFY STRONGEST ROBUST ASSOCIATION
# ============================================================

strongest_association_row = (
    association_ranking_table
    .iloc[
        0
    ]
)


strongest_feature = (
    strongest_association_row[
        "FEATURE"
    ]
)


strongest_rank_biserial = float(
    strongest_association_row[
        "RANK_BISERIAL"
    ]
)


strongest_rank_strength = (
    strongest_association_row[
        "RANK_BISERIAL_STRENGTH"
    ]
)


# ============================================================
# 34. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Continuous Geographic - Target Association
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1500px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 13px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 8px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.map-container {{
    margin-top: 25px;
    margin-bottom: 45px;
}}

.map-container iframe {{
    width: 100%;
    height: 650px;
    border: 1px solid #ccc;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Target Association —
Continuous Geographic
</h1>


<p>

Geographic features analyzed:

</p>


<ul>

<li>{SEND_LAT_FEATURE}</li>
<li>{SEND_LONG_FEATURE}</li>
<li>{RECEIVE_LAT_FEATURE}</li>
<li>{RECEIVE_LONG_FEATURE}</li>
<li>{DISTANCE_FEATURE} — temporarily derived</li>

</ul>


<p>

Target:

<strong>
{TARGET_FEATURE}
</strong>

</p>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete finite observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. TARGET OVERVIEW
========================================================= -->


<h2>
1. Target overview
</h2>


<div class="table-container">

{target_overview_html}

</div>


<p class="result">

Overall fraud rate:
{fraud_percentage:.6f}%

</p>


<div class="note">

The target is used only for exploratory association
analysis.

TARGET_OMEGA must not be included in the feature matrix
used to construct PCA, t-SNE or GMM.

</div>


<!-- ========================================================
     2. GEOGRAPHIC VALIDATION
========================================================= -->


<h2>
2. Geographic feature validation
</h2>


<div class="table-container">

{geographic_validation_html}

</div>


<div class="table-container">

{geographic_range_html}

</div>


<div class="note">

Latitude values are required to remain within [-90, 90].

Longitude values are required to remain within [-180, 180].

No invalid geographic value is silently corrected during
this analysis.

</div>


<!-- ========================================================
     3. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
3. Geographic descriptive statistics by target
</h2>


<div class="table-container">

{descriptive_statistics_html}

</div>


<div class="note">

Descriptive statistics are calculated separately for fraud
and non-fraud observations.

Because raw geographic coordinates may be multimodal and
non-normal, differences in means should not be interpreted
alone.

Median-based and rank-based comparisons are also used.

</div>


<!-- ========================================================
     4. SENDER LOCATION
========================================================= -->


<h2>
4. Sender location and target
</h2>


<div class="table-container">

{centroid_html}

</div>


<div class="table-container">

{centroid_separation_html}

</div>


<div class="note">

Mean latitude and longitude are used only as descriptive
coordinate centers.

The distance between fraud and non-fraud coordinate centers
is calculated using the Haversine formula.

These centers should not be interpreted as complete
descriptions of potentially multimodal geographic
distributions.

</div>


<div class="map-container">

<iframe
    src="{SENDER_MAP_PATH.name}"
    title="Sender target map"
></iframe>

</div>


<div class="note">

The interactive map uses a balanced random visualization
sample of {map_sample_size} fraud and
{map_sample_size} non-fraud observations.

This balancing prevents the much larger non-fraud class
from visually hiding fraud observations.

Therefore, map point density does not represent the true
class prevalence.

All statistical calculations use the complete valid sample.

</div>


<!-- ========================================================
     5. RECEIVER LOCATION
========================================================= -->


<h2>
5. Receiver location and target
</h2>


<div class="map-container">

<iframe
    src="{RECEIVER_MAP_PATH.name}"
    title="Receiver target map"
></iframe>

</div>


<div class="note">

The receiver map uses the same balanced visualization
strategy as the sender map.

Sampling is used only for visualization.

</div>


<!-- ========================================================
     6. HAVERSINE DISTANCE
========================================================= -->


<h2>
6. Sender-receiver Haversine distance and target
</h2>


<p>

A temporary geographic distance is calculated between
sender and receiver coordinates using the Haversine
formula:

</p>


<p class="result">

{DISTANCE_FEATURE}

</p>


<div class="chart">

<img
    src="data:image/png;base64,{distance_distribution_base64}"
    alt="Distance distribution by target"
>

</div>


<div class="note">

The distance-distribution graph limits the horizontal axis
to the 99.5th percentile only for visualization.

No observation is removed from the statistical analysis.

</div>


<h3>
Fraud rate across distance quantiles
</h3>


<div class="table-container">

{distance_quantile_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{distance_fraud_rate_base64}"
    alt="Fraud rate by distance quantile"
>

</div>


<div class="note">

Distance quantiles are used only as a descriptive
visualization of possible non-linear target relationships.

The continuous Haversine distance remains unchanged.

</div>


<!-- ========================================================
     7. STATISTICAL ASSOCIATION
========================================================= -->


<h2>
7. Statistical association with target
</h2>


<div class="table-container">

{association_html}

</div>


<div class="note">

<strong>Welch t-test</strong> compares class means without
assuming equal variances.

<br><br>

<strong>Mann-Whitney U</strong> compares the relative ranks
of fraud and non-fraud observations and does not require
normality.

<br><br>

<strong>Point-biserial correlation</strong> measures linear
association between the binary target and a numerical
feature.

<br><br>

<strong>Rank-biserial correlation</strong> is derived from
the Mann-Whitney statistic and provides a rank-based effect
size.

Positive values mean that fraud observations tend to have
larger feature values.

Negative values mean that fraud observations tend to have
smaller feature values.

</div>


<div class="note">

Because this dataset contains a very large number of
observations, very small effects can produce extremely small
p-values.

Effect sizes should therefore receive more interpretive
weight than statistical significance alone.

</div>


<!-- ========================================================
     8. EFFECT RANKING
========================================================= -->


<h2>
8. Geographic target-association ranking
</h2>


<div class="table-container">

{association_ranking_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{rank_biserial_base64}"
    alt="Rank-biserial geographic effect ranking"
>

</div>


<p class="result">

Strongest rank-based association:

<br>

{strongest_feature}

<br>

Rank-biserial:
{strongest_rank_biserial:.6f}

<br>

Descriptive strength:
{strongest_rank_strength}

</p>


<!-- ========================================================
     9. MODELING IMPLICATIONS
========================================================= -->


<h2>
9. Potential modeling implications
</h2>


<div class="note">

<strong>Geographic redundancy:</strong>

<br><br>

Previous within-group exploratory analysis identified very
high sender-receiver latitude and longitude associations.

This potential redundancy should be revisited before PCA
and GMM because repeated geographic information may affect
principal components and covariance estimation.

</div>


<div class="note">

<strong>Geographic geometry:</strong>

<br><br>

Latitude and longitude are coordinates on the Earth's
surface.

Euclidean operations on raw latitude and longitude do not
perfectly represent geographic distance.

Standardization changes scale but does not solve spherical
geometry.

The Haversine distance and alternative displacement
representations should therefore be reviewed during the
final pre-modeling audit.

</div>


<div class="note">

<strong>PCA:</strong>

<br><br>

PCA is sensitive to scaling, covariance structure and
redundancy.

Strong geographic correlations can concentrate variance
into a smaller number of principal components.

</div>


<div class="note">

<strong>t-SNE:</strong>

<br><br>

t-SNE depends on pairwise distance structure.

The representation of geographic coordinates can therefore
materially affect neighborhood relationships.

</div>


<div class="note">

<strong>GMM:</strong>

<br><br>

GMM estimates covariance structure within components.

Highly redundant geographic features may contribute to
ill-conditioned covariance matrices, especially when using
full covariance.

</div>


<div class="note">

<strong>Target association:</strong>

<br><br>

A geographic variable showing an association with fraud is
not automatically selected, removed or transformed.

TARGET_OMEGA is not used to construct the unsupervised
representation.

The findings are documented for interpretation and the
final pre-modeling audit.

</div>


<!-- ========================================================
     10. SUMMARY
========================================================= -->


<h2>
10. Summary
</h2>


<p class="result">

Raw geographic features analyzed:
{len(GEOGRAPHIC_FEATURES)}

</p>


<p class="result">

Temporary geographic features derived:
1

</p>


<p class="result">

Complete observations analyzed:
{analysis_observations}

</p>


<p class="result">

Overall fraud rate:
{fraud_percentage:.6f}%

</p>


<p class="result">

Strongest rank-based geographic association:

<br>

{strongest_feature}

<br>

Rank-biserial:
{strongest_rank_biserial:.6f}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

Sender and receiver geographic coordinates were compared
between fraud and non-fraud observations.

A temporary Haversine distance was additionally calculated
without altering the original parquet dataset.

<br><br>

Class-specific descriptive statistics, Welch tests,
Mann-Whitney tests, point-biserial correlations and
rank-biserial effect sizes were used to evaluate target
association.

<br><br>

Interactive maps provide spatial visualization using
balanced samples only.

All statistical analyses use the complete valid dataset.

<br><br>

The relationship between sender-receiver distance and
fraud was also examined continuously and across descriptive
distance quantiles.

<br><br>

No geographic feature is removed or permanently transformed
during this exploratory stage.

The geographic representation should be reviewed again
before PCA, t-SNE and GMM.

</div>


</body>

</html>
"""


# ============================================================
# 35. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 36. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CONTINUOUS GEOGRAPHIC - TARGET ASSOCIATION"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete finite observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 37. DISPLAY TARGET OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TARGET OVERVIEW"
)


print(
    "=" * 100
)


display(
    target_overview_table
)


# ============================================================
# 38. DISPLAY GEOGRAPHIC VALIDATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "GEOGRAPHIC VALIDATION"
)


print(
    "=" * 100
)


display(
    geographic_validation_table
)


display(
    geographic_range_table
)


# ============================================================
# 39. DISPLAY DESCRIPTIVE STATISTICS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "DESCRIPTIVE STATISTICS BY TARGET"
)


print(
    "=" * 100
)


display(
    descriptive_statistics_table
)


# ============================================================
# 40. DISPLAY CENTROID ANALYSIS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "GEOGRAPHIC COORDINATE CENTERS"
)


print(
    "=" * 100
)


display(
    centroid_table
)


display(
    centroid_separation_table
)


# ============================================================
# 41. DISPLAY DISTANCE QUANTILE ANALYSIS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "HAVERSINE DISTANCE QUANTILES AND FRAUD RATE"
)


print(
    "=" * 100
)


display(
    distance_quantile_summary
)


# ============================================================
# 42. DISPLAY ASSOCIATION TESTS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "GEOGRAPHIC TARGET ASSOCIATION TESTS"
)


print(
    "=" * 100
)


display(
    association_table
)


# ============================================================
# 43. DISPLAY ASSOCIATION RANKING
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "GEOGRAPHIC TARGET ASSOCIATION RANKING"
)


print(
    "=" * 100
)


display(
    association_ranking_table
)


# ============================================================
# 44. DISPLAY STRONGEST ASSOCIATION
# ============================================================

print(
    "\nStrongest rank-based geographic association:"
)


print(
    strongest_feature
)


print(
    "Rank-biserial:",
    f"{strongest_rank_biserial:.6f}"
)


print(
    "Strength:",
    strongest_rank_strength
)


# ============================================================
# 45. RELEASE MEMORY
# ============================================================

del dataset_geographic_target
del working_data
del complete_data

del fraud_map_data
del non_fraud_map_data

del fraud_map_sample
del non_fraud_map_sample

del non_fraud_distance
del fraud_distance

del distance_quantile_data

gc.collect()


# ============================================================
# 46. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nInteractive maps:"
)


print(
    SENDER_MAP_PATH
)


print(
    RECEIVER_MAP_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    DISTANCE_DISTRIBUTION_PATH
)


print(
    DISTANCE_FRAUD_RATE_PATH
)


print(
    RANK_BISERIAL_PATH
)


CONTINUOUS GEOGRAPHIC - TARGET ASSOCIATION

Total dataset observations: 1852394
Complete finite observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

TARGET OVERVIEW


,TARGET_CLASS,TARGET_VALUE,COUNT,PERCENTAGE
0,Non-fraud,0,1842743,99.478999
1,Fraud,1,9651,0.521001



GEOGRAPHIC VALIDATION


,FEATURE,SOURCE_DATA_TYPE,MISSING_VALUES,NON_NUMERIC_VALUES,NON_FINITE_VALUES
0,SEND_LAT_REGISTER,float32,0,0,0
1,SEND_LONG_REGISTER,float32,0,0,0
2,RECEIVE_LAT,float32,0,0,0
3,RECEIVE_LONG,float32,0,0,0


,FEATURE,VALID_RANGE,OBSERVED_MIN,OBSERVED_MAX,INVALID_RANGE_VALUES
0,SEND_LAT_REGISTER,"[-90, 90]",20.027100,66.693298,0
1,RECEIVE_LAT,"[-90, 90]",19.027422,67.510269,0
2,SEND_LONG_REGISTER,"[-180, 180]",-165.672302,-67.950302,0
3,RECEIVE_LONG,"[-180, 180]",-166.671570,-66.950905,0


...


GEOGRAPHIC TARGET ASSOCIATION TESTS


,FEATURE,MEAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD,MEDIAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD,WELCH_T,WELCH_P_VALUE,MANN_WHITNEY_U,MANN_WHITNEY_P_VALUE,POINT_BISERIAL,POINT_BISERIAL_P_VALUE,POINT_BISERIAL_STRENGTH,RANK_BISERIAL,RANK_BISERIAL_STRENGTH,COMMON_LANGUAGE_PROBABILITY
0,SEND_LAT_REGISTER,0.204567,0.182697,3.890083,0.000101,9.049551e+09,0.002665,0.002904,0.000077,Very weak or negligible,0.017700,Very weak or negligible,0.508850
1,SEND_LONG_REGISTER,0.195119,0.433296,1.341863,0.179672,9.055989e+09,0.001767,0.001022,0.164334,Very weak or negligible,0.018424,Very weak or negligible,0.509212
2,RECEIVE_LAT,0.197012,0.148346,3.717683,0.000202,9.039613e+09,0.004888,0.002778,0.000156,Very weak or negligible,0.016583,Very weak or negligible,0.508291
3,RECEIVE_LONG,0.191016,0.274406,1.311926,0.189576,9.053575e+09,0.002065,0.000999,0.173758,Very weak or negligible,0.018153,Very weak or negligible,0.509076
4,SENDER_RECEIVER_DISTANCE_KM,0.145263,-0.114622,0.493067,0.621976,8.906486e+09,0.784477,0.000359,0.624961,Very weak or negligible,0.001611,Very weak or negligible,0.500806



GEOGRAPHIC TARGET ASSOCIATION RANKING


,FEATURE,POINT_BISERIAL,POINT_BISERIAL_STRENGTH,RANK_BISERIAL,RANK_BISERIAL_STRENGTH,COMMON_LANGUAGE_PROBABILITY,ABS_RANK_BISERIAL
0,SEND_LONG_REGISTER,0.001022,Very weak or negligible,0.018424,Very weak or negligible,0.509212,0.018424
1,RECEIVE_LONG,0.000999,Very weak or negligible,0.018153,Very weak or negligible,0.509076,0.018153
2,SEND_LAT_REGISTER,0.002904,Very weak or negligible,0.017700,Very weak or negligible,0.508850,0.017700
3,RECEIVE_LAT,0.002778,Very weak or negligible,0.016583,Very weak or negligible,0.508291,0.016583
4,SENDER_RECEIVER_DISTANCE_KM,0.000359,Very weak or negligible,0.001611,Very weak or negligible,0.500806,0.001611



Strongest rank-based geographic association:
SEND_LONG_REGISTER
Rank-biserial: 0.018424
Strength: Very weak or negligible

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/continuous_geographic

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/continuous_geographic/analysis_continuous_geographic_target_association.html

Interactive maps:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/continuous_geographic/continuous_geographic_sender_target_map.html
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/continuous_geographic/continuous_geographic_receiver_target_map.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_

## <span style="color:PINK"> CYCLICAL ENDING USING SINE AND COSINE </span> ##

In [9]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "02_relationships_with_target"
)

ASSOCIATION_GROUP = (
    "target_association"
)

FEATURE_GROUP = (
    "cyclical_encoding_using_sine_and_cosine"
)


MONTH_SIN_FEATURE = (
    "TRANS_MONTH_SIN"
)

MONTH_COS_FEATURE = (
    "TRANS_MONTH_COS"
)

HOUR_SIN_FEATURE = (
    "TRANS_HOUR_SIN"
)

HOUR_COS_FEATURE = (
    "TRANS_HOUR_COS"
)

TARGET_FEATURE = (
    "TARGET_OMEGA"
)


CYCLICAL_FEATURES = [
    MONTH_SIN_FEATURE,
    MONTH_COS_FEATURE,
    HOUR_SIN_FEATURE,
    HOUR_COS_FEATURE
]


REQUIRED_FEATURES = (
    CYCLICAL_FEATURES
    +
    [
        TARGET_FEATURE
    ]
)


TARGET_POSITIVE_VALUE = 1

ALPHA = 0.05

STANDARDIZED_RESIDUAL_THRESHOLD = 2.0

PNG_DPI = 300


MONTH_ORDER = list(
    range(
        1,
        13
    )
)


HOUR_ORDER = list(
    range(
        0,
        24
    )
)


MONTH_LABELS = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / ASSOCIATION_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_cyclical_encoding_target_association.html"
)


MONTH_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_target_month_fraud_rate.png"
)


HOUR_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_target_hour_fraud_rate.png"
)


MONTH_RESIDUALS_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_target_month_adjusted_residuals.png"
)


HOUR_RESIDUALS_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_target_hour_adjusted_residuals.png"
)


MONTH_HOUR_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_target_month_hour_fraud_rate.png"
)


MONTH_HOUR_FRAUD_LIFT_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_target_month_hour_fraud_lift.png"
)


MONTH_CIRCULAR_SUMMARY_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_target_month_circular_summary.png"
)


HOUR_CIRCULAR_SUMMARY_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_target_hour_circular_summary.png"
)


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD REQUIRED FEATURES
# ============================================================

dataset_cyclical_target = pd.read_parquet(
    DATASET_PATH,
    columns=REQUIRED_FEATURES
)


total_observations = int(
    len(
        dataset_cyclical_target
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. VALIDATE REQUIRED FEATURES
# ============================================================

missing_features = [
    feature
    for feature in REQUIRED_FEATURES
    if feature not in dataset_cyclical_target.columns
]


if missing_features:

    raise KeyError(
        "Missing required features: "
        + ", ".join(
            missing_features
        )
    )


# ============================================================
# 09. TEMPORARY NUMERICAL COPY
#
# The source parquet dataset is never modified.
# ============================================================

working_data = (
    dataset_cyclical_target
    .copy()
)


for feature in CYCLICAL_FEATURES:

    numeric_series = pd.to_numeric(
        working_data[
            feature
        ],
        errors="coerce"
    )


    invalid_non_numeric_mask = (
        working_data[
            feature
        ]
        .notna()
        &
        numeric_series.isna()
    )


    invalid_non_numeric_count = int(
        invalid_non_numeric_mask.sum()
    )


    if invalid_non_numeric_count > 0:

        invalid_examples = (
            working_data.loc[
                invalid_non_numeric_mask,
                feature
            ]
            .drop_duplicates()
            .head(
                10
            )
            .tolist()
        )


        raise TypeError(
            f"{feature} contains non-numeric values. "
            f"Invalid observations: {invalid_non_numeric_count}. "
            f"Examples: {invalid_examples}"
        )


    working_data[
        feature
    ] = (
        numeric_series.astype(
            "float64"
        )
    )


# ============================================================
# 10. PREPARE COMPLETE FINITE SAMPLE
# ============================================================

complete_data = (
    working_data[
        REQUIRED_FEATURES
    ]
    .dropna()
    .copy()
)


finite_mask = np.isfinite(
    complete_data[
        CYCLICAL_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


analysis_data = (
    complete_data.loc[
        finite_mask,
        REQUIRED_FEATURES
    ]
    .copy()
)


analysis_observations = int(
    len(
        analysis_data
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite cyclical observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 11. VALIDATE CYCLICAL COMPONENT RANGE
#
# Sine and cosine values should remain within [-1, 1].
# ============================================================

cyclical_validation_records = []


for feature in CYCLICAL_FEATURES:

    minimum_value = float(
        analysis_data[
            feature
        ].min()
    )


    maximum_value = float(
        analysis_data[
            feature
        ].max()
    )


    outside_range_count = int(
        (
            (analysis_data[feature] < -1.0000000001)
            |
            (analysis_data[feature] > 1.0000000001)
        )
        .sum()
    )


    cyclical_validation_records.append({

        "FEATURE":
            feature,

        "MIN":
            minimum_value,

        "MAX":
            maximum_value,

        "VALUES_OUTSIDE_EXPECTED_RANGE":
            outside_range_count
    })


cyclical_validation_table = pd.DataFrame(
    cyclical_validation_records
)


if (
    cyclical_validation_table[
        "VALUES_OUTSIDE_EXPECTED_RANGE"
    ]
    .sum()
    > 0
):

    display(
        cyclical_validation_table
    )


    raise ValueError(
        "Invalid sine/cosine values were detected."
    )


# ============================================================
# 12. VALIDATE TARGET
# ============================================================

target_values = (
    analysis_data[
        TARGET_FEATURE
    ]
    .drop_duplicates()
    .tolist()
)


if len(
    target_values
) != 2:

    raise ValueError(
        f"{TARGET_FEATURE} must contain exactly two classes. "
        f"Observed values: {target_values}"
    )


target_positive_value = None


for value in target_values:

    if (
        value == TARGET_POSITIVE_VALUE
        or
        str(
            value
        )
        == str(
            TARGET_POSITIVE_VALUE
        )
    ):

        target_positive_value = (
            value
        )

        break


if target_positive_value is None:

    raise ValueError(
        f"Fraud target value {TARGET_POSITIVE_VALUE} was not found. "
        f"Observed values: {target_values}"
    )


target_negative_values = [
    value
    for value in target_values
    if value != target_positive_value
]


if len(
    target_negative_values
) != 1:

    raise ValueError(
        "Unable to determine the non-fraud target class."
    )


target_negative_value = (
    target_negative_values[
        0
    ]
)


# ============================================================
# 13. CREATE TEMPORARY BINARY TARGET
# ============================================================

analysis_data[
    "_TARGET_BINARY"
] = (
    analysis_data[
        TARGET_FEATURE
    ]
    .map({

        target_negative_value:
            0,

        target_positive_value:
            1
    })
    .astype(
        "int8"
    )
)


# ============================================================
# 14. TARGET OVERVIEW
# ============================================================

target_counts = (
    analysis_data[
        "_TARGET_BINARY"
    ]
    .value_counts()
    .reindex(
        [
            0,
            1
        ],
        fill_value=0
    )
)


non_fraud_count = int(
    target_counts.loc[
        0
    ]
)


fraud_count = int(
    target_counts.loc[
        1
    ]
)


non_fraud_percentage = (
    non_fraud_count
    / analysis_observations
    * 100
)


fraud_percentage = (
    fraud_count
    / analysis_observations
    * 100
)


overall_fraud_rate = (
    fraud_count
    / analysis_observations
)


target_overview_table = pd.DataFrame({

    "TARGET_CLASS": [
        "Non-fraud",
        "Fraud"
    ],

    "TARGET_VALUE": [
        target_negative_value,
        target_positive_value
    ],

    "COUNT": [
        non_fraud_count,
        fraud_count
    ],

    "PERCENTAGE": [
        non_fraud_percentage,
        fraud_percentage
    ]
})


# ============================================================
# 15. RECONSTRUCT MONTH
#
# Original preprocessing:
#
# sin(2*pi*(month - 1)/12)
# cos(2*pi*(month - 1)/12)
# ============================================================

month_sin_values = (
    analysis_data[
        MONTH_SIN_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


month_cos_values = (
    analysis_data[
        MONTH_COS_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


month_angle = np.mod(

    np.arctan2(
        month_sin_values,
        month_cos_values
    ),

    2
    *
    np.pi
)


month_index = np.mod(

    np.rint(
        month_angle
        *
        12
        /
        (
            2
            *
            np.pi
        )
    )
    .astype(
        int
    ),

    12
)


analysis_data[
    "_RECONSTRUCTED_MONTH"
] = (
    month_index
    + 1
)


# ============================================================
# 16. RECONSTRUCT HOUR
#
# Original preprocessing used decimal hour:
#
# hour + minute/60 + second/3600
#
# The exact circular angle is retained separately.
#
# For categorical target analysis, the reconstructed
# decimal hour is grouped into integer hours 0-23.
# ============================================================

hour_sin_values = (
    analysis_data[
        HOUR_SIN_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


hour_cos_values = (
    analysis_data[
        HOUR_COS_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


hour_angle = np.mod(

    np.arctan2(
        hour_sin_values,
        hour_cos_values
    ),

    2
    *
    np.pi
)


reconstructed_decimal_hour = np.mod(

    hour_angle
    *
    24
    /
    (
        2
        *
        np.pi
    ),

    24
)


analysis_data[
    "_RECONSTRUCTED_DECIMAL_HOUR"
] = (
    reconstructed_decimal_hour
)


analysis_data[
    "_RECONSTRUCTED_HOUR"
] = np.mod(

    np.floor(
        reconstructed_decimal_hour
        + 1e-10
    )
    .astype(
        int
    ),

    24
)


# ============================================================
# 17. RECONSTRUCTION OVERVIEW
# ============================================================

reconstruction_overview_table = pd.DataFrame({

    "TEMPORARY_FEATURE": [
        "_RECONSTRUCTED_MONTH",
        "_RECONSTRUCTED_DECIMAL_HOUR",
        "_RECONSTRUCTED_HOUR"
    ],

    "MIN": [
        int(
            analysis_data[
                "_RECONSTRUCTED_MONTH"
            ].min()
        ),

        float(
            analysis_data[
                "_RECONSTRUCTED_DECIMAL_HOUR"
            ].min()
        ),

        int(
            analysis_data[
                "_RECONSTRUCTED_HOUR"
            ].min()
        )
    ],

    "MAX": [
        int(
            analysis_data[
                "_RECONSTRUCTED_MONTH"
            ].max()
        ),

        float(
            analysis_data[
                "_RECONSTRUCTED_DECIMAL_HOUR"
            ].max()
        ),

        int(
            analysis_data[
                "_RECONSTRUCTED_HOUR"
            ].max()
        )
    ],

    "UNIQUE_VALUES": [
        int(
            analysis_data[
                "_RECONSTRUCTED_MONTH"
            ].nunique()
        ),

        int(
            analysis_data[
                "_RECONSTRUCTED_DECIMAL_HOUR"
            ].nunique()
        ),

        int(
            analysis_data[
                "_RECONSTRUCTED_HOUR"
            ].nunique()
        )
    ]
})


# ============================================================
# 18. CRAMER'S V INTERPRETATION
# ============================================================

def interpret_association_strength(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        return (
            "Weak"
        )


    elif absolute_value < 0.50:

        return (
            "Moderate"
        )


    elif absolute_value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


# ============================================================
# 19. CATEGORICAL TARGET ASSOCIATION FUNCTION
#
# Used for:
#
# Month × TARGET
# Hour × TARGET
# Combined Month-Hour × TARGET
# ============================================================

def analyze_categorical_target(
    dataframe,
    category_column,
    category_order=None
):

    contingency_table = pd.crosstab(

        dataframe[
            category_column
        ],

        dataframe[
            "_TARGET_BINARY"
        ]
    )


    contingency_table = (
        contingency_table
        .reindex(
            columns=[
                0,
                1
            ],
            fill_value=0
        )
    )


    contingency_table.columns = [
        "NON_FRAUD",
        "FRAUD"
    ]


    if category_order is not None:

        contingency_table = (
            contingency_table
            .reindex(
                category_order,
                fill_value=0
            )
        )


    valid_rows = (
        contingency_table
        .sum(
            axis=1
        )
        > 0
    )


    statistical_table = (
        contingency_table.loc[
            valid_rows
        ]
        .copy()
    )


    observed_matrix = (
        statistical_table
        .to_numpy(
            dtype="float64"
        )
    )


    (
        chi_square_statistic,
        chi_square_p_value,
        chi_square_degrees_of_freedom,
        expected_matrix
    ) = stats.chi2_contingency(
        observed_matrix,
        correction=False
    )


    chi_square_statistic = float(
        chi_square_statistic
    )


    chi_square_p_value = float(
        chi_square_p_value
    )


    chi_square_degrees_of_freedom = int(
        chi_square_degrees_of_freedom
    )


    expected_matrix = np.asarray(
        expected_matrix,
        dtype="float64"
    )


    number_rows = int(
        observed_matrix.shape[
            0
        ]
    )


    number_columns = int(
        observed_matrix.shape[
            1
        ]
    )


    cramers_dimension = min(
        number_rows - 1,
        number_columns - 1
    )


    if cramers_dimension > 0:

        cramers_v = float(
            np.sqrt(
                chi_square_statistic
                /
                (
                    observed_matrix.sum()
                    *
                    cramers_dimension
                )
            )
        )


    else:

        cramers_v = np.nan


    cramers_v_strength = (
        interpret_association_strength(
            cramers_v
        )
    )


    minimum_expected_frequency = float(
        np.min(
            expected_matrix
        )
    )


    cells_expected_below_5 = int(
        np.sum(
            expected_matrix < 5
        )
    )


    expected_cells = int(
        expected_matrix.size
    )


    expected_below_5_percentage = (
        cells_expected_below_5
        /
        expected_cells
        * 100
    )


    # --------------------------------------------------------
    # ADJUSTED STANDARDIZED RESIDUALS
    # --------------------------------------------------------

    row_totals = (
        observed_matrix
        .sum(
            axis=1,
            keepdims=True
        )
    )


    column_totals = (
        observed_matrix
        .sum(
            axis=0,
            keepdims=True
        )
    )


    grand_total = float(
        observed_matrix.sum()
    )


    row_proportions = (
        row_totals
        /
        grand_total
    )


    column_proportions = (
        column_totals
        /
        grand_total
    )


    residual_denominator = np.sqrt(

        expected_matrix

        *

        (
            1
            -
            row_proportions
        )

        *

        (
            1
            -
            column_proportions
        )
    )


    adjusted_residual_matrix = np.divide(

        observed_matrix
        -
        expected_matrix,

        residual_denominator,

        out=np.full_like(
            observed_matrix,
            np.nan,
            dtype="float64"
        ),

        where=(
            residual_denominator
            > 0
        )
    )


    adjusted_residuals_table = pd.DataFrame(

        adjusted_residual_matrix,

        index=statistical_table.index,

        columns=[
            "NON_FRAUD",
            "FRAUD"
        ]
    )


    # --------------------------------------------------------
    # FRAUD RATE
    # --------------------------------------------------------

    category_totals = (
        contingency_table
        .sum(
            axis=1
        )
    )


    fraud_rate = np.divide(

        contingency_table[
            "FRAUD"
        ]
        .to_numpy(
            dtype="float64"
        ),

        category_totals
        .to_numpy(
            dtype="float64"
        ),

        out=np.full(
            len(
                contingency_table
            ),
            np.nan,
            dtype="float64"
        ),

        where=(
            category_totals
            .to_numpy(
                dtype="float64"
            )
            > 0
        )
    )


    fraud_rate_table = pd.DataFrame({

        "CATEGORY":
            contingency_table.index,

        "TOTAL":
            category_totals
            .astype(
                int
            )
            .to_numpy(),

        "NON_FRAUD":
            contingency_table[
                "NON_FRAUD"
            ]
            .astype(
                int
            )
            .to_numpy(),

        "FRAUD":
            contingency_table[
                "FRAUD"
            ]
            .astype(
                int
            )
            .to_numpy(),

        "FRAUD_RATE":
            fraud_rate,

        "FRAUD_PERCENTAGE":
            fraud_rate
            * 100,

        "FRAUD_RATE_LIFT":
            fraud_rate
            /
            overall_fraud_rate
    })


    # --------------------------------------------------------
    # MUTUAL INFORMATION
    # --------------------------------------------------------

    category_codes, _ = pd.factorize(
        dataframe[
            category_column
        ],
        sort=True
    )


    target_codes = (
        dataframe[
            "_TARGET_BINARY"
        ]
        .to_numpy(
            dtype="int8"
        )
    )


    mutual_information = float(
        mutual_info_score(
            category_codes,
            target_codes
        )
    )


    normalized_mutual_information = float(
        normalized_mutual_info_score(
            category_codes,
            target_codes,
            average_method="arithmetic"
        )
    )


    # --------------------------------------------------------
    # SIGNIFICANCE DECISION
    # --------------------------------------------------------

    if chi_square_p_value < ALPHA:

        decision = (
            "Reject the null hypothesis"
        )


    else:

        decision = (
            "Do not reject the null hypothesis"
        )


    flagged_residual_cells = int(
        np.sum(
            np.abs(
                adjusted_residual_matrix
            )
            >= STANDARDIZED_RESIDUAL_THRESHOLD
        )
    )


    return {

        "contingency_table":
            contingency_table,

        "statistical_table":
            statistical_table,

        "fraud_rate_table":
            fraud_rate_table,

        "chi_square_statistic":
            chi_square_statistic,

        "chi_square_p_value":
            chi_square_p_value,

        "degrees_of_freedom":
            chi_square_degrees_of_freedom,

        "cramers_v":
            cramers_v,

        "cramers_v_strength":
            cramers_v_strength,

        "minimum_expected_frequency":
            minimum_expected_frequency,

        "cells_expected_below_5":
            cells_expected_below_5,

        "expected_below_5_percentage":
            expected_below_5_percentage,

        "adjusted_residuals_table":
            adjusted_residuals_table,

        "flagged_residual_cells":
            flagged_residual_cells,

        "mutual_information":
            mutual_information,

        "normalized_mutual_information":
            normalized_mutual_information,

        "decision":
            decision
    }


# ============================================================
# 20. MONTH × TARGET
# ============================================================

month_result = analyze_categorical_target(

    dataframe=analysis_data,

    category_column="_RECONSTRUCTED_MONTH",

    category_order=MONTH_ORDER
)


# ============================================================
# 21. HOUR × TARGET
# ============================================================

hour_result = analyze_categorical_target(

    dataframe=analysis_data,

    category_column="_RECONSTRUCTED_HOUR",

    category_order=HOUR_ORDER
)


# ============================================================
# 22. CREATE COMBINED MONTH-HOUR CATEGORY
#
# 12 months × 24 hours = up to 288 temporal cells.
#
# This temporary category is used to detect target
# association that may exist in the joint temporal
# structure even when month or hour alone shows only
# weak association.
# ============================================================

analysis_data[
    "_MONTH_HOUR_CATEGORY"
] = (

    analysis_data[
        "_RECONSTRUCTED_MONTH"
    ]
    .astype(
        str
    )

    +

    "_"

    +

    analysis_data[
        "_RECONSTRUCTED_HOUR"
    ]
    .astype(
        str
    )
)


month_hour_result = analyze_categorical_target(

    dataframe=analysis_data,

    category_column="_MONTH_HOUR_CATEGORY"
)


# ============================================================
# 23. CIRCULAR SUMMARY FUNCTION
#
# Mean resultant length:
#
# R = sqrt(mean(sin)^2 + mean(cos)^2)
#
# R close to 1:
# observations are concentrated around a direction.
#
# R close to 0:
# observations are widely dispersed around the cycle.
# ============================================================

def circular_summary(
    sine_values,
    cosine_values,
    cycle_length,
    offset=0.0
):

    mean_sine = float(
        np.mean(
            sine_values
        )
    )


    mean_cosine = float(
        np.mean(
            cosine_values
        )
    )


    mean_angle = float(
        np.mod(
            np.arctan2(
                mean_sine,
                mean_cosine
            ),
            2
            *
            np.pi
        )
    )


    mean_resultant_length = float(
        np.sqrt(
            mean_sine ** 2
            +
            mean_cosine ** 2
        )
    )


    circular_variance = float(
        1
        -
        mean_resultant_length
    )


    mean_cycle_position = float(

        mean_angle
        *
        cycle_length
        /
        (
            2
            *
            np.pi
        )

        +

        offset
    )


    return {

        "MEAN_SINE":
            mean_sine,

        "MEAN_COSINE":
            mean_cosine,

        "MEAN_ANGLE_RADIANS":
            mean_angle,

        "MEAN_CYCLE_POSITION":
            mean_cycle_position,

        "MEAN_RESULTANT_LENGTH":
            mean_resultant_length,

        "CIRCULAR_VARIANCE":
            circular_variance
    }


# ============================================================
# 24. CIRCULAR SUMMARY BY TARGET
# ============================================================

circular_summary_records = []


for target_code, target_label in [
    (
        0,
        "Non-fraud"
    ),
    (
        1,
        "Fraud"
    )
]:

    class_mask = (
        analysis_data[
            "_TARGET_BINARY"
        ]
        == target_code
    )


    # --------------------------------------------------------
    # MONTH
    # --------------------------------------------------------

    month_summary = circular_summary(

        sine_values=analysis_data.loc[
            class_mask,
            MONTH_SIN_FEATURE
        ]
        .to_numpy(
            dtype="float64"
        ),

        cosine_values=analysis_data.loc[
            class_mask,
            MONTH_COS_FEATURE
        ]
        .to_numpy(
            dtype="float64"
        ),

        cycle_length=12,

        offset=1
    )


    circular_summary_records.append({

        "CYCLE":
            "Month",

        "TARGET_CLASS":
            target_label,

        **month_summary
    })


    # --------------------------------------------------------
    # HOUR
    # --------------------------------------------------------

    hour_summary = circular_summary(

        sine_values=analysis_data.loc[
            class_mask,
            HOUR_SIN_FEATURE
        ]
        .to_numpy(
            dtype="float64"
        ),

        cosine_values=analysis_data.loc[
            class_mask,
            HOUR_COS_FEATURE
        ]
        .to_numpy(
            dtype="float64"
        ),

        cycle_length=24,

        offset=0
    )


    circular_summary_records.append({

        "CYCLE":
            "Hour",

        "TARGET_CLASS":
            target_label,

        **hour_summary
    })


circular_summary_table = pd.DataFrame(
    circular_summary_records
)


# ============================================================
# 25. CIRCULAR MEAN DIFFERENCE FUNCTION
#
# Returns the shortest signed angular difference:
#
# Fraud - Non-fraud
# ============================================================

def circular_mean_difference(
    angle_fraud,
    angle_non_fraud,
    cycle_length
):

    angular_difference = float(
        np.arctan2(

            np.sin(
                angle_fraud
                -
                angle_non_fraud
            ),

            np.cos(
                angle_fraud
                -
                angle_non_fraud
            )
        )
    )


    cycle_difference = float(
        angular_difference
        *
        cycle_length
        /
        (
            2
            *
            np.pi
        )
    )


    return (
        angular_difference,
        cycle_difference
    )


# ============================================================
# 26. CIRCULAR CLASS DIFFERENCE
# ============================================================

circular_difference_records = []


for cycle_name, cycle_length in [
    (
        "Month",
        12
    ),
    (
        "Hour",
        24
    )
]:

    cycle_table = (
        circular_summary_table[
            circular_summary_table[
                "CYCLE"
            ]
            == cycle_name
        ]
        .set_index(
            "TARGET_CLASS"
        )
    )


    fraud_angle = float(
        cycle_table.loc[
            "Fraud",
            "MEAN_ANGLE_RADIANS"
        ]
    )


    non_fraud_angle = float(
        cycle_table.loc[
            "Non-fraud",
            "MEAN_ANGLE_RADIANS"
        ]
    )


    (
        angular_difference,
        cycle_difference
    ) = circular_mean_difference(

        fraud_angle,

        non_fraud_angle,

        cycle_length
    )


    circular_difference_records.append({

        "CYCLE":
            cycle_name,

        "SIGNED_ANGLE_DIFFERENCE_RADIANS":
            angular_difference,

        "SIGNED_CYCLE_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
            cycle_difference,

        "ABSOLUTE_CYCLE_DIFFERENCE":
            abs(
                cycle_difference
            )
    })


circular_difference_table = pd.DataFrame(
    circular_difference_records
)


# ============================================================
# 27. MONTH-HOUR FRAUD RATE MATRIX
# ============================================================

month_hour_total_table = pd.crosstab(

    analysis_data[
        "_RECONSTRUCTED_MONTH"
    ],

    analysis_data[
        "_RECONSTRUCTED_HOUR"
    ]
)


month_hour_total_table = (
    month_hour_total_table
    .reindex(
        index=MONTH_ORDER,
        columns=HOUR_ORDER,
        fill_value=0
    )
)


fraud_only_data = (
    analysis_data[
        analysis_data[
            "_TARGET_BINARY"
        ]
        == 1
    ]
)


month_hour_fraud_table = pd.crosstab(

    fraud_only_data[
        "_RECONSTRUCTED_MONTH"
    ],

    fraud_only_data[
        "_RECONSTRUCTED_HOUR"
    ]
)


month_hour_fraud_table = (
    month_hour_fraud_table
    .reindex(
        index=MONTH_ORDER,
        columns=HOUR_ORDER,
        fill_value=0
    )
)


month_hour_fraud_rate_table = (
    month_hour_fraud_table
    .div(
        month_hour_total_table
        .replace(
            0,
            np.nan
        )
    )
)


month_hour_fraud_percentage_table = (
    month_hour_fraud_rate_table
    * 100
)


month_hour_fraud_lift_table = (
    month_hour_fraud_rate_table
    /
    overall_fraud_rate
)


# ============================================================
# 28. ASSOCIATION COMPARISON TABLE
# ============================================================

association_comparison_table = pd.DataFrame({

    "TEMPORAL_STRUCTURE": [
        "Month",
        "Hour",
        "Month-Hour joint cell"
    ],

    "CHI_SQUARE":
        [
            month_result[
                "chi_square_statistic"
            ],

            hour_result[
                "chi_square_statistic"
            ],

            month_hour_result[
                "chi_square_statistic"
            ]
        ],

    "DEGREES_OF_FREEDOM":
        [
            month_result[
                "degrees_of_freedom"
            ],

            hour_result[
                "degrees_of_freedom"
            ],

            month_hour_result[
                "degrees_of_freedom"
            ]
        ],

    "P_VALUE":
        [
            month_result[
                "chi_square_p_value"
            ],

            hour_result[
                "chi_square_p_value"
            ],

            month_hour_result[
                "chi_square_p_value"
            ]
        ],

    "CRAMERS_V":
        [
            month_result[
                "cramers_v"
            ],

            hour_result[
                "cramers_v"
            ],

            month_hour_result[
                "cramers_v"
            ]
        ],

    "CRAMERS_V_STRENGTH":
        [
            month_result[
                "cramers_v_strength"
            ],

            hour_result[
                "cramers_v_strength"
            ],

            month_hour_result[
                "cramers_v_strength"
            ]
        ],

    "MUTUAL_INFORMATION":
        [
            month_result[
                "mutual_information"
            ],

            hour_result[
                "mutual_information"
            ],

            month_hour_result[
                "mutual_information"
            ]
        ],

    "NORMALIZED_MUTUAL_INFORMATION":
        [
            month_result[
                "normalized_mutual_information"
            ],

            hour_result[
                "normalized_mutual_information"
            ],

            month_hour_result[
                "normalized_mutual_information"
            ]
        ]
})


association_comparison_table[
    "ABS_CRAMERS_V"
] = (
    association_comparison_table[
        "CRAMERS_V"
    ]
    .abs()
)


association_comparison_table = (
    association_comparison_table
    .sort_values(
        by="ABS_CRAMERS_V",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 29. CREATE FRAUD RATE LINE PLOT FUNCTION
# ============================================================

def create_fraud_rate_plot(
    fraud_rate_table,
    title,
    x_label,
    output_path,
    labels=None
):

    plot_table = (
        fraud_rate_table
        .copy()
    )


    x_values = np.arange(
        len(
            plot_table
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            12,
            7
        )
    )


    ax.plot(

        x_values,

        plot_table[
            "FRAUD_PERCENTAGE"
        ],

        marker="o"
    )


    ax.axhline(

        overall_fraud_rate
        * 100,

        linestyle="--",

        linewidth=1.5,

        label="Overall fraud rate"
    )


    if labels is None:

        labels = (
            plot_table[
                "CATEGORY"
            ]
            .astype(
                str
            )
            .tolist()
        )


    ax.set_xticks(
        x_values
    )


    ax.set_xticklabels(
        labels,
        rotation=45,
        ha="right"
    )


    ax.set_xlabel(
        x_label
    )


    ax.set_ylabel(
        "Fraud rate (%)"
    )


    ax.set_title(
        title
    )


    ax.legend()


    ax.grid(
        alpha=0.20
    )


    fig.tight_layout()


    fig.savefig(
        output_path,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 30. MONTH FRAUD RATE PLOT
# ============================================================

month_plot_labels = [
    MONTH_LABELS[
        month
    ]
    for month in MONTH_ORDER
]


create_fraud_rate_plot(

    fraud_rate_table=month_result[
        "fraud_rate_table"
    ],

    title=(
        "Fraud rate by reconstructed month"
    ),

    x_label="Month",

    output_path=MONTH_FRAUD_RATE_PATH,

    labels=month_plot_labels
)


# ============================================================
# 31. HOUR FRAUD RATE PLOT
# ============================================================

hour_plot_labels = [
    str(
        hour
    )
    for hour in HOUR_ORDER
]


create_fraud_rate_plot(

    fraud_rate_table=hour_result[
        "fraud_rate_table"
    ],

    title=(
        "Fraud rate by reconstructed hour"
    ),

    x_label="Hour",

    output_path=HOUR_FRAUD_RATE_PATH,

    labels=hour_plot_labels
)


# ============================================================
# 32. RESIDUAL HEATMAP FUNCTION
# ============================================================

def create_residual_heatmap(
    residual_table,
    title,
    output_path,
    y_labels=None
):

    values = (
        residual_table
        .to_numpy(
            dtype="float64"
        )
    )


    absolute_max = float(
        np.nanmax(
            np.abs(
                values
            )
        )
    )


    if (
        not np.isfinite(
            absolute_max
        )
        or
        absolute_max == 0
    ):

        absolute_max = 1.0


    fig, ax = plt.subplots(
        figsize=(
            8,
            max(
                6,
                len(
                    residual_table
                )
                * 0.35
            )
        )
    )


    image = ax.imshow(

        values,

        aspect="auto",

        vmin=-absolute_max,

        vmax=absolute_max
    )


    ax.set_xticks(
        [
            0,
            1
        ]
    )


    ax.set_xticklabels(
        [
            "Non-fraud",
            "Fraud"
        ]
    )


    ax.set_yticks(
        np.arange(
            len(
                residual_table
            )
        )
    )


    if y_labels is None:

        y_labels = (
            residual_table.index
            .astype(
                str
            )
            .tolist()
        )


    ax.set_yticklabels(
        y_labels
    )


    for row_index in range(
        values.shape[
            0
        ]
    ):

        for column_index in range(
            values.shape[
                1
            ]
        ):

            value = (
                values[
                    row_index,
                    column_index
                ]
            )


            ax.text(

                column_index,

                row_index,

                f"{value:.2f}",

                ha="center",

                va="center"
            )


    ax.set_title(
        title
    )


    fig.colorbar(

        image,

        ax=ax,

        label=(
            "Adjusted standardized residual"
        )
    )


    fig.tight_layout()


    fig.savefig(

        output_path,

        format="png",

        dpi=PNG_DPI,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 33. MONTH RESIDUAL HEATMAP
# ============================================================

month_residual_labels = [
    MONTH_LABELS[
        int(
            month
        )
    ]
    for month in month_result[
        "adjusted_residuals_table"
    ].index
]


create_residual_heatmap(

    residual_table=month_result[
        "adjusted_residuals_table"
    ],

    title=(
        "Adjusted standardized residuals - Month × Target"
    ),

    output_path=MONTH_RESIDUALS_PATH,

    y_labels=month_residual_labels
)


# ============================================================
# 34. HOUR RESIDUAL HEATMAP
# ============================================================

create_residual_heatmap(

    residual_table=hour_result[
        "adjusted_residuals_table"
    ],

    title=(
        "Adjusted standardized residuals - Hour × Target"
    ),

    output_path=HOUR_RESIDUALS_PATH
)


# ============================================================
# 35. MONTH-HOUR FRAUD RATE HEATMAP
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        16,
        8
    )
)


fraud_rate_image = ax.imshow(

    month_hour_fraud_percentage_table
    .to_numpy(
        dtype="float64"
    ),

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        24
    )
)


ax.set_xticklabels(
    HOUR_ORDER
)


ax.set_yticks(
    np.arange(
        12
    )
)


ax.set_yticklabels(
    [
        MONTH_LABELS[
            month
        ]
        for month in MONTH_ORDER
    ]
)


ax.set_xlabel(
    "Hour"
)


ax.set_ylabel(
    "Month"
)


ax.set_title(
    "Fraud rate (%) by month and hour"
)


fig.colorbar(

    fraud_rate_image,

    ax=ax,

    label="Fraud rate (%)"
)


fig.tight_layout()


fig.savefig(

    MONTH_HOUR_FRAUD_RATE_PATH,

    format="png",

    dpi=PNG_DPI,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 36. MONTH-HOUR FRAUD LIFT HEATMAP
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        16,
        8
    )
)


fraud_lift_image = ax.imshow(

    month_hour_fraud_lift_table
    .to_numpy(
        dtype="float64"
    ),

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        24
    )
)


ax.set_xticklabels(
    HOUR_ORDER
)


ax.set_yticks(
    np.arange(
        12
    )
)


ax.set_yticklabels(
    [
        MONTH_LABELS[
            month
        ]
        for month in MONTH_ORDER
    ]
)


ax.set_xlabel(
    "Hour"
)


ax.set_ylabel(
    "Month"
)


ax.set_title(
    "Fraud-rate lift by month and hour"
)


fig.colorbar(

    fraud_lift_image,

    ax=ax,

    label=(
        "Fraud rate / overall fraud rate"
    )
)


fig.tight_layout()


fig.savefig(

    MONTH_HOUR_FRAUD_LIFT_PATH,

    format="png",

    dpi=PNG_DPI,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 37. CIRCULAR SUMMARY POLAR PLOT FUNCTION
# ============================================================

def create_circular_summary_plot(
    circular_table,
    cycle_name,
    output_path
):

    cycle_data = (
        circular_table[
            circular_table[
                "CYCLE"
            ]
            == cycle_name
        ]
    )


    fig = plt.figure(
        figsize=(
            8,
            8
        )
    )


    ax = fig.add_subplot(
        111,
        projection="polar"
    )


    ax.set_theta_zero_location(
        "N"
    )


    ax.set_theta_direction(
        -1
    )


    for _, row in cycle_data.iterrows():

        angle = float(
            row[
                "MEAN_ANGLE_RADIANS"
            ]
        )


        resultant_length = float(
            row[
                "MEAN_RESULTANT_LENGTH"
            ]
        )


        ax.plot(

            [
                angle,
                angle
            ],

            [
                0,
                resultant_length
            ],

            marker="o",

            label=row[
                "TARGET_CLASS"
            ]
        )


    ax.set_ylim(
        0,
        1
    )


    ax.set_title(
        f"{cycle_name} circular mean direction by target"
    )


    ax.legend(
        loc="upper right",
        bbox_to_anchor=(
            1.25,
            1.15
        )
    )


    fig.tight_layout()


    fig.savefig(

        output_path,

        format="png",

        dpi=PNG_DPI,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 38. MONTH CIRCULAR SUMMARY PLOT
# ============================================================

create_circular_summary_plot(

    circular_table=circular_summary_table,

    cycle_name="Month",

    output_path=MONTH_CIRCULAR_SUMMARY_PATH
)


# ============================================================
# 39. HOUR CIRCULAR SUMMARY PLOT
# ============================================================

create_circular_summary_plot(

    circular_table=circular_summary_table,

    cycle_name="Hour",

    output_path=HOUR_CIRCULAR_SUMMARY_PATH
)


# ============================================================
# 40. IMAGE TO BASE64 FUNCTION
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 41. CONVERT PNG FILES TO BASE64
# ============================================================

month_fraud_rate_base64 = (
    image_to_base64(
        MONTH_FRAUD_RATE_PATH
    )
)


hour_fraud_rate_base64 = (
    image_to_base64(
        HOUR_FRAUD_RATE_PATH
    )
)


month_residuals_base64 = (
    image_to_base64(
        MONTH_RESIDUALS_PATH
    )
)


hour_residuals_base64 = (
    image_to_base64(
        HOUR_RESIDUALS_PATH
    )
)


month_hour_fraud_rate_base64 = (
    image_to_base64(
        MONTH_HOUR_FRAUD_RATE_PATH
    )
)


month_hour_fraud_lift_base64 = (
    image_to_base64(
        MONTH_HOUR_FRAUD_LIFT_PATH
    )
)


month_circular_summary_base64 = (
    image_to_base64(
        MONTH_CIRCULAR_SUMMARY_PATH
    )
)


hour_circular_summary_base64 = (
    image_to_base64(
        HOUR_CIRCULAR_SUMMARY_PATH
    )
)


# ============================================================
# 42. HTML TABLE PREPARATION
# ============================================================

target_overview_html = (
    target_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PERCENTAGE":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


cyclical_validation_html = (
    cyclical_validation_table
    .to_html(
        index=False,
        border=0,
        float_format=lambda value:
            f"{value:.8f}"
    )
)


reconstruction_overview_html = (
    reconstruction_overview_table
    .to_html(
        index=False,
        border=0
    )
)


month_fraud_rate_html = (
    month_result[
        "fraud_rate_table"
    ]
    .to_html(
        index=False,
        border=0,
        formatters={

            "FRAUD_RATE":
                lambda value:
                    f"{value:.8f}",

            "FRAUD_PERCENTAGE":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE_LIFT":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


hour_fraud_rate_html = (
    hour_result[
        "fraud_rate_table"
    ]
    .to_html(
        index=False,
        border=0,
        formatters={

            "FRAUD_RATE":
                lambda value:
                    f"{value:.8f}",

            "FRAUD_PERCENTAGE":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE_LIFT":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


month_residuals_html = (
    month_result[
        "adjusted_residuals_table"
    ]
    .to_html(
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


hour_residuals_html = (
    hour_result[
        "adjusted_residuals_table"
    ]
    .to_html(
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


circular_summary_html = (
    circular_summary_table
    .to_html(
        index=False,
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


circular_difference_html = (
    circular_difference_table
    .to_html(
        index=False,
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


association_comparison_html = (
    association_comparison_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "CHI_SQUARE":
                lambda value:
                    f"{value:.6f}",

            "P_VALUE":
                lambda value:
                    f"{value:.12g}",

            "CRAMERS_V":
                lambda value:
                    f"{value:.6f}",

            "MUTUAL_INFORMATION":
                lambda value:
                    f"{value:.8f}",

            "NORMALIZED_MUTUAL_INFORMATION":
                lambda value:
                    f"{value:.8f}",

            "ABS_CRAMERS_V":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


month_hour_fraud_rate_html = (
    month_hour_fraud_percentage_table
    .to_html(
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


month_hour_fraud_lift_html = (
    month_hour_fraud_lift_table
    .to_html(
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


# ============================================================
# 43. IDENTIFY STRONGEST TEMPORAL ASSOCIATION
# ============================================================

strongest_temporal_row = (
    association_comparison_table
    .iloc[
        0
    ]
)


strongest_temporal_structure = (
    strongest_temporal_row[
        "TEMPORAL_STRUCTURE"
    ]
)


strongest_temporal_cramers_v = float(
    strongest_temporal_row[
        "CRAMERS_V"
    ]
)


strongest_temporal_strength = (
    strongest_temporal_row[
        "CRAMERS_V_STRENGTH"
    ]
)


# ============================================================
# 44. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Cyclical Encoding - Target Association
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1500px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 13px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 8px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Target Association —
Cyclical Encoding Using Sine and Cosine
</h1>


<p>

Cyclical features analyzed:

</p>


<ul>

<li>{MONTH_SIN_FEATURE}</li>
<li>{MONTH_COS_FEATURE}</li>
<li>{HOUR_SIN_FEATURE}</li>
<li>{HOUR_COS_FEATURE}</li>

</ul>


<p>

Target:

<strong>
{TARGET_FEATURE}
</strong>

</p>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete finite observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. TARGET OVERVIEW
========================================================= -->


<h2>
1. Target overview
</h2>


<div class="table-container">

{target_overview_html}

</div>


<p class="result">

Overall fraud rate:
{fraud_percentage:.6f}%

</p>


<div class="note">

TARGET_OMEGA is used exclusively as an outcome variable
for exploratory association analysis.

It must not be included in the explanatory feature matrix
used to construct PCA, t-SNE or GMM.

</div>


<!-- ========================================================
     2. CYCLICAL VALIDATION AND RECONSTRUCTION
========================================================= -->


<h2>
2. Cyclical validation and temporary reconstruction
</h2>


<div class="table-container">

{cyclical_validation_html}

</div>


<div class="table-container">

{reconstruction_overview_html}

</div>


<div class="note">

The original sine and cosine components are preserved
without modification.

Month and hour are reconstructed temporarily only for
categorical target-association analysis.

<br><br>

Month reconstruction produces values from 1 to 12.

Hour reconstruction produces integer hours from 0 to 23.

The original hourly sine and cosine components still retain
minute and second information through the decimal-hour
angle.

</div>


<!-- ========================================================
     3. MONTH × TARGET
========================================================= -->


<h2>
3. Month × Target association
</h2>


<div class="table-container">

{month_fraud_rate_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{month_fraud_rate_base64}"
    alt="Fraud rate by month"
>

</div>


<p class="result">

Chi-square:
{month_result["chi_square_statistic"]:.6f}

<br>

Degrees of freedom:
{month_result["degrees_of_freedom"]}

<br>

p-value:
{month_result["chi_square_p_value"]:.12g}

<br>

Cramer's V:
{month_result["cramers_v"]:.6f}

<br>

Association strength:
{month_result["cramers_v_strength"]}

<br>

Normalized Mutual Information:
{month_result["normalized_mutual_information"]:.8f}

</p>


<div class="note">

The null hypothesis states that reconstructed month and
fraud status are independent.

Because the dataset is very large, the p-value should not
be interpreted as an effect-size measure.

Cramer's V and fraud-rate differences are more informative
for practical interpretation.

</div>


<h3>
Adjusted standardized residuals
</h3>


<div class="table-container">

{month_residuals_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{month_residuals_base64}"
    alt="Month adjusted standardized residuals"
>

</div>


<div class="note">

Positive residuals in the fraud column indicate months
with more fraud observations than expected under
independence.

Negative residuals indicate fewer fraud observations
than expected.

<br><br>

The exploratory reference:

<strong>
|residual| >= {STANDARDIZED_RESIDUAL_THRESHOLD:.1f}
</strong>

is used only for descriptive highlighting.

</div>


<!-- ========================================================
     4. HOUR × TARGET
========================================================= -->


<h2>
4. Hour × Target association
</h2>


<div class="table-container">

{hour_fraud_rate_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{hour_fraud_rate_base64}"
    alt="Fraud rate by hour"
>

</div>


<p class="result">

Chi-square:
{hour_result["chi_square_statistic"]:.6f}

<br>

Degrees of freedom:
{hour_result["degrees_of_freedom"]}

<br>

p-value:
{hour_result["chi_square_p_value"]:.12g}

<br>

Cramer's V:
{hour_result["cramers_v"]:.6f}

<br>

Association strength:
{hour_result["cramers_v_strength"]}

<br>

Normalized Mutual Information:
{hour_result["normalized_mutual_information"]:.8f}

</p>


<h3>
Adjusted standardized residuals
</h3>


<div class="table-container">

{hour_residuals_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{hour_residuals_base64}"
    alt="Hour adjusted standardized residuals"
>

</div>


<!-- ========================================================
     5. CIRCULAR CLASS SUMMARY
========================================================= -->


<h2>
5. Circular structure by target class
</h2>


<div class="table-container">

{circular_summary_html}

</div>


<div class="table-container">

{circular_difference_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{month_circular_summary_base64}"
    alt="Month circular summary"
>

</div>


<div class="chart">

<img
    src="data:image/png;base64,{hour_circular_summary_base64}"
    alt="Hour circular summary"
>

</div>


<div class="note">

The circular mean direction summarizes the average angular
position of each target class.

The mean resultant length ranges from 0 to 1.

Values near 1 indicate greater concentration around the
mean direction.

Values near 0 indicate that observations are broadly
distributed around the cycle.

<br><br>

These circular summaries are descriptive.

They do not replace the categorical Month × Target and
Hour × Target association analyses.

</div>


<!-- ========================================================
     6. MONTH × HOUR × TARGET
========================================================= -->


<h2>
6. Joint Month × Hour target structure
</h2>


<p>

The following analysis evaluates the fraud rate for every
observed combination of reconstructed month and hour.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{month_hour_fraud_rate_base64}"
    alt="Month hour fraud rate"
>

</div>


<div class="table-container">

{month_hour_fraud_rate_html}

</div>


<h3>
Fraud-rate lift
</h3>


<p>

Fraud-rate lift is defined as:

</p>


<p class="result">

cell fraud rate / overall fraud rate

</p>


<p>

A value greater than 1 indicates a temporal cell with a
fraud rate above the dataset-wide fraud rate.

A value below 1 indicates a fraud rate below the overall
rate.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{month_hour_fraud_lift_base64}"
    alt="Month hour fraud-rate lift"
>

</div>


<div class="table-container">

{month_hour_fraud_lift_html}

</div>


<p class="result">

Joint Month-Hour Cramer's V:
{month_hour_result["cramers_v"]:.6f}

<br>

Strength:
{month_hour_result["cramers_v_strength"]}

<br>

Joint Month-Hour Normalized Mutual Information:
{month_hour_result["normalized_mutual_information"]:.8f}

</p>


<div class="note">

The joint Month-Hour category contains up to 288 temporal
cells.

This analysis can identify temporal target structure that
may be hidden when month and hour are examined separately.

No new Month-Hour feature is permanently created in the
dataset.

</div>


<!-- ========================================================
     7. ASSOCIATION COMPARISON
========================================================= -->


<h2>
7. Temporal target-association comparison
</h2>


<div class="table-container">

{association_comparison_html}

</div>


<p class="result">

Strongest temporal association:

<br>

{strongest_temporal_structure}

<br>

Cramer's V:
{strongest_temporal_cramers_v:.6f}

<br>

Strength:
{strongest_temporal_strength}

</p>


<div class="note">

Chi-square significance, Cramer's V and Mutual Information
answer complementary questions.

<br><br>

Chi-square evaluates whether statistical independence is
plausible.

Cramer's V measures the magnitude of categorical
association.

Mutual Information measures shared information without
requiring a linear or monotonic relationship.

</div>


<!-- ========================================================
     8. MODELING IMPLICATIONS
========================================================= -->


<h2>
8. Potential modeling implications
</h2>


<div class="note">

<strong>Preserve cyclical structure:</strong>

<br><br>

Raw month numbers and raw hour numbers should not replace
the sine-cosine representation in PCA, t-SNE or GMM without
careful justification.

The sine-cosine representation preserves continuity across
the cycle boundaries:

December ↔ January

and

23:59 ↔ 00:00.

</div>


<div class="note">

<strong>Paired components:</strong>

<br><br>

TRANS_MONTH_SIN and TRANS_MONTH_COS jointly represent one
cyclical variable.

TRANS_HOUR_SIN and TRANS_HOUR_COS jointly represent another
cyclical variable.

The components should therefore be interpreted as pairs
rather than as four unrelated numerical measurements.

</div>


<div class="note">

<strong>Deterministic structure:</strong>

<br><br>

Each sine-cosine pair approximately satisfies:

sin² + cos² = 1.

This deterministic geometric structure should be remembered
when reviewing covariance, PCA loadings and GMM covariance
matrices.

</div>


<div class="note">

<strong>Target association:</strong>

<br><br>

A month, hour or Month-Hour pattern associated with fraud
does not mean that TARGET_OMEGA should be used to construct
the unsupervised representation.

The target remains excluded from PCA, t-SNE and GMM.

These results are documented for interpretation and for the
final pre-modeling audit.

</div>


<div class="note">

<strong>Large sample size:</strong>

<br><br>

With approximately {analysis_observations:,} observations,
very small temporal effects may generate statistically
significant chi-square tests.

Effect sizes, fraud-rate patterns and lift should therefore
receive more interpretive emphasis than p-values alone.

</div>


<!-- ========================================================
     9. SUMMARY
========================================================= -->


<h2>
9. Summary
</h2>


<p class="result">

Cyclical source features analyzed:
{len(CYCLICAL_FEATURES)}

</p>


<p class="result">

Overall fraud rate:
{fraud_percentage:.6f}%

</p>


<p class="result">

Month Cramer's V:
{month_result["cramers_v"]:.6f}

</p>


<p class="result">

Hour Cramer's V:
{hour_result["cramers_v"]:.6f}

</p>


<p class="result">

Joint Month-Hour Cramer's V:
{month_hour_result["cramers_v"]:.6f}

</p>


<p class="result">

Strongest temporal structure:
{strongest_temporal_structure}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The original sine and cosine components were preserved
without modification.

Month and hour were reconstructed temporarily to study
their association with fraud.

<br><br>

Month × Target and Hour × Target relationships were
evaluated using fraud rates, chi-square tests, Cramer's V,
Adjusted Standardized Residuals and Mutual Information.

<br><br>

Circular summaries were additionally used to compare the
temporal concentration and average direction of fraud and
non-fraud observations while respecting the cyclical
geometry.

<br><br>

The joint Month-Hour fraud-rate surface evaluates temporal
interaction patterns across up to 288 combinations.

<br><br>

No cyclical feature is removed or permanently transformed
during this exploratory stage.

All results should be revisited during the final
pre-modeling audit before PCA, t-SNE and GMM.

</div>


</body>

</html>
"""


# ============================================================
# 45. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 46. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CYCLICAL ENCODING USING SINE AND COSINE - TARGET ASSOCIATION"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete finite observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 47. DISPLAY TARGET OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TARGET OVERVIEW"
)


print(
    "=" * 100
)


display(
    target_overview_table
)


# ============================================================
# 48. DISPLAY VALIDATION AND RECONSTRUCTION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CYCLICAL VALIDATION"
)


print(
    "=" * 100
)


display(
    cyclical_validation_table
)


print(
    "\nTemporary reconstruction:"
)


display(
    reconstruction_overview_table
)


# ============================================================
# 49. DISPLAY MONTH ASSOCIATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "MONTH × TARGET"
)


print(
    "=" * 100
)


display(
    month_result[
        "fraud_rate_table"
    ]
)


print(
    "\nChi-square:",
    f'{month_result["chi_square_statistic"]:.6f}'
)


print(
    "p-value:",
    f'{month_result["chi_square_p_value"]:.12g}'
)


print(
    "Cramer's V:",
    f'{month_result["cramers_v"]:.6f}'
)


print(
    "Strength:",
    month_result[
        "cramers_v_strength"
    ]
)


print(
    "Normalized Mutual Information:",
    f'{month_result["normalized_mutual_information"]:.8f}'
)


print(
    "\nAdjusted standardized residuals:"
)


display(
    month_result[
        "adjusted_residuals_table"
    ]
)


# ============================================================
# 50. DISPLAY HOUR ASSOCIATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "HOUR × TARGET"
)


print(
    "=" * 100
)


display(
    hour_result[
        "fraud_rate_table"
    ]
)


print(
    "\nChi-square:",
    f'{hour_result["chi_square_statistic"]:.6f}'
)


print(
    "p-value:",
    f'{hour_result["chi_square_p_value"]:.12g}'
)


print(
    "Cramer's V:",
    f'{hour_result["cramers_v"]:.6f}'
)


print(
    "Strength:",
    hour_result[
        "cramers_v_strength"
    ]
)


print(
    "Normalized Mutual Information:",
    f'{hour_result["normalized_mutual_information"]:.8f}'
)


print(
    "\nAdjusted standardized residuals:"
)


display(
    hour_result[
        "adjusted_residuals_table"
    ]
)


# ============================================================
# 51. DISPLAY CIRCULAR SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CIRCULAR SUMMARY BY TARGET"
)


print(
    "=" * 100
)


display(
    circular_summary_table
)


print(
    "\nCircular class differences:"
)


display(
    circular_difference_table
)


# ============================================================
# 52. DISPLAY MONTH-HOUR ASSOCIATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "JOINT MONTH-HOUR × TARGET"
)


print(
    "=" * 100
)


print(
    "\nCramer's V:",
    f'{month_hour_result["cramers_v"]:.6f}'
)


print(
    "Strength:",
    month_hour_result[
        "cramers_v_strength"
    ]
)


print(
    "Normalized Mutual Information:",
    f'{month_hour_result["normalized_mutual_information"]:.8f}'
)


print(
    "\nMonth-hour fraud rate (%):"
)


display(
    month_hour_fraud_percentage_table
)


print(
    "\nMonth-hour fraud-rate lift:"
)


display(
    month_hour_fraud_lift_table
)


# ============================================================
# 53. DISPLAY ASSOCIATION COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TEMPORAL TARGET-ASSOCIATION COMPARISON"
)


print(
    "=" * 100
)


display(
    association_comparison_table
)


print(
    "\nStrongest temporal association:"
)


print(
    strongest_temporal_structure
)


print(
    "Cramer's V:",
    f"{strongest_temporal_cramers_v:.6f}"
)


print(
    "Strength:",
    strongest_temporal_strength
)


# ============================================================
# 54. RELEASE MEMORY
# ============================================================

del dataset_cyclical_target
del working_data
del complete_data

del month_sin_values
del month_cos_values

del hour_sin_values
del hour_cos_values

del fraud_only_data

gc.collect()


# ============================================================
# 55. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    MONTH_FRAUD_RATE_PATH
)


print(
    HOUR_FRAUD_RATE_PATH
)


print(
    MONTH_RESIDUALS_PATH
)


print(
    HOUR_RESIDUALS_PATH
)


print(
    MONTH_HOUR_FRAUD_RATE_PATH
)


print(
    MONTH_HOUR_FRAUD_LIFT_PATH
)


print(
    MONTH_CIRCULAR_SUMMARY_PATH
)


print(
    HOUR_CIRCULAR_SUMMARY_PATH
)


CYCLICAL ENCODING USING SINE AND COSINE - TARGET ASSOCIATION

Total dataset observations: 1852394
Complete finite observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

TARGET OVERVIEW


,TARGET_CLASS,TARGET_VALUE,COUNT,PERCENTAGE
0,Non-fraud,0,1842743,99.478999
1,Fraud,1,9651,0.521001



CYCLICAL VALIDATION


,FEATURE,MIN,MAX,VALUES_OUTSIDE_EXPECTED_RANGE
0,TRANS_MONTH_SIN,-1.0,1.0,0
1,TRANS_MONTH_COS,-1.0,1.0,0
2,TRANS_HOUR_SIN,-1.0,1.0,0
3,TRANS_HOUR_COS,-1.0,1.0,0



Temporary reconstruction:


...


Month-hour fraud-rate lift:


_RECONSTRUCTED_HOUR,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
_RECONSTRUCTED_MONTH,,,,,,,,,,,,,,,,,,,,,
1,3.964970,4.095725,4.245164,4.327739,0.693543,0.225147,0.333033,0.667224,0.402748,0.568705,...,0.289827,0.525666,0.361942,0.185054,0.289172,0.357893,0.109159,0.320847,6.747548,7.697386
2,4.555866,5.628515,4.244036,3.582047,0.062684,0.357871,0.307199,0.422904,0.184319,0.364324,...,0.592767,0.236086,0.195975,0.386816,0.346667,0.313752,0.312031,0.518967,8.248208,7.652015
3,3.318347,3.677280,2.600451,2.653704,0.162144,0.209494,0.251502,0.379407,0.246285,0.332001,...,0.239789,0.323446,0.366144,0.258921,0.231406,0.318569,0.316164,0.160282,6.227846,6.090834
4,2.403119,2.649518,2.668795,2.779822,0.175687,0.355276,0.131315,0.000000,0.214552,0.171068,...,0.169757,0.250064,0.140203,0.168885,0.253514,0.309441,0.142726,0.110979,4.432940,4.889091
5,3.232221,3.176094,3.531880,3.174149,0.365364,0.321706,0.361012,0.159285,0.158006,0.282975,...,0.182997,0.210199,0.231561,0.103905,0.256020,0.212497,0.181883,0.154643,5.813293,6.180927
6,1.930411,2.320730,2.682786,2.354051,0.068746,0.273757,0.203828,0.237673,0.133452,0.067847,...,0.152834,0.128832,0.240305,0.284742,0.282838,0.173523,0.152470,0.197942,4.580193,4.450241
7,1.682191,1.972782,1.247884,1.982266,0.202040,0.067835,0.033834,0.066715,0.202360,0.134434,...,0.152418,0.221356,0.066262,0.133306,0.173445,0.088075,0.110589,0.178880,3.966573,3.833503
8,2.596456,2.054383,2.900997,2.165085,0.032643,0.202609,0.134457,0.375079,0.235507,0.168603,...,0.130807,0.107456,0.148608,0.172956,0.193725,0.194247,0.150692,0.172393,4.185565,4.053086
9,2.814466,2.436915,2.057214,2.769728,0.170309,0.253998,0.214456,0.415630,0.212133,0.208946,...,0.242414,0.277367,0.213918,0.323264,0.244750,0.192543,0.326703,0.265218,5.281964,4.714268



TEMPORAL TARGET-ASSOCIATION COMPARISON


,TEMPORAL_STRUCTURE,CHI_SQUARE,DEGREES_OF_FREEDOM,P_VALUE,CRAMERS_V,CRAMERS_V_STRENGTH,MUTUAL_INFORMATION,NORMALIZED_MUTUAL_INFORMATION,ABS_CRAMERS_V
0,Month-Hour joint cell,26360.210016,287,0.000000e+00,0.119291,Weak,0.004943,0.001755,0.119291
1,Hour,23386.696711,23,0.000000e+00,0.112362,Weak,0.004641,0.002912,0.112362
2,Month,865.679577,11,1.476427e-178,0.021618,Very weak or negligible,0.000229,0.000185,0.021618



Strongest temporal association:
Month-Hour joint cell
Cramer's V: 0.119291
Strength: Weak

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/cyclical_encoding_using_sine_and_cosine

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/cyclical_encoding_using_sine_and_cosine/analysis_cyclical_encoding_target_association.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/cyclical_encoding_using_sine_and_cosine/cyclical_target_month_fraud_rate.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/cyclical_encoding_using_sine_and_cosine/cyclical_target_hour_fraud_rate.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_wi

## <span style="color:PINK"> DISCRETE NUMERAL AND CONTINUOS NUMERAL </span> ##

In [10]:


# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "02_relationships_with_target"
)

ASSOCIATION_GROUP = (
    "target_association"
)

FEATURE_GROUP = (
    "discrete_numera_and_continuos_numeral"
)


AGE_FEATURE = (
    "SEND_AGE"
)

VALUE_FEATURE = (
    "TRANS_VALUE"
)

DAY_FEATURE = (
    "TRANS_DAY"
)

POPULATION_FEATURE = (
    "SEND_POP_REGISTER"
)

TARGET_FEATURE = (
    "TARGET_OMEGA"
)


CONTINUOUS_NUMERAL_FEATURES = [
    AGE_FEATURE,
    VALUE_FEATURE
]


DISCRETE_NUMERAL_FEATURES = [
    DAY_FEATURE,
    POPULATION_FEATURE
]


NUMERICAL_FEATURES = (
    CONTINUOUS_NUMERAL_FEATURES
    +
    DISCRETE_NUMERAL_FEATURES
)


REQUIRED_FEATURES = (
    NUMERICAL_FEATURES
    +
    [
        TARGET_FEATURE
    ]
)


TARGET_POSITIVE_VALUE = 1

ALPHA = 0.05

QUANTILE_GROUPS = 10

PNG_DPI = 300


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / ASSOCIATION_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_discrete_and_continuous_numerical_target_association.html"
)


AGE_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "numerical_target_send_age_fraud_rate.png"
)


VALUE_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "numerical_target_trans_value_fraud_rate.png"
)


DAY_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "numerical_target_trans_day_fraud_rate.png"
)


POPULATION_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "numerical_target_send_population_fraud_rate.png"
)


RANK_BISERIAL_PATH = (
    RESULTS_DIRECTORY
    / "numerical_target_rank_biserial_effect.png"
)


FEATURE_PLOT_PATHS = {

    AGE_FEATURE:
        AGE_FRAUD_RATE_PATH,

    VALUE_FEATURE:
        VALUE_FRAUD_RATE_PATH,

    DAY_FEATURE:
        DAY_FRAUD_RATE_PATH,

    POPULATION_FEATURE:
        POPULATION_FRAUD_RATE_PATH
}


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD REQUIRED FEATURES
# ============================================================

dataset_numerical_target = pd.read_parquet(
    DATASET_PATH,
    columns=REQUIRED_FEATURES
)


total_observations = int(
    len(
        dataset_numerical_target
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. VALIDATE REQUIRED FEATURES
# ============================================================

missing_features = [
    feature
    for feature in REQUIRED_FEATURES
    if feature not in dataset_numerical_target.columns
]


if missing_features:

    raise KeyError(
        "Missing required features: "
        + ", ".join(
            missing_features
        )
    )


# ============================================================
# 09. TEMPORARY NUMERICAL COPY
#
# Numerical coercion is performed only in memory.
#
# The original parquet dataset is never modified.
# ============================================================

working_data = (
    dataset_numerical_target
    .copy()
)


validation_records = []


for feature in NUMERICAL_FEATURES:

    source_series = (
        working_data[
            feature
        ]
    )


    numeric_series = pd.to_numeric(
        source_series,
        errors="coerce"
    )


    invalid_non_numeric_mask = (
        source_series.notna()
        &
        numeric_series.isna()
    )


    invalid_non_numeric_count = int(
        invalid_non_numeric_mask.sum()
    )


    if invalid_non_numeric_count > 0:

        invalid_examples = (
            source_series.loc[
                invalid_non_numeric_mask
            ]
            .drop_duplicates()
            .head(
                10
            )
            .tolist()
        )


        raise TypeError(
            f"{feature} contains non-numeric values. "
            f"Invalid observations: {invalid_non_numeric_count}. "
            f"Examples: {invalid_examples}"
        )


    working_data[
        feature
    ] = (
        numeric_series.astype(
            "float64"
        )
    )


    finite_values = (
        numeric_series
        .dropna()
        .to_numpy(
            dtype="float64"
        )
    )


    non_finite_count = int(
        np.sum(
            ~np.isfinite(
                finite_values
            )
        )
    )


    feature_type = (
        "Continuous numeral"
        if feature in CONTINUOUS_NUMERAL_FEATURES
        else "Discrete numeral"
    )


    validation_records.append({

        "FEATURE":
            feature,

        "FEATURE_TYPE":
            feature_type,

        "SOURCE_DATA_TYPE":
            str(
                source_series.dtype
            ),

        "MISSING_VALUES":
            int(
                source_series.isna().sum()
            ),

        "NON_NUMERIC_VALUES":
            invalid_non_numeric_count,

        "NON_FINITE_VALUES":
            non_finite_count,

        "UNIQUE_VALUES":
            int(
                numeric_series.nunique(
                    dropna=True
                )
            )
    })


validation_table = pd.DataFrame(
    validation_records
)


# ============================================================
# 10. PREPARE COMMON COMPLETE FINITE SAMPLE
#
# All statistical analyses use the same complete and
# finite observations.
# ============================================================

complete_data = (
    working_data[
        REQUIRED_FEATURES
    ]
    .dropna()
    .copy()
)


finite_mask = np.isfinite(
    complete_data[
        NUMERICAL_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


analysis_data = (
    complete_data.loc[
        finite_mask,
        REQUIRED_FEATURES
    ]
    .copy()
)


analysis_observations = int(
    len(
        analysis_data
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite numerical observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 11. BASIC NUMERICAL RANGE VALIDATION
# ============================================================

range_validation_records = []


for feature in NUMERICAL_FEATURES:

    feature_values = (
        analysis_data[
            feature
        ]
    )


    range_validation_records.append({

        "FEATURE":
            feature,

        "MIN":
            float(
                feature_values.min()
            ),

        "MAX":
            float(
                feature_values.max()
            ),

        "NEGATIVE_VALUES":
            int(
                (
                    feature_values
                    < 0
                )
                .sum()
            )
    })


range_validation_table = pd.DataFrame(
    range_validation_records
)


# ============================================================
# 12. SPECIFIC DOMAIN VALIDATION
# ============================================================

specific_validation_table = pd.DataFrame({

    "CHECK": [
        "SEND_AGE below 0",
        "SEND_AGE above 100",
        "TRANS_VALUE below 0",
        "TRANS_DAY below 1",
        "TRANS_DAY above 31",
        "SEND_POP_REGISTER below 0"
    ],

    "INVALID_COUNT": [

        int(
            (
                analysis_data[
                    AGE_FEATURE
                ]
                < 0
            )
            .sum()
        ),

        int(
            (
                analysis_data[
                    AGE_FEATURE
                ]
                > 100
            )
            .sum()
        ),

        int(
            (
                analysis_data[
                    VALUE_FEATURE
                ]
                < 0
            )
            .sum()
        ),

        int(
            (
                analysis_data[
                    DAY_FEATURE
                ]
                < 1
            )
            .sum()
        ),

        int(
            (
                analysis_data[
                    DAY_FEATURE
                ]
                > 31
            )
            .sum()
        ),

        int(
            (
                analysis_data[
                    POPULATION_FEATURE
                ]
                < 0
            )
            .sum()
        )
    ]
})


# ============================================================
# 13. VALIDATE TARGET
# ============================================================

target_values = (
    analysis_data[
        TARGET_FEATURE
    ]
    .drop_duplicates()
    .tolist()
)


if len(
    target_values
) != 2:

    raise ValueError(
        f"{TARGET_FEATURE} must contain exactly two classes. "
        f"Observed values: {target_values}"
    )


target_positive_value = None


for value in target_values:

    if (
        value == TARGET_POSITIVE_VALUE
        or
        str(
            value
        )
        == str(
            TARGET_POSITIVE_VALUE
        )
    ):

        target_positive_value = (
            value
        )

        break


if target_positive_value is None:

    raise ValueError(
        f"Fraud target value {TARGET_POSITIVE_VALUE} was not found. "
        f"Observed values: {target_values}"
    )


target_negative_values = [
    value
    for value in target_values
    if value != target_positive_value
]


if len(
    target_negative_values
) != 1:

    raise ValueError(
        "Unable to determine the non-fraud target class."
    )


target_negative_value = (
    target_negative_values[
        0
    ]
)


# ============================================================
# 14. CREATE TEMPORARY BINARY TARGET
#
# Non-fraud = 0
# Fraud     = 1
#
# Temporary only.
# ============================================================

analysis_data[
    "_TARGET_BINARY"
] = (
    analysis_data[
        TARGET_FEATURE
    ]
    .map({

        target_negative_value:
            0,

        target_positive_value:
            1
    })
    .astype(
        "int8"
    )
)


# ============================================================
# 15. TARGET OVERVIEW
# ============================================================

target_counts = (
    analysis_data[
        "_TARGET_BINARY"
    ]
    .value_counts()
    .reindex(
        [
            0,
            1
        ],
        fill_value=0
    )
)


non_fraud_count = int(
    target_counts.loc[
        0
    ]
)


fraud_count = int(
    target_counts.loc[
        1
    ]
)


non_fraud_percentage = (
    non_fraud_count
    / analysis_observations
    * 100
)


fraud_percentage = (
    fraud_count
    / analysis_observations
    * 100
)


overall_fraud_rate = (
    fraud_count
    / analysis_observations
)


target_overview_table = pd.DataFrame({

    "TARGET_CLASS": [
        "Non-fraud",
        "Fraud"
    ],

    "TARGET_VALUE": [
        target_negative_value,
        target_positive_value
    ],

    "COUNT": [
        non_fraud_count,
        fraud_count
    ],

    "PERCENTAGE": [
        non_fraud_percentage,
        fraud_percentage
    ]
})


# ============================================================
# 16. DESCRIPTIVE STATISTICS BY TARGET
# ============================================================

descriptive_records = []


for feature in NUMERICAL_FEATURES:

    for target_code, target_label in [
        (
            0,
            "Non-fraud"
        ),
        (
            1,
            "Fraud"
        )
    ]:

        values = (
            analysis_data.loc[
                analysis_data[
                    "_TARGET_BINARY"
                ]
                == target_code,
                feature
            ]
            .to_numpy(
                dtype="float64"
            )
        )


        q01 = float(
            np.quantile(
                values,
                0.01
            )
        )


        q05 = float(
            np.quantile(
                values,
                0.05
            )
        )


        q25 = float(
            np.quantile(
                values,
                0.25
            )
        )


        q50 = float(
            np.quantile(
                values,
                0.50
            )
        )


        q75 = float(
            np.quantile(
                values,
                0.75
            )
        )


        q95 = float(
            np.quantile(
                values,
                0.95
            )
        )


        q99 = float(
            np.quantile(
                values,
                0.99
            )
        )


        descriptive_records.append({

            "FEATURE":
                feature,

            "TARGET_CLASS":
                target_label,

            "COUNT":
                int(
                    len(
                        values
                    )
                ),

            "MEAN":
                float(
                    np.mean(
                        values
                    )
                ),

            "STANDARD_DEVIATION":
                float(
                    np.std(
                        values,
                        ddof=1
                    )
                ),

            "VARIANCE":
                float(
                    np.var(
                        values,
                        ddof=1
                    )
                ),

            "MIN":
                float(
                    np.min(
                        values
                    )
                ),

            "P01":
                q01,

            "P05":
                q05,

            "P25":
                q25,

            "MEDIAN":
                q50,

            "P75":
                q75,

            "P95":
                q95,

            "P99":
                q99,

            "MAX":
                float(
                    np.max(
                        values
                    )
                ),

            "IQR":
                float(
                    q75
                    - q25
                ),

            "SKEWNESS":
                float(
                    stats.skew(
                        values,
                        bias=False
                    )
                ),

            "EXCESS_KURTOSIS":
                float(
                    stats.kurtosis(
                        values,
                        fisher=True,
                        bias=False
                    )
                )
        })


descriptive_statistics_table = pd.DataFrame(
    descriptive_records
)


# ============================================================
# 17. EFFECT STRENGTH INTERPRETATION
#
# Used for correlation-type effect measures.
# ============================================================

def interpret_effect_strength(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        return (
            "Weak"
        )


    elif absolute_value < 0.50:

        return (
            "Moderate"
        )


    elif absolute_value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


# ============================================================
# 18. ASSOCIATION TESTS AGAINST TARGET
#
# For every numerical variable:
#
# - Mean difference
# - Median difference
# - Welch t-test
# - Mann-Whitney U
# - Point-biserial correlation
# - Rank-biserial correlation
#
# Fraud is treated as the first group in Mann-Whitney.
#
# Therefore:
#
# positive rank-biserial
# -> fraud observations tend to have larger values
#
# negative rank-biserial
# -> fraud observations tend to have smaller values
# ============================================================

association_records = []


target_binary_array = (
    analysis_data[
        "_TARGET_BINARY"
    ]
    .to_numpy(
        dtype="int8"
    )
)


for feature in NUMERICAL_FEATURES:

    non_fraud_values = (
        analysis_data.loc[
            analysis_data[
                "_TARGET_BINARY"
            ]
            == 0,
            feature
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    fraud_values = (
        analysis_data.loc[
            analysis_data[
                "_TARGET_BINARY"
            ]
            == 1,
            feature
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    # --------------------------------------------------------
    # LOCATION DIFFERENCES
    # --------------------------------------------------------

    mean_difference = float(
        np.mean(
            fraud_values
        )
        -
        np.mean(
            non_fraud_values
        )
    )


    median_difference = float(
        np.median(
            fraud_values
        )
        -
        np.median(
            non_fraud_values
        )
    )


    # --------------------------------------------------------
    # WELCH T-TEST
    # --------------------------------------------------------

    welch_result = stats.ttest_ind(

        fraud_values,

        non_fraud_values,

        equal_var=False
    )


    welch_statistic = float(
        welch_result.statistic
    )


    welch_p_value = float(
        welch_result.pvalue
    )


    # --------------------------------------------------------
    # MANN-WHITNEY U
    # --------------------------------------------------------

    mann_whitney_result = stats.mannwhitneyu(

        fraud_values,

        non_fraud_values,

        alternative="two-sided",

        method="asymptotic"
    )


    mann_whitney_u = float(
        mann_whitney_result.statistic
    )


    mann_whitney_p_value = float(
        mann_whitney_result.pvalue
    )


    n_fraud = float(
        len(
            fraud_values
        )
    )


    n_non_fraud = float(
        len(
            non_fraud_values
        )
    )


    common_language_probability = float(
        mann_whitney_u
        /
        (
            n_fraud
            *
            n_non_fraud
        )
    )


    rank_biserial = float(
        2
        *
        common_language_probability
        -
        1
    )


    rank_biserial_strength = (
        interpret_effect_strength(
            rank_biserial
        )
    )


    # --------------------------------------------------------
    # POINT-BISERIAL CORRELATION
    # --------------------------------------------------------

    feature_values = (
        analysis_data[
            feature
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    point_biserial_result = stats.pointbiserialr(

        target_binary_array,

        feature_values
    )


    point_biserial = float(
        point_biserial_result.statistic
    )


    point_biserial_p_value = float(
        point_biserial_result.pvalue
    )


    point_biserial_strength = (
        interpret_effect_strength(
            point_biserial
        )
    )


    association_records.append({

        "FEATURE":
            feature,

        "FEATURE_TYPE":
            (
                "Continuous numeral"
                if feature in CONTINUOUS_NUMERAL_FEATURES
                else "Discrete numeral"
            ),

        "MEAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
            mean_difference,

        "MEDIAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
            median_difference,

        "WELCH_T":
            welch_statistic,

        "WELCH_P_VALUE":
            welch_p_value,

        "MANN_WHITNEY_U":
            mann_whitney_u,

        "MANN_WHITNEY_P_VALUE":
            mann_whitney_p_value,

        "POINT_BISERIAL":
            point_biserial,

        "POINT_BISERIAL_P_VALUE":
            point_biserial_p_value,

        "POINT_BISERIAL_STRENGTH":
            point_biserial_strength,

        "RANK_BISERIAL":
            rank_biserial,

        "RANK_BISERIAL_STRENGTH":
            rank_biserial_strength,

        "COMMON_LANGUAGE_PROBABILITY":
            common_language_probability
    })


association_table = pd.DataFrame(
    association_records
)


# ============================================================
# 19. ASSOCIATION RANKING
#
# Rank-biserial correlation is used as the main robust
# effect-size ranking because it does not require normality.
# ============================================================

association_ranking_table = (
    association_table[
        [
            "FEATURE",
            "FEATURE_TYPE",
            "POINT_BISERIAL",
            "POINT_BISERIAL_STRENGTH",
            "RANK_BISERIAL",
            "RANK_BISERIAL_STRENGTH",
            "COMMON_LANGUAGE_PROBABILITY"
        ]
    ]
    .copy()
)


association_ranking_table[
    "ABS_RANK_BISERIAL"
] = (
    association_ranking_table[
        "RANK_BISERIAL"
    ]
    .abs()
)


association_ranking_table = (
    association_ranking_table
    .sort_values(
        by="ABS_RANK_BISERIAL",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 20. QUANTILE FRAUD-RATE FUNCTION
#
# Quantile groups are descriptive only.
#
# They are not permanently added to the dataset.
# ============================================================

def create_quantile_fraud_rate_table(
    dataframe,
    feature,
    number_quantiles=10
):

    temporary_bins = pd.qcut(

        dataframe[
            feature
        ],

        q=number_quantiles,

        duplicates="drop"
    )


    temporary_table = pd.DataFrame({

        "BIN":
            temporary_bins,

        "FEATURE_VALUE":
            dataframe[
                feature
            ],

        "TARGET":
            dataframe[
                "_TARGET_BINARY"
            ]
    })


    summary = (
        temporary_table
        .groupby(
            "BIN",
            observed=True
        )
        .agg(

            COUNT=(
                "TARGET",
                "size"
            ),

            FRAUD_COUNT=(
                "TARGET",
                "sum"
            ),

            VALUE_MIN=(
                "FEATURE_VALUE",
                "min"
            ),

            VALUE_MEDIAN=(
                "FEATURE_VALUE",
                "median"
            ),

            VALUE_MAX=(
                "FEATURE_VALUE",
                "max"
            )
        )
        .reset_index()
    )


    summary[
        "FRAUD_RATE"
    ] = (
        summary[
            "FRAUD_COUNT"
        ]
        /
        summary[
            "COUNT"
        ]
    )


    summary[
        "FRAUD_PERCENTAGE"
    ] = (
        summary[
            "FRAUD_RATE"
        ]
        * 100
    )


    summary[
        "FRAUD_RATE_LIFT"
    ] = (
        summary[
            "FRAUD_RATE"
        ]
        /
        overall_fraud_rate
    )


    summary[
        "GROUP"
    ] = np.arange(
        1,
        len(
            summary
        )
        + 1
    )


    summary[
        "BIN"
    ] = (
        summary[
            "BIN"
        ]
        .astype(
            str
        )
    )


    return (
        summary
    )


# ============================================================
# 21. CREATE QUANTILE FRAUD-RATE TABLES
#
# TRANS_DAY is treated separately because the original
# discrete values 1-31 are directly interpretable.
# ============================================================

quantile_feature_tables = {}


for feature in [
    AGE_FEATURE,
    VALUE_FEATURE,
    POPULATION_FEATURE
]:

    quantile_feature_tables[
        feature
    ] = (
        create_quantile_fraud_rate_table(

            dataframe=analysis_data,

            feature=feature,

            number_quantiles=QUANTILE_GROUPS
        )
    )


# ============================================================
# 22. TRANS_DAY FRAUD RATE
# ============================================================

day_fraud_rate_table = (
    analysis_data
    .groupby(
        DAY_FEATURE,
        observed=True
    )
    .agg(

        COUNT=(
            "_TARGET_BINARY",
            "size"
        ),

        FRAUD_COUNT=(
            "_TARGET_BINARY",
            "sum"
        )
    )
    .reset_index()
)


day_fraud_rate_table[
    "FRAUD_RATE"
] = (
    day_fraud_rate_table[
        "FRAUD_COUNT"
    ]
    /
    day_fraud_rate_table[
        "COUNT"
    ]
)


day_fraud_rate_table[
    "FRAUD_PERCENTAGE"
] = (
    day_fraud_rate_table[
        "FRAUD_RATE"
    ]
    * 100
)


day_fraud_rate_table[
    "FRAUD_RATE_LIFT"
] = (
    day_fraud_rate_table[
        "FRAUD_RATE"
    ]
    /
    overall_fraud_rate
)


day_fraud_rate_table = (
    day_fraud_rate_table
    .sort_values(
        by=DAY_FEATURE
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 23. FRAUD RATE PLOT FUNCTION
# ============================================================

def create_fraud_rate_plot(
    x_values,
    fraud_percentages,
    title,
    x_label,
    output_path,
    x_tick_labels=None
):

    fig, ax = plt.subplots(
        figsize=(
            11,
            7
        )
    )


    ax.plot(

        x_values,

        fraud_percentages,

        marker="o"
    )


    ax.axhline(

        overall_fraud_rate
        * 100,

        linestyle="--",

        linewidth=1.5,

        label="Overall fraud rate"
    )


    if x_tick_labels is not None:

        ax.set_xticks(
            x_values
        )


        ax.set_xticklabels(
            x_tick_labels,
            rotation=45,
            ha="right"
        )


    ax.set_xlabel(
        x_label
    )


    ax.set_ylabel(
        "Fraud rate (%)"
    )


    ax.set_title(
        title
    )


    ax.legend()


    ax.grid(
        alpha=0.20
    )


    fig.tight_layout()


    fig.savefig(
        output_path,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 24. CREATE AGE FRAUD-RATE PLOT
# ============================================================

age_quantile_table = (
    quantile_feature_tables[
        AGE_FEATURE
    ]
)


create_fraud_rate_plot(

    x_values=age_quantile_table[
        "GROUP"
    ],

    fraud_percentages=age_quantile_table[
        "FRAUD_PERCENTAGE"
    ],

    title=(
        "Fraud rate across SEND_AGE quantile groups"
    ),

    x_label="SEND_AGE quantile group",

    output_path=AGE_FRAUD_RATE_PATH
)


# ============================================================
# 25. CREATE TRANS_VALUE FRAUD-RATE PLOT
# ============================================================

value_quantile_table = (
    quantile_feature_tables[
        VALUE_FEATURE
    ]
)


create_fraud_rate_plot(

    x_values=value_quantile_table[
        "GROUP"
    ],

    fraud_percentages=value_quantile_table[
        "FRAUD_PERCENTAGE"
    ],

    title=(
        "Fraud rate across TRANS_VALUE quantile groups"
    ),

    x_label="TRANS_VALUE quantile group",

    output_path=VALUE_FRAUD_RATE_PATH
)


# ============================================================
# 26. CREATE SEND_POP_REGISTER FRAUD-RATE PLOT
# ============================================================

population_quantile_table = (
    quantile_feature_tables[
        POPULATION_FEATURE
    ]
)


create_fraud_rate_plot(

    x_values=population_quantile_table[
        "GROUP"
    ],

    fraud_percentages=population_quantile_table[
        "FRAUD_PERCENTAGE"
    ],

    title=(
        "Fraud rate across SEND_POP_REGISTER quantile groups"
    ),

    x_label="SEND_POP_REGISTER quantile group",

    output_path=POPULATION_FRAUD_RATE_PATH
)


# ============================================================
# 27. CREATE TRANS_DAY FRAUD-RATE PLOT
# ============================================================

create_fraud_rate_plot(

    x_values=day_fraud_rate_table[
        DAY_FEATURE
    ],

    fraud_percentages=day_fraud_rate_table[
        "FRAUD_PERCENTAGE"
    ],

    title=(
        "Fraud rate by transaction day of month"
    ),

    x_label=DAY_FEATURE,

    output_path=DAY_FRAUD_RATE_PATH
)


# ============================================================
# 28. RANK-BISERIAL EFFECT PLOT
# ============================================================

ranking_plot_table = (
    association_ranking_table
    .sort_values(
        by="RANK_BISERIAL",
        ascending=True
    )
)


fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)


ax.barh(

    ranking_plot_table[
        "FEATURE"
    ],

    ranking_plot_table[
        "RANK_BISERIAL"
    ]
)


ax.axvline(
    0,
    linewidth=1
)


ax.set_xlabel(
    "Rank-biserial correlation"
)


ax.set_ylabel(
    "Feature"
)


ax.set_title(
    "Numerical association with fraud - rank-biserial effect"
)


ax.grid(
    axis="x",
    alpha=0.20
)


fig.tight_layout()


fig.savefig(
    RANK_BISERIAL_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 29. IDENTIFY STRONGEST ROBUST ASSOCIATION
# ============================================================

strongest_association_row = (
    association_ranking_table
    .iloc[
        0
    ]
)


strongest_feature = (
    strongest_association_row[
        "FEATURE"
    ]
)


strongest_rank_biserial = float(
    strongest_association_row[
        "RANK_BISERIAL"
    ]
)


strongest_rank_biserial_strength = (
    strongest_association_row[
        "RANK_BISERIAL_STRENGTH"
    ]
)


# ============================================================
# 30. IMAGE TO BASE64 FUNCTION
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 31. CONVERT IMAGES TO BASE64
# ============================================================

age_fraud_rate_base64 = (
    image_to_base64(
        AGE_FRAUD_RATE_PATH
    )
)


value_fraud_rate_base64 = (
    image_to_base64(
        VALUE_FRAUD_RATE_PATH
    )
)


day_fraud_rate_base64 = (
    image_to_base64(
        DAY_FRAUD_RATE_PATH
    )
)


population_fraud_rate_base64 = (
    image_to_base64(
        POPULATION_FRAUD_RATE_PATH
    )
)


rank_biserial_base64 = (
    image_to_base64(
        RANK_BISERIAL_PATH
    )
)


# ============================================================
# 32. PREPARE HTML TABLES
# ============================================================

target_overview_html = (
    target_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PERCENTAGE":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


validation_html = (
    validation_table
    .to_html(
        index=False,
        border=0
    )
)


range_validation_html = (
    range_validation_table
    .to_html(
        index=False,
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


specific_validation_html = (
    specific_validation_table
    .to_html(
        index=False,
        border=0
    )
)


descriptive_statistics_html = (
    descriptive_statistics_table
    .to_html(
        index=False,
        border=0,
        float_format=lambda value:
            f"{value:.6f}"
    )
)


association_html = (
    association_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MEAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
                lambda value:
                    f"{value:.6f}",

            "MEDIAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
                lambda value:
                    f"{value:.6f}",

            "WELCH_T":
                lambda value:
                    f"{value:.6f}",

            "WELCH_P_VALUE":
                lambda value:
                    f"{value:.12g}",

            "MANN_WHITNEY_U":
                lambda value:
                    f"{value:.6f}",

            "MANN_WHITNEY_P_VALUE":
                lambda value:
                    f"{value:.12g}",

            "POINT_BISERIAL":
                lambda value:
                    f"{value:.6f}",

            "POINT_BISERIAL_P_VALUE":
                lambda value:
                    f"{value:.12g}",

            "RANK_BISERIAL":
                lambda value:
                    f"{value:.6f}",

            "COMMON_LANGUAGE_PROBABILITY":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


association_ranking_html = (
    association_ranking_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "POINT_BISERIAL":
                lambda value:
                    f"{value:.6f}",

            "RANK_BISERIAL":
                lambda value:
                    f"{value:.6f}",

            "COMMON_LANGUAGE_PROBABILITY":
                lambda value:
                    f"{value:.6f}",

            "ABS_RANK_BISERIAL":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


age_quantile_html = (
    age_quantile_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "VALUE_MIN":
                lambda value:
                    f"{value:.6f}",

            "VALUE_MEDIAN":
                lambda value:
                    f"{value:.6f}",

            "VALUE_MAX":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE":
                lambda value:
                    f"{value:.8f}",

            "FRAUD_PERCENTAGE":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE_LIFT":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


value_quantile_html = (
    value_quantile_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "VALUE_MIN":
                lambda value:
                    f"{value:.6f}",

            "VALUE_MEDIAN":
                lambda value:
                    f"{value:.6f}",

            "VALUE_MAX":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE":
                lambda value:
                    f"{value:.8f}",

            "FRAUD_PERCENTAGE":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE_LIFT":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


population_quantile_html = (
    population_quantile_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "VALUE_MIN":
                lambda value:
                    f"{value:.6f}",

            "VALUE_MEDIAN":
                lambda value:
                    f"{value:.6f}",

            "VALUE_MAX":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE":
                lambda value:
                    f"{value:.8f}",

            "FRAUD_PERCENTAGE":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE_LIFT":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


day_fraud_rate_html = (
    day_fraud_rate_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "FRAUD_RATE":
                lambda value:
                    f"{value:.8f}",

            "FRAUD_PERCENTAGE":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE_LIFT":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


# ============================================================
# 33. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Discrete and Continuous Numerical - Target Association
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1500px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 13px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 8px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Target Association —
Discrete and Continuous Numerical
</h1>


<p>

Numerical features analyzed:

</p>


<ul>

<li>{AGE_FEATURE} — continuous numeral</li>
<li>{VALUE_FEATURE} — continuous numeral</li>
<li>{DAY_FEATURE} — discrete numeral</li>
<li>{POPULATION_FEATURE} — discrete numeral</li>

</ul>


<p>

Target:

<strong>
{TARGET_FEATURE}
</strong>

</p>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete finite observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. TARGET OVERVIEW
========================================================= -->


<h2>
1. Target overview
</h2>


<div class="table-container">

{target_overview_html}

</div>


<p class="result">

Overall fraud rate:
{fraud_percentage:.6f}%

</p>


<div class="note">

TARGET_OMEGA is used only as an outcome variable for
exploratory association analysis.

It must not be included in the explanatory matrix used to
construct PCA, t-SNE or GMM.

</div>


<!-- ========================================================
     2. NUMERICAL VALIDATION
========================================================= -->


<h2>
2. Numerical feature validation
</h2>


<div class="table-container">

{validation_html}

</div>


<div class="table-container">

{range_validation_html}

</div>


<div class="table-container">

{specific_validation_html}

</div>


<div class="note">

The analysis validates numerical conversion, finite values
and basic domain constraints.

No invalid value is silently corrected.

The original parquet dataset is never modified.

</div>


<!-- ========================================================
     3. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
3. Descriptive statistics by target
</h2>


<div class="table-container">

{descriptive_statistics_html}

</div>


<div class="note">

Fraud and non-fraud observations are summarized separately.

Mean, median, dispersion, quantiles, skewness and excess
kurtosis are reported because the numerical distributions
may be asymmetric or heavy-tailed.

</div>


<!-- ========================================================
     4. STATISTICAL ASSOCIATION
========================================================= -->


<h2>
4. Statistical association with target
</h2>


<div class="table-container">

{association_html}

</div>


<div class="note">

<strong>Welch t-test</strong> compares class means without
assuming equal variances.

<br><br>

<strong>Mann-Whitney U</strong> evaluates rank differences
between fraud and non-fraud observations without requiring
normality.

<br><br>

<strong>Point-biserial correlation</strong> measures linear
association between the binary target and each numerical
feature.

<br><br>

<strong>Rank-biserial correlation</strong> provides a
rank-based effect size derived from Mann-Whitney U.

Positive values indicate that fraud observations tend to
have larger values.

Negative values indicate that fraud observations tend to
have smaller values.

</div>


<div class="note">

Because the dataset is very large, small differences can
produce extremely small p-values.

Effect sizes and fraud-rate patterns should therefore
receive more interpretive emphasis than statistical
significance alone.

</div>


<!-- ========================================================
     5. SEND_AGE
========================================================= -->


<h2>
5. SEND_AGE target profile
</h2>


<div class="table-container">

{age_quantile_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{age_fraud_rate_base64}"
    alt="SEND_AGE fraud rate"
>

</div>


<div class="note">

SEND_AGE is divided into quantile groups only for
descriptive visualization.

The continuous source feature is not discretized in the
dataset.

Fraud-rate lift values above 1 indicate quantile groups
with fraud prevalence above the global fraud rate.

</div>


<!-- ========================================================
     6. TRANS_VALUE
========================================================= -->


<h2>
6. TRANS_VALUE target profile
</h2>


<div class="table-container">

{value_quantile_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{value_fraud_rate_base64}"
    alt="TRANS_VALUE fraud rate"
>

</div>


<div class="note">

TRANS_VALUE is expected to potentially exhibit strong
asymmetry and extreme values.

Quantile-based fraud-rate analysis avoids allowing a small
number of very large values to define the descriptive
group boundaries.

No log transformation or clipping is applied during this
EDA.

</div>


<!-- ========================================================
     7. TRANS_DAY
========================================================= -->


<h2>
7. TRANS_DAY target profile
</h2>


<div class="table-container">

{day_fraud_rate_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{day_fraud_rate_base64}"
    alt="TRANS_DAY fraud rate"
>

</div>


<div class="note">

TRANS_DAY is retained in its original discrete support
from day 1 to day 31.

Unlike the other numerical features, quantile binning is
not required because the original values are directly
interpretable.

TRANS_DAY should not automatically be treated as a fixed
cyclical variable because month lengths differ.

</div>


<!-- ========================================================
     8. SEND_POP_REGISTER
========================================================= -->


<h2>
8. SEND_POP_REGISTER target profile
</h2>


<div class="table-container">

{population_quantile_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{population_fraud_rate_base64}"
    alt="SEND_POP_REGISTER fraud rate"
>

</div>


<div class="note">

SEND_POP_REGISTER is formally a discrete count variable but
may contain enough distinct values to behave approximately
like a continuous numerical feature in several statistical
procedures.

Quantile grouping is used only for descriptive target
profiling.

</div>


<!-- ========================================================
     9. EFFECT RANKING
========================================================= -->


<h2>
9. Numerical target-association ranking
</h2>


<div class="table-container">

{association_ranking_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{rank_biserial_base64}"
    alt="Numerical rank-biserial effect ranking"
>

</div>


<p class="result">

Strongest rank-based numerical association:

<br>

{strongest_feature}

<br>

Rank-biserial:
{strongest_rank_biserial:.6f}

<br>

Descriptive strength:
{strongest_rank_biserial_strength}

</p>


<!-- ========================================================
     10. MODELING IMPLICATIONS
========================================================= -->


<h2>
10. Potential modeling implications
</h2>


<div class="note">

<strong>Scaling:</strong>

<br><br>

SEND_AGE, TRANS_VALUE, TRANS_DAY and SEND_POP_REGISTER have
different units and numerical scales.

PCA, t-SNE and GMM are all sensitive to the geometry of the
feature space.

A scaling strategy must therefore be reviewed before
modeling.

</div>


<div class="note">

<strong>Skewness and heavy tails:</strong>

<br><br>

Strongly asymmetric variables, particularly transaction
value or population, may influence covariance, principal
components, pairwise distances and Gaussian component
estimation.

Any transformation decision should be made only after the
full EDA review.

</div>


<div class="note">

<strong>Outliers and extreme values:</strong>

<br><br>

Extreme numerical observations can strongly affect PCA and
GMM and may influence t-SNE neighborhood structure.

The presence of extremes does not automatically justify
their removal.

Their validity and modeling impact must be evaluated first.

</div>


<div class="note">

<strong>Discrete versus continuous structure:</strong>

<br><br>

TRANS_DAY and SEND_POP_REGISTER are formally discrete.

SEND_AGE and TRANS_VALUE are treated as continuous
numerical variables.

All four are quantitative, but their support and
interpretation remain different.

This distinction should be preserved during the final
pre-modeling audit.

</div>


<div class="note">

<strong>Non-linear target patterns:</strong>

<br><br>

A weak point-biserial or rank-biserial effect does not
guarantee the absence of target structure.

Fraud-rate profiles across days or quantile groups can
reveal non-monotonic relationships that a single
correlation coefficient cannot capture.

</div>


<div class="note">

<strong>Target exclusion:</strong>

<br><br>

TARGET_OMEGA must remain outside the matrix used to fit PCA,
t-SNE and GMM.

The target-association results are used for interpretation
and diagnostic review rather than supervised construction
of the unsupervised representation.

</div>


<!-- ========================================================
     11. SUMMARY
========================================================= -->


<h2>
11. Summary
</h2>


<p class="result">

Numerical features analyzed:
{len(NUMERICAL_FEATURES)}

</p>


<p class="result">

Continuous numerical features:
{len(CONTINUOUS_NUMERAL_FEATURES)}

</p>


<p class="result">

Discrete numerical features:
{len(DISCRETE_NUMERAL_FEATURES)}

</p>


<p class="result">

Overall fraud rate:
{fraud_percentage:.6f}%

</p>


<p class="result">

Strongest rank-based association:

<br>

{strongest_feature}

<br>

Rank-biserial:
{strongest_rank_biserial:.6f}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The four quantitative variables were compared between fraud
and non-fraud observations using descriptive statistics,
Welch tests, Mann-Whitney tests, point-biserial correlation
and rank-biserial effect sizes.

<br><br>

Quantile-based fraud-rate profiles were additionally used
for SEND_AGE, TRANS_VALUE and SEND_POP_REGISTER to reveal
possible non-linear patterns.

TRANS_DAY was analyzed directly across its original day
values.

<br><br>

The use of both continuous and discrete numerical features
in the same analysis is intentional because all four
features carry quantitative information and support many
of the same statistical procedures.

Their formal measurement characteristics remain documented
for later interpretation.

<br><br>

No feature is removed, clipped, transformed or permanently
discretized during this exploratory stage.

Scaling, skewness, extreme values, support structure and
possible non-linear behavior should all be reconsidered
during the final pre-modeling audit before PCA, t-SNE and
GMM.

</div>


</body>

</html>
"""


# ============================================================
# 34. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 35. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "DISCRETE AND CONTINUOUS NUMERICAL - TARGET ASSOCIATION"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete finite observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 36. DISPLAY TARGET OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TARGET OVERVIEW"
)


print(
    "=" * 100
)


display(
    target_overview_table
)


# ============================================================
# 37. DISPLAY VALIDATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "NUMERICAL VALIDATION"
)


print(
    "=" * 100
)


display(
    validation_table
)


display(
    range_validation_table
)


display(
    specific_validation_table
)


# ============================================================
# 38. DISPLAY DESCRIPTIVE STATISTICS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "DESCRIPTIVE STATISTICS BY TARGET"
)


print(
    "=" * 100
)


display(
    descriptive_statistics_table
)


# ============================================================
# 39. DISPLAY ASSOCIATION TESTS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "NUMERICAL TARGET ASSOCIATION TESTS"
)


print(
    "=" * 100
)


display(
    association_table
)


# ============================================================
# 40. DISPLAY SEND_AGE PROFILE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SEND_AGE FRAUD-RATE PROFILE"
)


print(
    "=" * 100
)


display(
    age_quantile_table
)


# ============================================================
# 41. DISPLAY TRANS_VALUE PROFILE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TRANS_VALUE FRAUD-RATE PROFILE"
)


print(
    "=" * 100
)


display(
    value_quantile_table
)


# ============================================================
# 42. DISPLAY TRANS_DAY PROFILE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TRANS_DAY FRAUD-RATE PROFILE"
)


print(
    "=" * 100
)


display(
    day_fraud_rate_table
)


# ============================================================
# 43. DISPLAY SEND_POP_REGISTER PROFILE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SEND_POP_REGISTER FRAUD-RATE PROFILE"
)


print(
    "=" * 100
)


display(
    population_quantile_table
)


# ============================================================
# 44. DISPLAY ASSOCIATION RANKING
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "NUMERICAL TARGET-ASSOCIATION RANKING"
)


print(
    "=" * 100
)


display(
    association_ranking_table
)


print(
    "\nStrongest rank-based numerical association:"
)


print(
    strongest_feature
)


print(
    "Rank-biserial:",
    f"{strongest_rank_biserial:.6f}"
)


print(
    "Strength:",
    strongest_rank_biserial_strength
)


# ============================================================
# 45. RELEASE MEMORY
# ============================================================

del dataset_numerical_target
del working_data
del complete_data

gc.collect()


# ============================================================
# 46. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    AGE_FRAUD_RATE_PATH
)


print(
    VALUE_FRAUD_RATE_PATH
)


print(
    DAY_FRAUD_RATE_PATH
)


print(
    POPULATION_FRAUD_RATE_PATH
)


print(
    RANK_BISERIAL_PATH
)


DISCRETE AND CONTINUOUS NUMERICAL - TARGET ASSOCIATION

Total dataset observations: 1852394
Complete finite observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

TARGET OVERVIEW


,TARGET_CLASS,TARGET_VALUE,COUNT,PERCENTAGE
0,Non-fraud,0,1842743,99.478999
1,Fraud,1,9651,0.521001



NUMERICAL VALIDATION


,FEATURE,FEATURE_TYPE,SOURCE_DATA_TYPE,MISSING_VALUES,NON_NUMERIC_VALUES,NON_FINITE_VALUES,UNIQUE_VALUES
0,SEND_AGE,Continuous numeral,float32,0,0,0,927
1,TRANS_VALUE,Continuous numeral,float64,0,0,0,60616
2,TRANS_DAY,Discrete numeral,int8,0,0,0,31
3,SEND_POP_REGISTER,Discrete numeral,int32,0,0,0,891


,FEATURE,MIN,MAX,NEGATIVE_VALUES
0,SEND_AGE,21.59,1.018400e+02,0
1,TRANS_VALUE,1.00,2.894890e+04,0
2,TRANS_DAY,1.00,3.100000e+01,0
3,SEND_POP_REGISTER,23.00,2.906700e+06,0


...


SEND_POP_REGISTER FRAUD-RATE PROFILE


,BIN,COUNT,FRAUD_COUNT,VALUE_MIN,VALUE_MEDIAN,VALUE_MAX,FRAUD_RATE,FRAUD_PERCENTAGE,FRAUD_RATE_LIFT,GROUP
0,"(22.999, 260.0]",186576,948,23.0,139.0,260.0,0.005081,0.508104,0.975245,1
1,"(260.0, 566.0]",184388,900,263.0,372.0,566.0,0.004881,0.488101,0.936852,2
2,"(566.0, 937.0]",185116,904,568.0,743.0,937.0,0.004883,0.488342,0.937315,3
3,"(937.0, 1628.0]",185120,943,964.0,1274.0,1628.0,0.005094,0.509399,0.977731,4
4,"(1628.0, 2443.0]",185868,956,1631.0,1923.0,2443.0,0.005143,0.514344,0.987221,5
5,"(2443.0, 4677.0]",185287,1057,2456.0,3487.0,4677.0,0.005705,0.570466,1.094942,6
6,"(4677.0, 9993.0]",185215,1009,4680.0,6006.0,9993.0,0.005448,0.544772,1.045625,7
7,"(9993.0, 42384.0]",185223,1051,10076.0,20328.0,42384.0,0.005674,0.567424,1.089103,8
8,"(42384.0, 186140.0]",186556,884,42619.0,88735.0,186140.0,0.004739,0.473852,0.909503,9
9,"(186140.0, 2906700.0]",183045,999,190178.0,525713.0,2906700.0,0.005458,0.545767,1.047535,10



NUMERICAL TARGET-ASSOCIATION RANKING


,FEATURE,FEATURE_TYPE,POINT_BISERIAL,POINT_BISERIAL_STRENGTH,RANK_BISERIAL,RANK_BISERIAL_STRENGTH,COMMON_LANGUAGE_PROBABILITY,ABS_RANK_BISERIAL
0,TRANS_VALUE,Continuous numeral,0.209308,Weak,0.668566,Strong,0.834283,0.668566
1,SEND_AGE,Continuous numeral,0.011130,Very weak or negligible,0.083888,Very weak or negligible,0.541944,0.083888
2,SEND_POP_REGISTER,Discrete numeral,0.000325,Very weak or negligible,0.015790,Very weak or negligible,0.507895,0.015790
3,TRANS_DAY,Discrete numeral,-0.000131,Very weak or negligible,-0.001194,Very weak or negligible,0.499403,0.001194



Strongest rank-based numerical association:
TRANS_VALUE
Rank-biserial: 0.668566
Strength: Strong

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/discrete_numera_and_continuos_numeral

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/discrete_numera_and_continuos_numeral/analysis_discrete_and_continuous_numerical_target_association.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/discrete_numera_and_continuos_numeral/numerical_target_send_age_fraud_rate.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/discrete_numera_and_continuos_numeral/numerical_target_trans_value_fraud_rate.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_va

## <span style="color:PINK"> FREQUENCY ENCODING WITH FALLBACK </span> ##

In [11]:

# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "02_relationships_with_target"
)

ASSOCIATION_GROUP = (
    "target_association"
)

FEATURE_GROUP = (
    "frequency_encoding_with_fallback"
)


CARD_FEATURE = (
    "TRANS_NUM_CARD_FEWF"
)

NAME_FEATURE = (
    "SEND_NAME_FEWF"
)

JOB_FEATURE = (
    "SEND_JOB_FEWF"
)

RECEIVER_LOCATION_FEATURE = (
    "RECEIVE_LOC_FEWF"
)

TARGET_FEATURE = (
    "TARGET_OMEGA"
)


FEWF_FEATURES = [
    CARD_FEATURE,
    NAME_FEATURE,
    JOB_FEATURE,
    RECEIVER_LOCATION_FEATURE
]


REQUIRED_FEATURES = (
    FEWF_FEATURES
    +
    [
        TARGET_FEATURE
    ]
)


TARGET_POSITIVE_VALUE = 1

ALPHA = 0.05

QUANTILE_GROUPS = 10

PNG_DPI = 300


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / ASSOCIATION_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_frequency_encoding_with_fallback_target_association.html"
)


CARD_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "fewf_target_card_frequency_fraud_rate.png"
)


NAME_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "fewf_target_name_frequency_fraud_rate.png"
)


JOB_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "fewf_target_job_frequency_fraud_rate.png"
)


RECEIVER_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "fewf_target_receiver_frequency_fraud_rate.png"
)


RANK_BISERIAL_PATH = (
    RESULTS_DIRECTORY
    / "fewf_target_rank_biserial_effect.png"
)


FEATURE_PLOT_PATHS = {

    CARD_FEATURE:
        CARD_FRAUD_RATE_PATH,

    NAME_FEATURE:
        NAME_FRAUD_RATE_PATH,

    JOB_FEATURE:
        JOB_FRAUD_RATE_PATH,

    RECEIVER_LOCATION_FEATURE:
        RECEIVER_FRAUD_RATE_PATH
}


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD ORIGINAL CATEGORICAL FEATURES AND TARGET
#
# FEWF source columns still contain the original
# categorical values.
#
# Frequency Encoding With Fallback is generated
# temporarily in memory.
#
# The parquet dataset is never modified.
# ============================================================

dataset_fewf_target = pd.read_parquet(
    DATASET_PATH,
    columns=REQUIRED_FEATURES
)


total_observations = int(
    len(
        dataset_fewf_target
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. VALIDATE REQUIRED FEATURES
# ============================================================

missing_features = [
    feature
    for feature in REQUIRED_FEATURES
    if feature not in dataset_fewf_target.columns
]


if missing_features:

    raise KeyError(
        "Missing required features: "
        + ", ".join(
            missing_features
        )
    )


# ============================================================
# 09. CREATE TEMPORARY FEWF REPRESENTATIONS
#
# FEWF(category) =
#
# category count / total dataset observations
#
# Fallback =
#
# 1 / total dataset observations
#
# Because the mapping is fitted and applied to the
# same exploratory dataset, fallback usage should
# normally be zero.
# ============================================================

temporary_fewf = pd.DataFrame(
    index=dataset_fewf_target.index
)


fewf_mappings = {}

fewf_fallback_values = {}

encoding_overview_records = []


for feature in FEWF_FEATURES:

    source_series = (
        dataset_fewf_target[
            feature
        ]
    )


    category_counts = (
        source_series
        .value_counts(
            dropna=True
        )
    )


    frequency_map = (
        category_counts
        /
        total_observations
    )


    fallback_value = (
        1.0
        /
        total_observations
    )


    fewf_mappings[
        feature
    ] = (
        frequency_map
    )


    fewf_fallback_values[
        feature
    ] = (
        fallback_value
    )


    encoded_series = (
        source_series
        .map(
            frequency_map
        )
        .astype(
            "float64"
        )
    )


    unseen_non_missing_mask = (
        source_series.notna()
        &
        encoded_series.isna()
    )


    fallback_uses = int(
        unseen_non_missing_mask.sum()
    )


    if fallback_uses > 0:

        encoded_series.loc[
            unseen_non_missing_mask
        ] = (
            fallback_value
        )


    temporary_fewf[
        feature
    ] = (
        encoded_series
    )


    observed_encoded_values = (
        encoded_series
        .dropna()
    )


    encoding_overview_records.append({

        "FEATURE":
            feature,

        "SOURCE_DATA_TYPE":
            str(
                source_series.dtype
            ),

        "ORIGINAL_UNIQUE_CATEGORIES":
            int(
                source_series.nunique(
                    dropna=True
                )
            ),

        "ENCODED_UNIQUE_FREQUENCIES":
            int(
                observed_encoded_values.nunique()
            ),

        "MISSING_VALUES":
            int(
                source_series.isna().sum()
            ),

        "MIN_OBSERVED_FREQUENCY":
            float(
                observed_encoded_values.min()
            ),

        "MEDIAN_OBSERVED_FREQUENCY":
            float(
                observed_encoded_values.median()
            ),

        "MAX_OBSERVED_FREQUENCY":
            float(
                observed_encoded_values.max()
            ),

        "FALLBACK_VALUE":
            float(
                fallback_value
            ),

        "FALLBACK_USES_IN_EDA":
            fallback_uses
    })


encoding_overview_table = pd.DataFrame(
    encoding_overview_records
)


# ============================================================
# 10. PREPARE TEMPORARY ANALYSIS DATA
# ============================================================

working_data = (
    temporary_fewf
    .copy()
)


working_data[
    TARGET_FEATURE
] = (
    dataset_fewf_target[
        TARGET_FEATURE
    ]
)


complete_data = (
    working_data[
        REQUIRED_FEATURES
    ]
    .dropna()
    .copy()
)


finite_mask = np.isfinite(
    complete_data[
        FEWF_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


analysis_data = (
    complete_data.loc[
        finite_mask,
        REQUIRED_FEATURES
    ]
    .copy()
)


analysis_observations = int(
    len(
        analysis_data
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite FEWF observations are available."
    )


excluded_observations = (
    total_observations
    -
    analysis_observations
)


excluded_percentage = (
    excluded_observations
    /
    total_observations
    *
    100
)


# ============================================================
# 11. VALIDATE TEMPORARY FEWF FEATURES
# ============================================================

for feature in FEWF_FEATURES:

    if not pd.api.types.is_numeric_dtype(
        analysis_data[
            feature
        ]
    ):

        raise TypeError(
            f"Temporary FEWF feature is not numerical: {feature}"
        )


# ============================================================
# 12. VALIDATE TARGET
# ============================================================

target_values = (
    analysis_data[
        TARGET_FEATURE
    ]
    .drop_duplicates()
    .tolist()
)


if len(
    target_values
) != 2:

    raise ValueError(
        f"{TARGET_FEATURE} must contain exactly two classes. "
        f"Observed values: {target_values}"
    )


target_positive_value = None


for value in target_values:

    if (
        value == TARGET_POSITIVE_VALUE
        or
        str(
            value
        )
        == str(
            TARGET_POSITIVE_VALUE
        )
    ):

        target_positive_value = (
            value
        )

        break


if target_positive_value is None:

    raise ValueError(
        f"Fraud target value {TARGET_POSITIVE_VALUE} was not found. "
        f"Observed values: {target_values}"
    )


target_negative_values = [
    value
    for value in target_values
    if value != target_positive_value
]


if len(
    target_negative_values
) != 1:

    raise ValueError(
        "Unable to determine the non-fraud target class."
    )


target_negative_value = (
    target_negative_values[
        0
    ]
)


# ============================================================
# 13. CREATE TEMPORARY BINARY TARGET
#
# Non-fraud = 0
# Fraud     = 1
# ============================================================

analysis_data[
    "_TARGET_BINARY"
] = (
    analysis_data[
        TARGET_FEATURE
    ]
    .map({

        target_negative_value:
            0,

        target_positive_value:
            1
    })
    .astype(
        "int8"
    )
)


# ============================================================
# 14. TARGET OVERVIEW
# ============================================================

target_counts = (
    analysis_data[
        "_TARGET_BINARY"
    ]
    .value_counts()
    .reindex(
        [
            0,
            1
        ],
        fill_value=0
    )
)


non_fraud_count = int(
    target_counts.loc[
        0
    ]
)


fraud_count = int(
    target_counts.loc[
        1
    ]
)


non_fraud_percentage = (
    non_fraud_count
    /
    analysis_observations
    *
    100
)


fraud_percentage = (
    fraud_count
    /
    analysis_observations
    *
    100
)


overall_fraud_rate = (
    fraud_count
    /
    analysis_observations
)


target_overview_table = pd.DataFrame({

    "TARGET_CLASS": [
        "Non-fraud",
        "Fraud"
    ],

    "TARGET_VALUE": [
        target_negative_value,
        target_positive_value
    ],

    "COUNT": [
        non_fraud_count,
        fraud_count
    ],

    "PERCENTAGE": [
        non_fraud_percentage,
        fraud_percentage
    ]
})


# ============================================================
# 15. DESCRIPTIVE STATISTICS BY TARGET
# ============================================================

descriptive_records = []


for feature in FEWF_FEATURES:

    for target_code, target_label in [
        (
            0,
            "Non-fraud"
        ),
        (
            1,
            "Fraud"
        )
    ]:

        values = (
            analysis_data.loc[
                analysis_data[
                    "_TARGET_BINARY"
                ]
                == target_code,
                feature
            ]
            .to_numpy(
                dtype="float64"
            )
        )


        q01 = float(
            np.quantile(
                values,
                0.01
            )
        )


        q05 = float(
            np.quantile(
                values,
                0.05
            )
        )


        q25 = float(
            np.quantile(
                values,
                0.25
            )
        )


        q50 = float(
            np.quantile(
                values,
                0.50
            )
        )


        q75 = float(
            np.quantile(
                values,
                0.75
            )
        )


        q95 = float(
            np.quantile(
                values,
                0.95
            )
        )


        q99 = float(
            np.quantile(
                values,
                0.99
            )
        )


        descriptive_records.append({

            "FEATURE":
                feature,

            "TARGET_CLASS":
                target_label,

            "COUNT":
                int(
                    len(
                        values
                    )
                ),

            "MEAN":
                float(
                    np.mean(
                        values
                    )
                ),

            "STANDARD_DEVIATION":
                float(
                    np.std(
                        values,
                        ddof=1
                    )
                ),

            "VARIANCE":
                float(
                    np.var(
                        values,
                        ddof=1
                    )
                ),

            "MIN":
                float(
                    np.min(
                        values
                    )
                ),

            "P01":
                q01,

            "P05":
                q05,

            "P25":
                q25,

            "MEDIAN":
                q50,

            "P75":
                q75,

            "P95":
                q95,

            "P99":
                q99,

            "MAX":
                float(
                    np.max(
                        values
                    )
                ),

            "IQR":
                float(
                    q75
                    -
                    q25
                ),

            "SKEWNESS":
                float(
                    stats.skew(
                        values,
                        bias=False
                    )
                ),

            "EXCESS_KURTOSIS":
                float(
                    stats.kurtosis(
                        values,
                        fisher=True,
                        bias=False
                    )
                )
        })


descriptive_statistics_table = pd.DataFrame(
    descriptive_records
)


# ============================================================
# 16. EFFECT STRENGTH INTERPRETATION
# ============================================================

def interpret_effect_strength(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        return (
            "Weak"
        )


    elif absolute_value < 0.50:

        return (
            "Moderate"
        )


    elif absolute_value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


# ============================================================
# 17. ASSOCIATION TESTS
#
# Every temporary FEWF feature is evaluated using:
#
# - Mean difference
# - Median difference
# - Welch t-test
# - Mann-Whitney U
# - Point-biserial correlation
# - Rank-biserial correlation
# - Mutual Information
# - Normalized Mutual Information
# ============================================================

association_records = []


target_binary_array = (
    analysis_data[
        "_TARGET_BINARY"
    ]
    .to_numpy(
        dtype="int8"
    )
)


for feature in FEWF_FEATURES:

    non_fraud_values = (
        analysis_data.loc[
            analysis_data[
                "_TARGET_BINARY"
            ]
            == 0,
            feature
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    fraud_values = (
        analysis_data.loc[
            analysis_data[
                "_TARGET_BINARY"
            ]
            == 1,
            feature
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    # --------------------------------------------------------
    # LOCATION DIFFERENCES
    # --------------------------------------------------------

    mean_difference = float(
        np.mean(
            fraud_values
        )
        -
        np.mean(
            non_fraud_values
        )
    )


    median_difference = float(
        np.median(
            fraud_values
        )
        -
        np.median(
            non_fraud_values
        )
    )


    # --------------------------------------------------------
    # WELCH T-TEST
    # --------------------------------------------------------

    if (
        len(
            fraud_values
        )
        > 1
        and
        len(
            non_fraud_values
        )
        > 1
    ):

        welch_result = stats.ttest_ind(

            fraud_values,

            non_fraud_values,

            equal_var=False
        )


        welch_statistic = float(
            welch_result.statistic
        )


        welch_p_value = float(
            welch_result.pvalue
        )


    else:

        welch_statistic = np.nan

        welch_p_value = np.nan


    # --------------------------------------------------------
    # MANN-WHITNEY U
    # --------------------------------------------------------

    mann_whitney_result = stats.mannwhitneyu(

        fraud_values,

        non_fraud_values,

        alternative="two-sided",

        method="asymptotic"
    )


    mann_whitney_u = float(
        mann_whitney_result.statistic
    )


    mann_whitney_p_value = float(
        mann_whitney_result.pvalue
    )


    n_fraud = float(
        len(
            fraud_values
        )
    )


    n_non_fraud = float(
        len(
            non_fraud_values
        )
    )


    common_language_probability = float(
        mann_whitney_u
        /
        (
            n_fraud
            *
            n_non_fraud
        )
    )


    rank_biserial = float(
        2
        *
        common_language_probability
        -
        1
    )


    rank_biserial_strength = (
        interpret_effect_strength(
            rank_biserial
        )
    )


    # --------------------------------------------------------
    # POINT-BISERIAL CORRELATION
    # --------------------------------------------------------

    feature_values = (
        analysis_data[
            feature
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    if np.std(
        feature_values
    ) > 0:

        point_biserial_result = stats.pointbiserialr(

            target_binary_array,

            feature_values
        )


        point_biserial = float(
            point_biserial_result.statistic
        )


        point_biserial_p_value = float(
            point_biserial_result.pvalue
        )


    else:

        point_biserial = np.nan

        point_biserial_p_value = np.nan


    point_biserial_strength = (
        interpret_effect_strength(
            point_biserial
        )
    )


    # --------------------------------------------------------
    # MUTUAL INFORMATION BETWEEN FEWF LEVEL AND TARGET
    #
    # Different original categories with the same frequency
    # become the same FEWF level.
    # --------------------------------------------------------

    fewf_level_codes, _ = pd.factorize(

        analysis_data[
            feature
        ],

        sort=True
    )


    mutual_information = float(
        mutual_info_score(

            fewf_level_codes,

            target_binary_array
        )
    )


    normalized_mutual_information = float(
        normalized_mutual_info_score(

            fewf_level_codes,

            target_binary_array,

            average_method="arithmetic"
        )
    )


    association_records.append({

        "FEATURE":
            feature,

        "MEAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
            mean_difference,

        "MEDIAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
            median_difference,

        "WELCH_T":
            welch_statistic,

        "WELCH_P_VALUE":
            welch_p_value,

        "MANN_WHITNEY_U":
            mann_whitney_u,

        "MANN_WHITNEY_P_VALUE":
            mann_whitney_p_value,

        "POINT_BISERIAL":
            point_biserial,

        "POINT_BISERIAL_P_VALUE":
            point_biserial_p_value,

        "POINT_BISERIAL_STRENGTH":
            point_biserial_strength,

        "RANK_BISERIAL":
            rank_biserial,

        "RANK_BISERIAL_STRENGTH":
            rank_biserial_strength,

        "COMMON_LANGUAGE_PROBABILITY":
            common_language_probability,

        "MUTUAL_INFORMATION":
            mutual_information,

        "NORMALIZED_MUTUAL_INFORMATION":
            normalized_mutual_information
    })


association_table = pd.DataFrame(
    association_records
)


# ============================================================
# 18. ASSOCIATION RANKING
#
# Rank-biserial is used as the principal robust
# effect-size ranking.
# ============================================================

association_ranking_table = (
    association_table[
        [
            "FEATURE",
            "POINT_BISERIAL",
            "POINT_BISERIAL_STRENGTH",
            "RANK_BISERIAL",
            "RANK_BISERIAL_STRENGTH",
            "COMMON_LANGUAGE_PROBABILITY",
            "MUTUAL_INFORMATION",
            "NORMALIZED_MUTUAL_INFORMATION"
        ]
    ]
    .copy()
)


association_ranking_table[
    "ABS_RANK_BISERIAL"
] = (
    association_ranking_table[
        "RANK_BISERIAL"
    ]
    .abs()
)


association_ranking_table = (
    association_ranking_table
    .sort_values(
        by="ABS_RANK_BISERIAL",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 19. CREATE FEWF QUANTILE FRAUD-RATE FUNCTION
#
# The central exploratory question is:
#
# Do observations associated with rarer or more frequent
# categories have different fraud rates?
#
# Quantile groups are temporary and descriptive only.
# ============================================================

def create_fewf_quantile_table(
    dataframe,
    feature,
    number_quantiles=10
):

    feature_values = (
        dataframe[
            feature
        ]
    )


    unique_values = int(
        feature_values.nunique()
    )


    if unique_values <= 1:

        summary = pd.DataFrame({

            "GROUP": [
                1
            ],

            "COUNT": [
                int(
                    len(
                        dataframe
                    )
                )
            ],

            "FRAUD_COUNT": [
                int(
                    dataframe[
                        "_TARGET_BINARY"
                    ].sum()
                )
            ],

            "FEWF_MIN": [
                float(
                    feature_values.min()
                )
            ],

            "FEWF_MEDIAN": [
                float(
                    feature_values.median()
                )
            ],

            "FEWF_MAX": [
                float(
                    feature_values.max()
                )
            ]
        })


    else:

        effective_quantiles = min(
            number_quantiles,
            unique_values
        )


        temporary_bins = pd.qcut(

            feature_values,

            q=effective_quantiles,

            duplicates="drop"
        )


        temporary_table = pd.DataFrame({

            "BIN":
                temporary_bins,

            "FEWF_VALUE":
                feature_values,

            "TARGET":
                dataframe[
                    "_TARGET_BINARY"
                ]
        })


        summary = (
            temporary_table
            .groupby(
                "BIN",
                observed=True
            )
            .agg(

                COUNT=(
                    "TARGET",
                    "size"
                ),

                FRAUD_COUNT=(
                    "TARGET",
                    "sum"
                ),

                FEWF_MIN=(
                    "FEWF_VALUE",
                    "min"
                ),

                FEWF_MEDIAN=(
                    "FEWF_VALUE",
                    "median"
                ),

                FEWF_MAX=(
                    "FEWF_VALUE",
                    "max"
                )
            )
            .reset_index(
                drop=True
            )
        )


        summary[
            "GROUP"
        ] = np.arange(
            1,
            len(
                summary
            )
            + 1
        )


    summary[
        "FRAUD_RATE"
    ] = (
        summary[
            "FRAUD_COUNT"
        ]
        /
        summary[
            "COUNT"
        ]
    )


    summary[
        "FRAUD_PERCENTAGE"
    ] = (
        summary[
            "FRAUD_RATE"
        ]
        *
        100
    )


    summary[
        "FRAUD_RATE_LIFT"
    ] = (
        summary[
            "FRAUD_RATE"
        ]
        /
        overall_fraud_rate
    )


    summary = (
        summary[
            [
                "GROUP",
                "COUNT",
                "FRAUD_COUNT",
                "FEWF_MIN",
                "FEWF_MEDIAN",
                "FEWF_MAX",
                "FRAUD_RATE",
                "FRAUD_PERCENTAGE",
                "FRAUD_RATE_LIFT"
            ]
        ]
        .copy()
    )


    return (
        summary
    )


# ============================================================
# 20. CREATE FEWF FRAUD-RATE PROFILES
# ============================================================

fewf_profile_tables = {}


for feature in FEWF_FEATURES:

    fewf_profile_tables[
        feature
    ] = create_fewf_quantile_table(

        dataframe=analysis_data,

        feature=feature,

        number_quantiles=QUANTILE_GROUPS
    )


# ============================================================
# 21. RARE VS FREQUENT PROFILE SUMMARY
#
# First quantile group:
# rarer categories
#
# Last quantile group:
# more frequent categories
# ============================================================

frequency_profile_summary_records = []


for feature in FEWF_FEATURES:

    profile_table = (
        fewf_profile_tables[
            feature
        ]
    )


    rarest_group = (
        profile_table
        .iloc[
            0
        ]
    )


    most_frequent_group = (
        profile_table
        .iloc[
            -1
        ]
    )


    rare_fraud_rate = float(
        rarest_group[
            "FRAUD_RATE"
        ]
    )


    frequent_fraud_rate = float(
        most_frequent_group[
            "FRAUD_RATE"
        ]
    )


    fraud_rate_difference = (
        rare_fraud_rate
        -
        frequent_fraud_rate
    )


    if frequent_fraud_rate > 0:

        rare_to_frequent_risk_ratio = (
            rare_fraud_rate
            /
            frequent_fraud_rate
        )


    else:

        rare_to_frequent_risk_ratio = (
            np.nan
        )


    frequency_profile_summary_records.append({

        "FEATURE":
            feature,

        "RAREST_GROUP_FRAUD_PERCENTAGE":
            rare_fraud_rate
            *
            100,

        "MOST_FREQUENT_GROUP_FRAUD_PERCENTAGE":
            frequent_fraud_rate
            *
            100,

        "RARE_MINUS_FREQUENT_PERCENTAGE_POINTS":
            fraud_rate_difference
            *
            100,

        "RARE_TO_FREQUENT_RISK_RATIO":
            rare_to_frequent_risk_ratio
    })


frequency_profile_summary_table = pd.DataFrame(
    frequency_profile_summary_records
)


# ============================================================
# 22. FRAUD-RATE PROFILE PLOT FUNCTION
# ============================================================

def create_fewf_fraud_rate_plot(
    profile_table,
    feature,
    output_path
):

    fig, ax = plt.subplots(
        figsize=(
            11,
            7
        )
    )


    ax.plot(

        profile_table[
            "GROUP"
        ],

        profile_table[
            "FRAUD_PERCENTAGE"
        ],

        marker="o"
    )


    ax.axhline(

        overall_fraud_rate
        *
        100,

        linestyle="--",

        linewidth=1.5,

        label="Overall fraud rate"
    )


    ax.set_xlabel(
        f"{feature} frequency quantile group"
    )


    ax.set_ylabel(
        "Fraud rate (%)"
    )


    ax.set_title(
        f"Fraud rate across temporary FEWF levels - {feature}"
    )


    ax.set_xticks(
        profile_table[
            "GROUP"
        ]
    )


    ax.legend()


    ax.grid(
        alpha=0.20
    )


    fig.tight_layout()


    fig.savefig(

        output_path,

        format="png",

        dpi=PNG_DPI,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 23. CREATE FEATURE PROFILE PLOTS
# ============================================================

for feature in FEWF_FEATURES:

    create_fewf_fraud_rate_plot(

        profile_table=fewf_profile_tables[
            feature
        ],

        feature=feature,

        output_path=FEATURE_PLOT_PATHS[
            feature
        ]
    )


# ============================================================
# 24. RANK-BISERIAL EFFECT PLOT
# ============================================================

ranking_plot_table = (
    association_ranking_table
    .sort_values(
        by="RANK_BISERIAL",
        ascending=True
    )
)


fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)


ax.barh(

    ranking_plot_table[
        "FEATURE"
    ],

    ranking_plot_table[
        "RANK_BISERIAL"
    ]
)


ax.axvline(
    0,
    linewidth=1
)


ax.set_xlabel(
    "Rank-biserial correlation"
)


ax.set_ylabel(
    "Feature"
)


ax.set_title(
    "Temporary FEWF association with fraud"
)


ax.grid(
    axis="x",
    alpha=0.20
)


fig.tight_layout()


fig.savefig(

    RANK_BISERIAL_PATH,

    format="png",

    dpi=PNG_DPI,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 25. IDENTIFY STRONGEST ASSOCIATION
# ============================================================

strongest_association_row = (
    association_ranking_table
    .iloc[
        0
    ]
)


strongest_feature = (
    strongest_association_row[
        "FEATURE"
    ]
)


strongest_rank_biserial = float(
    strongest_association_row[
        "RANK_BISERIAL"
    ]
)


strongest_rank_biserial_strength = (
    strongest_association_row[
        "RANK_BISERIAL_STRENGTH"
    ]
)


# ============================================================
# 26. IMAGE TO BASE64 FUNCTION
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 27. CONVERT IMAGES TO BASE64
# ============================================================

feature_plot_base64 = {}


for feature in FEWF_FEATURES:

    feature_plot_base64[
        feature
    ] = image_to_base64(

        FEATURE_PLOT_PATHS[
            feature
        ]
    )


rank_biserial_base64 = (
    image_to_base64(
        RANK_BISERIAL_PATH
    )
)


# ============================================================
# 28. PREPARE HTML TABLES
# ============================================================

target_overview_html = (
    target_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PERCENTAGE":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


encoding_overview_html = (
    encoding_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MIN_OBSERVED_FREQUENCY":
                lambda value:
                    f"{value:.12f}",

            "MEDIAN_OBSERVED_FREQUENCY":
                lambda value:
                    f"{value:.12f}",

            "MAX_OBSERVED_FREQUENCY":
                lambda value:
                    f"{value:.12f}",

            "FALLBACK_VALUE":
                lambda value:
                    f"{value:.12f}"
        }
    )
)


descriptive_statistics_html = (
    descriptive_statistics_table
    .to_html(
        index=False,
        border=0,
        float_format=lambda value:
            f"{value:.10f}"
    )
)


association_html = (
    association_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MEAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
                lambda value:
                    f"{value:.10f}",

            "MEDIAN_DIFFERENCE_FRAUD_MINUS_NON_FRAUD":
                lambda value:
                    f"{value:.10f}",

            "WELCH_T":
                lambda value:
                    (
                        "NaN"
                        if pd.isna(
                            value
                        )
                        else f"{value:.6f}"
                    ),

            "WELCH_P_VALUE":
                lambda value:
                    (
                        "NaN"
                        if pd.isna(
                            value
                        )
                        else f"{value:.12g}"
                    ),

            "MANN_WHITNEY_U":
                lambda value:
                    f"{value:.6f}",

            "MANN_WHITNEY_P_VALUE":
                lambda value:
                    f"{value:.12g}",

            "POINT_BISERIAL":
                lambda value:
                    (
                        "NaN"
                        if pd.isna(
                            value
                        )
                        else f"{value:.6f}"
                    ),

            "POINT_BISERIAL_P_VALUE":
                lambda value:
                    (
                        "NaN"
                        if pd.isna(
                            value
                        )
                        else f"{value:.12g}"
                    ),

            "RANK_BISERIAL":
                lambda value:
                    f"{value:.6f}",

            "COMMON_LANGUAGE_PROBABILITY":
                lambda value:
                    f"{value:.6f}",

            "MUTUAL_INFORMATION":
                lambda value:
                    f"{value:.8f}",

            "NORMALIZED_MUTUAL_INFORMATION":
                lambda value:
                    f"{value:.8f}"
        }
    )
)


association_ranking_html = (
    association_ranking_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "POINT_BISERIAL":
                lambda value:
                    (
                        "NaN"
                        if pd.isna(
                            value
                        )
                        else f"{value:.6f}"
                    ),

            "RANK_BISERIAL":
                lambda value:
                    f"{value:.6f}",

            "COMMON_LANGUAGE_PROBABILITY":
                lambda value:
                    f"{value:.6f}",

            "MUTUAL_INFORMATION":
                lambda value:
                    f"{value:.8f}",

            "NORMALIZED_MUTUAL_INFORMATION":
                lambda value:
                    f"{value:.8f}",

            "ABS_RANK_BISERIAL":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


frequency_profile_summary_html = (
    frequency_profile_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "RAREST_GROUP_FRAUD_PERCENTAGE":
                lambda value:
                    f"{value:.6f}",

            "MOST_FREQUENT_GROUP_FRAUD_PERCENTAGE":
                lambda value:
                    f"{value:.6f}",

            "RARE_MINUS_FREQUENT_PERCENTAGE_POINTS":
                lambda value:
                    f"{value:.6f}",

            "RARE_TO_FREQUENT_RISK_RATIO":
                lambda value:
                    (
                        "NaN"
                        if pd.isna(
                            value
                        )
                        else f"{value:.6f}"
                    )
        }
    )
)


# ============================================================
# 29. CREATE FEATURE-SPECIFIC HTML SECTIONS
# ============================================================

feature_html_sections = []


for section_number, feature in enumerate(
    FEWF_FEATURES,
    start=5
):

    profile_table = (
        fewf_profile_tables[
            feature
        ]
    )


    profile_html = (
        profile_table
        .to_html(
            index=False,
            border=0,
            formatters={

                "FEWF_MIN":
                    lambda value:
                        f"{value:.12f}",

                "FEWF_MEDIAN":
                    lambda value:
                        f"{value:.12f}",

                "FEWF_MAX":
                    lambda value:
                        f"{value:.12f}",

                "FRAUD_RATE":
                    lambda value:
                        f"{value:.8f}",

                "FRAUD_PERCENTAGE":
                    lambda value:
                        f"{value:.6f}",

                "FRAUD_RATE_LIFT":
                    lambda value:
                        f"{value:.6f}"
            }
        )
    )


    feature_image_base64 = (
        feature_plot_base64[
            feature
        ]
    )


    feature_html_sections.append(
        f"""

        <h2>
        {section_number}. {feature} frequency profile
        </h2>


        <div class="table-container">

        {profile_html}

        </div>


        <div class="chart">

        <img
            src="data:image/png;base64,{feature_image_base64}"
            alt="{feature} FEWF fraud-rate profile"
        >

        </div>


        <div class="note">

        Quantile group 1 contains observations associated
        with relatively rarer categories.

        Higher groups contain observations associated with
        progressively more frequent categories.

        <br><br>

        The grouping is created temporarily only for
        descriptive target profiling.

        Neither the original category nor the temporary
        continuous FEWF representation is permanently
        discretized.

        <br><br>

        Fraud-rate lift greater than 1 indicates a group
        with fraud prevalence above the overall fraud rate.

        </div>

        """
    )


feature_html_content = "\n".join(
    feature_html_sections
)


# ============================================================
# 30. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Frequency Encoding With Fallback - Target Association
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1500px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 13px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 8px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Target Association —
Frequency Encoding With Fallback
</h1>


<p>

Source categorical features:

</p>


<ul>

<li>{CARD_FEATURE}</li>
<li>{NAME_FEATURE}</li>
<li>{JOB_FEATURE}</li>
<li>{RECEIVER_LOCATION_FEATURE}</li>

</ul>


<p>

Target:

<strong>
{TARGET_FEATURE}
</strong>

</p>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete finite observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. TARGET OVERVIEW
========================================================= -->


<h2>
1. Target overview
</h2>


<div class="table-container">

{target_overview_html}

</div>


<p class="result">

Overall fraud rate:
{fraud_percentage:.6f}%

</p>


<div class="note">

TARGET_OMEGA is used only as an outcome variable for
exploratory association analysis.

It must remain outside the explanatory matrix used to fit
PCA, t-SNE and GMM.

</div>


<!-- ========================================================
     2. TEMPORARY FEWF
========================================================= -->


<h2>
2. Temporary Frequency Encoding With Fallback
</h2>


<div class="table-container">

{encoding_overview_html}

</div>


<div class="note">

The parquet dataset preserves the original categorical
values.

Frequency Encoding With Fallback is generated temporarily
in memory.

<br><br>

For an observed category:

<strong>
FEWF = category count / total dataset observations
</strong>

<br><br>

For a previously unseen category:

<strong>
Fallback = 1 / total dataset observations
</strong>

<br><br>

Because the mapping is fitted and applied to the same
exploratory dataset, fallback usage is expected to be zero.

</div>


<!-- ========================================================
     3. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
3. FEWF descriptive statistics by target
</h2>


<div class="table-container">

{descriptive_statistics_html}

</div>


<div class="note">

The values represent category prevalence rather than an
ordinary physical numerical measurement.

A larger FEWF value means that the original category occurs
more frequently in the dataset.

A smaller FEWF value corresponds to a rarer category.

</div>


<!-- ========================================================
     4. STATISTICAL ASSOCIATION
========================================================= -->


<h2>
4. Statistical association with target
</h2>


<div class="table-container">

{association_html}

</div>


<div class="note">

<strong>Welch t-test</strong> compares mean FEWF values
between fraud and non-fraud observations.

<br><br>

<strong>Mann-Whitney U</strong> evaluates whether the
frequency ranks differ between target classes.

<br><br>

<strong>Point-biserial correlation</strong> measures linear
association between FEWF and the binary target.

<br><br>

<strong>Rank-biserial correlation</strong> provides a
rank-based effect size.

Positive values indicate that fraud observations tend to
belong to more frequent categories.

Negative values indicate that fraud observations tend to
belong to rarer categories.

<br><br>

<strong>Mutual Information</strong> evaluates dependence
between the temporary FEWF levels and fraud without
requiring a linear or monotonic relationship.

</div>


<div class="note">

Because the dataset contains a very large number of
observations, very small effects can produce extremely
small p-values.

Effect sizes, Mutual Information and fraud-rate profiles
should therefore receive more interpretive emphasis than
p-values alone.

</div>


{feature_html_content}


<!-- ========================================================
     9. RARE VS FREQUENT
========================================================= -->


<h2>
9. Rare versus frequent category profile
</h2>


<div class="table-container">

{frequency_profile_summary_html}

</div>


<div class="note">

This comparison contrasts the fraud rate in the lowest FEWF
group with the fraud rate in the highest FEWF group.

<br><br>

A positive rare-minus-frequent difference indicates that
observations associated with rarer categories have a higher
fraud rate.

A negative difference indicates a higher fraud rate among
more frequent categories.

<br><br>

This is an exploratory descriptive comparison and should
not be interpreted as a causal effect.

</div>


<!-- ========================================================
     10. EFFECT RANKING
========================================================= -->


<h2>
10. FEWF target-association ranking
</h2>


<div class="table-container">

{association_ranking_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{rank_biserial_base64}"
    alt="FEWF rank-biserial effect ranking"
>

</div>


<p class="result">

Strongest rank-based FEWF association:

<br>

{strongest_feature}

<br>

Rank-biserial:
{strongest_rank_biserial:.6f}

<br>

Descriptive strength:
{strongest_rank_biserial_strength}

</p>


<!-- ========================================================
     11. MODELING IMPLICATIONS
========================================================= -->


<h2>
11. Potential modeling implications
</h2>


<div class="note">

<strong>Skewness and concentration near zero:</strong>

<br><br>

Frequency-encoded variables may be highly concentrated near
zero because many original categories are relatively rare.

This distributional structure should be considered before
scaling, PCA, t-SNE and GMM.

</div>


<div class="note">

<strong>Repeated numerical levels:</strong>

<br><br>

Different original categories can occur the same number of
times and therefore receive exactly the same FEWF value.

The encoded variable contains less categorical identity
information than the original feature.

This should be remembered when interpreting PCA loadings,
distances and Gaussian components.

</div>


<div class="note">

<strong>Potential redundancy:</strong>

<br><br>

Previous within-group analysis evaluates whether different
FEWF features produce very similar frequency patterns.

Any strong redundancy should be reviewed together with the
current target-association evidence before PCA and GMM.

</div>


<div class="note">

<strong>Scaling:</strong>

<br><br>

The four temporary FEWF variables may have different ranges
and dispersions.

Their scale relative to continuous, binary, one-hot,
geographic and cyclical features should be reviewed before
constructing the final feature matrix.

</div>


<div class="note">

<strong>Fallback behavior:</strong>

<br><br>

The fallback is not expected to be activated during this
EDA because the mapping is learned from the same dataset.

When a fitted FEWF mapping is applied to future or held-out
data, unseen categories receive the fallback value.

That value represents an extremely rare category and can
therefore affect its location in PCA, t-SNE and GMM feature
space.

</div>


<div class="note">

<strong>Information leakage:</strong>

<br><br>

The complete dataset is used here only to study the
exploratory FEWF structure.

For a final modeling pipeline involving held-out or future
data, FEWF mappings should be fitted on the appropriate
training data and then applied to other datasets.

</div>


<div class="note">

<strong>Target exclusion:</strong>

<br><br>

Association with fraud does not mean TARGET_OMEGA should be
used to construct the unsupervised representation.

TARGET_OMEGA remains excluded from PCA, t-SNE and GMM and
is used only for interpretation and external evaluation.

</div>


<!-- ========================================================
     12. SUMMARY
========================================================= -->


<h2>
12. Summary
</h2>


<p class="result">

FEWF source features analyzed:
{len(FEWF_FEATURES)}

</p>


<p class="result">

Overall fraud rate:
{fraud_percentage:.6f}%

</p>


<p class="result">

Strongest rank-based FEWF association:

<br>

{strongest_feature}

<br>

Rank-biserial:
{strongest_rank_biserial:.6f}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The original categorical values were preserved in the
parquet dataset.

Temporary Frequency Encoding With Fallback representations
were generated only in memory.

<br><br>

Fraud and non-fraud observations were compared using
descriptive statistics, Welch tests, Mann-Whitney tests,
point-biserial correlations, rank-biserial effects and
Mutual Information.

<br><br>

Quantile-based fraud-rate profiles were used to evaluate
whether fraud prevalence changes as original categories
move from relatively rare to relatively frequent.

<br><br>

No FEWF value is permanently written to the dataset and no
feature is removed during this exploratory stage.

The FEWF distribution, scaling, redundancy, fallback
behavior and relationship with fraud should all be reviewed
during the final pre-modeling audit before PCA, t-SNE and
GMM.

</div>


</body>

</html>
"""


# ============================================================
# 31. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 32. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "FREQUENCY ENCODING WITH FALLBACK - TARGET ASSOCIATION"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete finite observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 33. DISPLAY TARGET OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TARGET OVERVIEW"
)


print(
    "=" * 100
)


display(
    target_overview_table
)


# ============================================================
# 34. DISPLAY TEMPORARY FEWF OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TEMPORARY FEWF OVERVIEW"
)


print(
    "=" * 100
)


display(
    encoding_overview_table
)


# ============================================================
# 35. DISPLAY DESCRIPTIVE STATISTICS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "FEWF DESCRIPTIVE STATISTICS BY TARGET"
)


print(
    "=" * 100
)


display(
    descriptive_statistics_table
)


# ============================================================
# 36. DISPLAY ASSOCIATION TESTS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "FEWF TARGET ASSOCIATION TESTS"
)


print(
    "=" * 100
)


display(
    association_table
)


# ============================================================
# 37. DISPLAY FEWF FRAUD-RATE PROFILES
# ============================================================

for feature in FEWF_FEATURES:

    print(
        "\n"
        + "=" * 100
    )


    print(
        f"{feature} FRAUD-RATE PROFILE"
    )


    print(
        "=" * 100
    )


    display(
        fewf_profile_tables[
            feature
        ]
    )


# ============================================================
# 38. DISPLAY RARE VS FREQUENT SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "RARE VS FREQUENT CATEGORY PROFILE"
)


print(
    "=" * 100
)


display(
    frequency_profile_summary_table
)


# ============================================================
# 39. DISPLAY ASSOCIATION RANKING
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "FEWF TARGET-ASSOCIATION RANKING"
)


print(
    "=" * 100
)


display(
    association_ranking_table
)


print(
    "\nStrongest rank-based FEWF association:"
)


print(
    strongest_feature
)


print(
    "Rank-biserial:",
    f"{strongest_rank_biserial:.6f}"
)


print(
    "Strength:",
    strongest_rank_biserial_strength
)


# ============================================================
# 40. RELEASE MEMORY
# ============================================================

del dataset_fewf_target
del temporary_fewf
del working_data
del complete_data

gc.collect()


# ============================================================
# 41. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    CARD_FRAUD_RATE_PATH
)


print(
    NAME_FRAUD_RATE_PATH
)


print(
    JOB_FRAUD_RATE_PATH
)


print(
    RECEIVER_FRAUD_RATE_PATH
)


print(
    RANK_BISERIAL_PATH
)


FREQUENCY ENCODING WITH FALLBACK - TARGET ASSOCIATION

Total dataset observations: 1852394
Complete finite observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

TARGET OVERVIEW


,TARGET_CLASS,TARGET_VALUE,COUNT,PERCENTAGE
0,Non-fraud,0,1842743,99.478999
1,Fraud,1,9651,0.521001



TEMPORARY FEWF OVERVIEW


,FEATURE,SOURCE_DATA_TYPE,ORIGINAL_UNIQUE_CATEGORIES,ENCODED_UNIQUE_FREQUENCIES,MISSING_VALUES,MIN_OBSERVED_FREQUENCY,MEDIAN_OBSERVED_FREQUENCY,MAX_OBSERVED_FREQUENCY,FALLBACK_VALUE,FALLBACK_USES_IN_EDA
0,TRANS_NUM_CARD_FEWF,int64,999,140,0,0.000003,0.001570,0.002371,5.398420e-07,0
1,SEND_NAME_FEWF,category,989,149,0,0.000003,0.001570,0.003554,5.398420e-07,0
2,SEND_JOB_FEWF,category,497,273,0,0.000004,0.002768,0.007503,5.398420e-07,0
3,RECEIVE_LOC_FEWF,category,693,569,0,0.000588,0.001529,0.003380,5.398420e-07,0



FEWF DESCRIPTIVE STATISTICS BY TARGET


...


RARE VS FREQUENT CATEGORY PROFILE


,FEATURE,RAREST_GROUP_FRAUD_PERCENTAGE,MOST_FREQUENT_GROUP_FRAUD_PERCENTAGE,RARE_MINUS_FREQUENT_PERCENTAGE_POINTS,RARE_TO_FREQUENT_RISK_RATIO
0,TRANS_NUM_CARD_FEWF,1.646276,0.229423,1.416853,7.175725
1,SEND_NAME_FEWF,1.581008,0.253718,1.327290,6.231357
2,SEND_JOB_FEWF,0.943280,0.401449,0.541830,2.349684
3,RECEIVE_LOC_FEWF,0.619434,0.491254,0.128180,1.260924



FEWF TARGET-ASSOCIATION RANKING


,FEATURE,POINT_BISERIAL,POINT_BISERIAL_STRENGTH,RANK_BISERIAL,RANK_BISERIAL_STRENGTH,COMMON_LANGUAGE_PROBABILITY,MUTUAL_INFORMATION,NORMALIZED_MUTUAL_INFORMATION,ABS_RANK_BISERIAL
0,TRANS_NUM_CARD_FEWF,-0.050725,Very weak or negligible,-0.336964,Moderate,0.331518,0.003513,0.001528,0.336964
1,SEND_NAME_FEWF,-0.048994,Very weak or negligible,-0.331953,Moderate,0.334024,0.003464,0.001493,0.331953
2,SEND_JOB_FEWF,-0.018052,Very weak or negligible,-0.132517,Weak,0.433741,0.001266,0.000465,0.132517
3,RECEIVE_LOC_FEWF,0.006737,Very weak or negligible,0.078815,Very weak or negligible,0.539408,0.001891,0.000604,0.078815



Strongest rank-based FEWF association:
TRANS_NUM_CARD_FEWF
Rank-biserial: -0.336964
Strength: Moderate

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/frequency_encoding_with_fallback

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/frequency_encoding_with_fallback/analysis_frequency_encoding_with_fallback_target_association.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/frequency_encoding_with_fallback/fewf_target_card_frequency_fraud_rate.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/frequency_encoding_with_fallback/fewf_target_name_frequency_fraud_rate.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relat

## <span style="color:PINK"> ONEHOT ENCODING WITH IGNORE </span> ##

In [14]:

# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "02_relationships_with_target"
)

ASSOCIATION_GROUP = (
    "target_association"
)

FEATURE_GROUP = (
    "onehot_encoding_with_ignore"
)


WEEK_FEATURE = (
    "TRANS_WEEK_OHEWI"
)

CATEGORY_FEATURE = (
    "RECEIVE_CATEGORY_OHEWI"
)

TARGET_FEATURE = (
    "TARGET_OMEGA"
)


OHEWI_FEATURES = [
    WEEK_FEATURE,
    CATEGORY_FEATURE
]


REQUIRED_FEATURES = (
    OHEWI_FEATURES
    +
    [
        TARGET_FEATURE
    ]
)


WEEKDAY_ORDER = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]


TARGET_POSITIVE_VALUE = 1

ALPHA = 0.05

STANDARDIZED_RESIDUAL_THRESHOLD = 2.0

PNG_DPI = 300


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / ASSOCIATION_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_onehot_encoding_with_ignore_target_association.html"
)


WEEK_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "ohewi_target_week_fraud_rate.png"
)


CATEGORY_FRAUD_RATE_PATH = (
    RESULTS_DIRECTORY
    / "ohewi_target_receiver_category_fraud_rate.png"
)


WEEK_RESIDUAL_PATH = (
    RESULTS_DIRECTORY
    / "ohewi_target_week_fraud_residuals.png"
)


CATEGORY_RESIDUAL_PATH = (
    RESULTS_DIRECTORY
    / "ohewi_target_receiver_category_fraud_residuals.png"
)


DUMMY_PHI_PATH = (
    RESULTS_DIRECTORY
    / "ohewi_target_dummy_phi_effect.png"
)


FEATURE_FRAUD_RATE_PATHS = {

    WEEK_FEATURE:
        WEEK_FRAUD_RATE_PATH,

    CATEGORY_FEATURE:
        CATEGORY_FRAUD_RATE_PATH
}


FEATURE_RESIDUAL_PATHS = {

    WEEK_FEATURE:
        WEEK_RESIDUAL_PATH,

    CATEGORY_FEATURE:
        CATEGORY_RESIDUAL_PATH
}


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD ORIGINAL CATEGORICAL FEATURES AND TARGET
#
# The source columns preserve the original categories.
#
# One-Hot Encoding With Ignore is generated temporarily
# in memory.
#
# The parquet dataset is never modified.
# ============================================================

dataset_ohewi_target = pd.read_parquet(
    DATASET_PATH,
    columns=REQUIRED_FEATURES
)


total_observations = int(
    len(
        dataset_ohewi_target
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. VALIDATE REQUIRED FEATURES
# ============================================================

missing_features = [
    feature
    for feature in REQUIRED_FEATURES
    if feature not in dataset_ohewi_target.columns
]


if missing_features:

    raise KeyError(
        "Missing required features: "
        + ", ".join(
            missing_features
        )
    )


# ============================================================
# 09. PREPARE COMMON COMPLETE SAMPLE
#
# All analyses use the same complete observations.
# ============================================================

analysis_data = (
    dataset_ohewi_target[
        REQUIRED_FEATURES
    ]
    .dropna()
    .copy()
)


analysis_observations = int(
    len(
        analysis_data
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete observations are available."
    )


excluded_observations = (
    total_observations
    -
    analysis_observations
)


excluded_percentage = (
    excluded_observations
    /
    total_observations
    *
    100
)


# ============================================================
# 10. VALIDATE TARGET
# ============================================================

target_values = (
    analysis_data[
        TARGET_FEATURE
    ]
    .drop_duplicates()
    .tolist()
)


if len(
    target_values
) != 2:

    raise ValueError(
        f"{TARGET_FEATURE} must contain exactly two classes. "
        f"Observed values: {target_values}"
    )


target_positive_value = None


for value in target_values:

    if (
        value == TARGET_POSITIVE_VALUE
        or
        str(
            value
        )
        == str(
            TARGET_POSITIVE_VALUE
        )
    ):

        target_positive_value = (
            value
        )

        break


if target_positive_value is None:

    raise ValueError(
        f"Fraud target value {TARGET_POSITIVE_VALUE} was not found. "
        f"Observed values: {target_values}"
    )


target_negative_values = [
    value
    for value in target_values
    if value != target_positive_value
]


if len(
    target_negative_values
) != 1:

    raise ValueError(
        "Unable to determine the non-fraud target class."
    )


target_negative_value = (
    target_negative_values[
        0
    ]
)


# ============================================================
# 11. CREATE TEMPORARY BINARY TARGET
#
# Non-fraud = 0
# Fraud     = 1
# ============================================================

analysis_data[
    "_TARGET_BINARY"
] = (
    analysis_data[
        TARGET_FEATURE
    ]
    .map({

        target_negative_value:
            0,

        target_positive_value:
            1
    })
    .astype(
        "int8"
    )
)


# ============================================================
# 12. TARGET OVERVIEW
# ============================================================

target_counts = (
    analysis_data[
        "_TARGET_BINARY"
    ]
    .value_counts()
    .reindex(
        [
            0,
            1
        ],
        fill_value=0
    )
)


non_fraud_count = int(
    target_counts.loc[
        0
    ]
)


fraud_count = int(
    target_counts.loc[
        1
    ]
)


non_fraud_percentage = (
    non_fraud_count
    /
    analysis_observations
    *
    100
)


fraud_percentage = (
    fraud_count
    /
    analysis_observations
    *
    100
)


overall_fraud_rate = (
    fraud_count
    /
    analysis_observations
)


target_overview_table = pd.DataFrame({

    "TARGET_CLASS": [
        "Non-fraud",
        "Fraud"
    ],

    "TARGET_VALUE": [
        target_negative_value,
        target_positive_value
    ],

    "COUNT": [
        non_fraud_count,
        fraud_count
    ],

    "PERCENTAGE": [
        non_fraud_percentage,
        fraud_percentage
    ]
})


# ============================================================
# 13. SOURCE FEATURE OVERVIEW
# ============================================================

source_overview_records = []


for feature in OHEWI_FEATURES:

    source_series = (
        analysis_data[
            feature
        ]
    )


    source_overview_records.append({

        "FEATURE":
            feature,

        "SOURCE_DATA_TYPE":
            str(
                source_series.dtype
            ),

        "UNIQUE_CATEGORIES":
            int(
                source_series.nunique()
            ),

        "MIN_CATEGORY_COUNT":
            int(
                source_series
                .value_counts()
                .min()
            ),

        "MAX_CATEGORY_COUNT":
            int(
                source_series
                .value_counts()
                .max()
            )
    })


source_overview_table = pd.DataFrame(
    source_overview_records
)


# ============================================================
# 14. CREATE TEMPORARY ONE-HOT ENCODER
#
# handle_unknown="ignore":
#
# Unknown categories are represented by an all-zero vector
# inside their corresponding feature group.
#
# No unknown category is expected in this EDA because
# fit and transform use the same exploratory dataset.
# ============================================================

ohe_encoder = OneHotEncoder(

    handle_unknown="ignore",

    sparse_output=True,

    dtype=np.uint8
)


temporary_ohe_matrix = (
    ohe_encoder.fit_transform(
        analysis_data[
            OHEWI_FEATURES
        ]
    )
)


encoder_categories = (
    ohe_encoder.categories_
)


generated_feature_names = (
    ohe_encoder.get_feature_names_out(
        OHEWI_FEATURES
    )
)


generated_rows = int(
    temporary_ohe_matrix.shape[
        0
    ]
)


generated_dummy_columns = int(
    temporary_ohe_matrix.shape[
        1
    ]
)


nonzero_entries = int(
    temporary_ohe_matrix.nnz
)


total_matrix_cells = int(
    generated_rows
    *
    generated_dummy_columns
)


matrix_density = (
    nonzero_entries
    /
    total_matrix_cells
)


matrix_sparsity = (
    1.0
    -
    matrix_density
)


active_values_per_observation = np.asarray(
    temporary_ohe_matrix.sum(
        axis=1
    )
).ravel()


# ============================================================
# 15. OHE FEATURE STRUCTURE
# ============================================================

ohe_feature_structure_records = []


for (
    feature,
    categories
) in zip(
    OHEWI_FEATURES,
    encoder_categories
):

    ohe_feature_structure_records.append({

        "FEATURE":
            feature,

        "SOURCE_CATEGORIES":
            int(
                len(
                    categories
                )
            ),

        "GENERATED_DUMMY_COLUMNS":
            int(
                len(
                    categories
                )
            )
    })


ohe_feature_structure_table = pd.DataFrame(
    ohe_feature_structure_records
)


ohe_matrix_structure_table = pd.DataFrame({

    "METRIC": [
        "Original categorical features",
        "Observations encoded",
        "Generated dummy columns",
        "Non-zero matrix entries",
        "Total matrix cells",
        "Matrix density",
        "Matrix sparsity",
        "Minimum active values per observation",
        "Mean active values per observation",
        "Median active values per observation",
        "Maximum active values per observation"
    ],

    "VALUE": [
        len(
            OHEWI_FEATURES
        ),

        generated_rows,

        generated_dummy_columns,

        nonzero_entries,

        total_matrix_cells,

        matrix_density,

        matrix_sparsity,

        float(
            np.min(
                active_values_per_observation
            )
        ),

        float(
            np.mean(
                active_values_per_observation
            )
        ),

        float(
            np.median(
                active_values_per_observation
            )
        ),

        float(
            np.max(
                active_values_per_observation
            )
        )
    ]
})


# ============================================================
# 16. ASSOCIATION STRENGTH INTERPRETATION
#
# Descriptive exploratory guidelines only.
# ============================================================

def interpret_association_strength(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        return (
            "Weak"
        )


    elif absolute_value < 0.50:

        return (
            "Moderate"
        )


    elif absolute_value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


# ============================================================
# 17. CATEGORY ORDER FUNCTION
# ============================================================

def get_category_order(
    dataframe,
    feature
):

    observed_categories = (
        dataframe[
            feature
        ]
        .drop_duplicates()
        .tolist()
    )


    if feature == WEEK_FEATURE:

        ordered_categories = [
            weekday
            for weekday in WEEKDAY_ORDER
            if weekday in observed_categories
        ]


        additional_categories = sorted(
            [
                category
                for category in observed_categories
                if category not in WEEKDAY_ORDER
            ],
            key=str
        )


        ordered_categories.extend(
            additional_categories
        )


        return (
            ordered_categories
        )


    return (
        dataframe[
            feature
        ]
        .value_counts()
        .index
        .tolist()
    )


# ============================================================
# 18. CATEGORICAL FEATURE × TARGET ANALYSIS FUNCTION
# ============================================================

def analyze_categorical_target(
    dataframe,
    feature,
    category_order
):

    # --------------------------------------------------------
    # CONTINGENCY TABLE
    # --------------------------------------------------------

    contingency_table = pd.crosstab(

        dataframe[
            feature
        ],

        dataframe[
            "_TARGET_BINARY"
        ]
    )


    contingency_table = (
        contingency_table
        .reindex(
            index=category_order,
            columns=[
                0,
                1
            ],
            fill_value=0
        )
    )


    contingency_table.columns = [
        "NON_FRAUD",
        "FRAUD"
    ]


    category_totals = (
        contingency_table
        .sum(
            axis=1
        )
    )


    # --------------------------------------------------------
    # FRAUD RATE
    # --------------------------------------------------------

    fraud_rate = np.divide(

        contingency_table[
            "FRAUD"
        ]
        .to_numpy(
            dtype="float64"
        ),

        category_totals
        .to_numpy(
            dtype="float64"
        ),

        out=np.full(
            len(
                contingency_table
            ),
            np.nan,
            dtype="float64"
        ),

        where=(
            category_totals
            .to_numpy(
                dtype="float64"
            )
            > 0
        )
    )


    fraud_rate_table = pd.DataFrame({

        "CATEGORY":
            contingency_table.index,

        "TOTAL":
            category_totals
            .astype(
                int
            )
            .to_numpy(),

        "NON_FRAUD":
            contingency_table[
                "NON_FRAUD"
            ]
            .astype(
                int
            )
            .to_numpy(),

        "FRAUD":
            contingency_table[
                "FRAUD"
            ]
            .astype(
                int
            )
            .to_numpy(),

        "FRAUD_RATE":
            fraud_rate,

        "FRAUD_PERCENTAGE":
            fraud_rate
            *
            100,

        "FRAUD_RATE_LIFT":
            fraud_rate
            /
            overall_fraud_rate
    })


    # --------------------------------------------------------
    # CHI-SQUARE
    # --------------------------------------------------------

    observed_matrix = (
        contingency_table
        .to_numpy(
            dtype="float64"
        )
    )


    (
        chi_square_statistic,
        chi_square_p_value,
        chi_square_degrees_of_freedom,
        expected_matrix
    ) = stats.chi2_contingency(

        observed_matrix,

        correction=False
    )


    chi_square_statistic = float(
        chi_square_statistic
    )


    chi_square_p_value = float(
        chi_square_p_value
    )


    chi_square_degrees_of_freedom = int(
        chi_square_degrees_of_freedom
    )


    expected_matrix = np.asarray(
        expected_matrix,
        dtype="float64"
    )


    minimum_expected_frequency = float(
        np.min(
            expected_matrix
        )
    )


    cells_expected_below_5 = int(
        np.sum(
            expected_matrix
            <
            5
        )
    )


    total_expected_cells = int(
        expected_matrix.size
    )


    expected_below_5_percentage = (
        cells_expected_below_5
        /
        total_expected_cells
        *
        100
    )


    # --------------------------------------------------------
    # CRAMER'S V
    # --------------------------------------------------------

    number_rows = int(
        observed_matrix.shape[
            0
        ]
    )


    number_columns = int(
        observed_matrix.shape[
            1
        ]
    )


    cramers_dimension = min(
        number_rows - 1,
        number_columns - 1
    )


    if cramers_dimension > 0:

        cramers_v = float(
            np.sqrt(

                chi_square_statistic

                /

                (
                    observed_matrix.sum()
                    *
                    cramers_dimension
                )
            )
        )


    else:

        cramers_v = (
            np.nan
        )


    cramers_v_strength = (
        interpret_association_strength(
            cramers_v
        )
    )


    # --------------------------------------------------------
    # ADJUSTED STANDARDIZED RESIDUALS
    # --------------------------------------------------------

    row_totals = (
        observed_matrix
        .sum(
            axis=1,
            keepdims=True
        )
    )


    column_totals = (
        observed_matrix
        .sum(
            axis=0,
            keepdims=True
        )
    )


    grand_total = float(
        observed_matrix.sum()
    )


    row_proportions = (
        row_totals
        /
        grand_total
    )


    column_proportions = (
        column_totals
        /
        grand_total
    )


    residual_denominator = np.sqrt(

        expected_matrix

        *

        (
            1
            -
            row_proportions
        )

        *

        (
            1
            -
            column_proportions
        )
    )


    adjusted_residual_matrix = np.divide(

        observed_matrix
        -
        expected_matrix,

        residual_denominator,

        out=np.full_like(
            observed_matrix,
            np.nan,
            dtype="float64"
        ),

        where=(
            residual_denominator
            >
            0
        )
    )


    adjusted_residuals_table = pd.DataFrame(

        adjusted_residual_matrix,

        index=contingency_table.index,

        columns=[
            "NON_FRAUD",
            "FRAUD"
        ]
    )


    flagged_residual_cells = int(
        np.sum(
            np.abs(
                adjusted_residual_matrix
            )
            >=
            STANDARDIZED_RESIDUAL_THRESHOLD
        )
    )


    # --------------------------------------------------------
    # MUTUAL INFORMATION
    # --------------------------------------------------------

    feature_codes, _ = pd.factorize(
        dataframe[
            feature
        ],
        sort=True
    )


    target_codes = (
        dataframe[
            "_TARGET_BINARY"
        ]
        .to_numpy(
            dtype="int8"
        )
    )


    mutual_information = float(
        mutual_info_score(
            feature_codes,
            target_codes
        )
    )


    normalized_mutual_information = float(
        normalized_mutual_info_score(

            feature_codes,

            target_codes,

            average_method="arithmetic"
        )
    )


    # --------------------------------------------------------
    # CATEGORY-VS-REST DUMMY EFFECTS
    #
    # Each category is temporarily treated exactly as the
    # binary dummy produced by One-Hot Encoding:
    #
    # category present = 1
    # all other categories = 0
    #
    # This allows direct evaluation of each OHE dummy
    # against TARGET_OMEGA.
    # --------------------------------------------------------

    dummy_effect_records = []


    target_binary = (
        dataframe[
            "_TARGET_BINARY"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    for category in category_order:

        dummy_values = (
            dataframe[
                feature
            ]
            .eq(
                category
            )
            .astype(
                "int8"
            )
            .to_numpy(
                dtype="float64"
            )
        )


        category_present = (
            dummy_values
            ==
            1
        )


        category_absent = (
            dummy_values
            ==
            0
        )


        a = float(
            np.sum(
                category_present
                &
                (
                    target_binary
                    ==
                    1
                )
            )
        )


        b = float(
            np.sum(
                category_present
                &
                (
                    target_binary
                    ==
                    0
                )
            )
        )


        c = float(
            np.sum(
                category_absent
                &
                (
                    target_binary
                    ==
                    1
                )
            )
        )


        d = float(
            np.sum(
                category_absent
                &
                (
                    target_binary
                    ==
                    0
                )
            )
        )


        # ----------------------------------------------------
        # SIGNED PHI
        # ----------------------------------------------------

        if (
            np.std(
                dummy_values
            )
            >
            0
        ):

            phi = float(
                np.corrcoef(
                    dummy_values,
                    target_binary
                )[
                    0,
                    1
                ]
            )


        else:

            phi = (
                np.nan
            )


        phi_strength = (
            interpret_association_strength(
                phi
            )
        )


        # ----------------------------------------------------
        # FRAUD RISKS
        # ----------------------------------------------------

        category_total = (
            a
            +
            b
        )


        rest_total = (
            c
            +
            d
        )


        category_fraud_risk = (
            a
            /
            category_total
            if category_total > 0
            else np.nan
        )


        rest_fraud_risk = (
            c
            /
            rest_total
            if rest_total > 0
            else np.nan
        )


        if (
            not pd.isna(
                rest_fraud_risk
            )
            and
            rest_fraud_risk
            >
            0
        ):

            risk_ratio = (
                category_fraud_risk
                /
                rest_fraud_risk
            )


        else:

            risk_ratio = (
                np.nan
            )


        risk_difference = (
            category_fraud_risk
            -
            rest_fraud_risk
        )


        # ----------------------------------------------------
        # ODDS RATIO
        #
        # Haldane-Anscombe correction is applied only
        # when at least one 2x2 cell is zero.
        # ----------------------------------------------------

        zero_cell_correction = bool(
            min(
                a,
                b,
                c,
                d
            )
            ==
            0
        )


        if zero_cell_correction:

            a_or = (
                a
                +
                0.5
            )

            b_or = (
                b
                +
                0.5
            )

            c_or = (
                c
                +
                0.5
            )

            d_or = (
                d
                +
                0.5
            )


        else:

            a_or = a
            b_or = b
            c_or = c
            d_or = d


        odds_ratio = float(
            (
                a_or
                *
                d_or
            )
            /
            (
                b_or
                *
                c_or
            )
        )


        log_or_standard_error = float(
            np.sqrt(

                1.0
                /
                a_or

                +

                1.0
                /
                b_or

                +

                1.0
                /
                c_or

                +

                1.0
                /
                d_or
            )
        )


        odds_ratio_ci_lower = float(
            np.exp(

                np.log(
                    odds_ratio
                )

                -

                1.96
                *
                log_or_standard_error
            )
        )


        odds_ratio_ci_upper = float(
            np.exp(

                np.log(
                    odds_ratio
                )

                +

                1.96
                *
                log_or_standard_error
            )
        )


        # ----------------------------------------------------
        # FRAUD RESIDUAL FOR CURRENT CATEGORY
        # ----------------------------------------------------

        category_position = (
            list(
                contingency_table.index
            )
            .index(
                category
            )
        )


        fraud_adjusted_residual = float(
            adjusted_residual_matrix[
                category_position,
                1
            ]
        )


        dummy_effect_records.append({

            "SOURCE_FEATURE":
                feature,

            "CATEGORY":
                category,

            "TOTAL":
                int(
                    category_total
                ),

            "FRAUD":
                int(
                    a
                ),

            "FRAUD_RATE":
                category_fraud_risk,

            "FRAUD_PERCENTAGE":
                category_fraud_risk
                *
                100,

            "FRAUD_RATE_LIFT":
                category_fraud_risk
                /
                overall_fraud_rate,

            "PHI":
                phi,

            "PHI_STRENGTH":
                phi_strength,

            "ODDS_RATIO_CATEGORY_VS_REST":
                odds_ratio,

            "OR_CI_95_LOWER":
                odds_ratio_ci_lower,

            "OR_CI_95_UPPER":
                odds_ratio_ci_upper,

            "RISK_RATIO_CATEGORY_VS_REST":
                risk_ratio,

            "RISK_DIFFERENCE_PERCENTAGE_POINTS":
                risk_difference
                *
                100,

            "FRAUD_ADJUSTED_RESIDUAL":
                fraud_adjusted_residual,

            "ZERO_CELL_CORRECTION_USED":
                (
                    "Yes"
                    if zero_cell_correction
                    else "No"
                )
        })


    dummy_effect_table = pd.DataFrame(
        dummy_effect_records
    )


    # --------------------------------------------------------
    # CHI-SQUARE DECISION
    # --------------------------------------------------------

    if chi_square_p_value < ALPHA:

        chi_square_decision = (
            "Reject the null hypothesis"
        )


    else:

        chi_square_decision = (
            "Do not reject the null hypothesis"
        )


    return {

        "contingency_table":
            contingency_table,

        "fraud_rate_table":
            fraud_rate_table,

        "chi_square_statistic":
            chi_square_statistic,

        "chi_square_p_value":
            chi_square_p_value,

        "degrees_of_freedom":
            chi_square_degrees_of_freedom,

        "chi_square_decision":
            chi_square_decision,

        "cramers_v":
            cramers_v,

        "cramers_v_strength":
            cramers_v_strength,

        "minimum_expected_frequency":
            minimum_expected_frequency,

        "cells_expected_below_5":
            cells_expected_below_5,

        "expected_below_5_percentage":
            expected_below_5_percentage,

        "adjusted_residuals_table":
            adjusted_residuals_table,

        "flagged_residual_cells":
            flagged_residual_cells,

        "mutual_information":
            mutual_information,

        "normalized_mutual_information":
            normalized_mutual_information,

        "dummy_effect_table":
            dummy_effect_table
    }


# ============================================================
# 19. ANALYZE BOTH OHEWI SOURCE FEATURES
# ============================================================

feature_results = {}


for feature in OHEWI_FEATURES:

    category_order = (
        get_category_order(
            analysis_data,
            feature
        )
    )


    feature_results[
        feature
    ] = (
        analyze_categorical_target(

            dataframe=analysis_data,

            feature=feature,

            category_order=category_order
        )
    )


# ============================================================
# 20. COMPARATIVE SOURCE-FEATURE SUMMARY
# ============================================================

association_summary_records = []


for feature in OHEWI_FEATURES:

    result = (
        feature_results[
            feature
        ]
    )


    association_summary_records.append({

        "FEATURE":
            feature,

        "NUMBER_OF_CATEGORIES":
            int(
                analysis_data[
                    feature
                ]
                .nunique()
            ),

        "CHI_SQUARE":
            result[
                "chi_square_statistic"
            ],

        "DEGREES_OF_FREEDOM":
            result[
                "degrees_of_freedom"
            ],

        "P_VALUE":
            result[
                "chi_square_p_value"
            ],

        "CRAMERS_V":
            result[
                "cramers_v"
            ],

        "CRAMERS_V_STRENGTH":
            result[
                "cramers_v_strength"
            ],

        "MUTUAL_INFORMATION":
            result[
                "mutual_information"
            ],

        "NORMALIZED_MUTUAL_INFORMATION":
            result[
                "normalized_mutual_information"
            ],

        "FLAGGED_RESIDUAL_CELLS":
            result[
                "flagged_residual_cells"
            ]
    })


association_summary_table = pd.DataFrame(
    association_summary_records
)


association_summary_table[
    "ABS_CRAMERS_V"
] = (
    association_summary_table[
        "CRAMERS_V"
    ]
    .abs()
)


association_summary_table = (
    association_summary_table
    .sort_values(
        by="ABS_CRAMERS_V",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 21. COMBINE ALL DUMMY EFFECT TABLES
# ============================================================

dummy_effect_table = pd.concat(

    [
        feature_results[
            feature
        ][
            "dummy_effect_table"
        ]

        for feature in OHEWI_FEATURES
    ],

    ignore_index=True
)


dummy_effect_table[
    "ABS_PHI"
] = (
    dummy_effect_table[
        "PHI"
    ]
    .abs()
)


dummy_effect_ranking_table = (
    dummy_effect_table
    .sort_values(
        by="ABS_PHI",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 22. IDENTIFY STRONGEST SOURCE FEATURE
# ============================================================

strongest_source_row = (
    association_summary_table
    .iloc[
        0
    ]
)


strongest_source_feature = (
    strongest_source_row[
        "FEATURE"
    ]
)


strongest_source_cramers_v = float(
    strongest_source_row[
        "CRAMERS_V"
    ]
)


strongest_source_strength = (
    strongest_source_row[
        "CRAMERS_V_STRENGTH"
    ]
)


# ============================================================
# 23. IDENTIFY STRONGEST INDIVIDUAL DUMMY
# ============================================================

strongest_dummy_row = (
    dummy_effect_ranking_table
    .iloc[
        0
    ]
)


strongest_dummy_source = (
    strongest_dummy_row[
        "SOURCE_FEATURE"
    ]
)


strongest_dummy_category = (
    strongest_dummy_row[
        "CATEGORY"
    ]
)


strongest_dummy_phi = float(
    strongest_dummy_row[
        "PHI"
    ]
)


strongest_dummy_phi_strength = (
    strongest_dummy_row[
        "PHI_STRENGTH"
    ]
)


# ============================================================
# 24. FRAUD-RATE PLOT FUNCTION
# ============================================================

def create_fraud_rate_plot(
    fraud_rate_table,
    feature,
    output_path
):

    plot_table = (
        fraud_rate_table
        .copy()
    )


    x_values = np.arange(
        len(
            plot_table
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            12,
            7
        )
    )


    ax.bar(

        x_values,

        plot_table[
            "FRAUD_PERCENTAGE"
        ]
    )


    ax.axhline(

        overall_fraud_rate
        *
        100,

        linestyle="--",

        linewidth=1.5,

        label="Overall fraud rate"
    )


    ax.set_xticks(
        x_values
    )


    ax.set_xticklabels(

        plot_table[
            "CATEGORY"
        ]
        .astype(
            str
        ),

        rotation=45,

        ha="right"
    )


    ax.set_xlabel(
        feature
    )


    ax.set_ylabel(
        "Fraud rate (%)"
    )


    ax.set_title(
        f"Fraud rate by {feature}"
    )


    ax.legend()


    ax.grid(
        axis="y",
        alpha=0.20
    )


    fig.tight_layout()


    fig.savefig(

        output_path,

        format="png",

        dpi=PNG_DPI,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 25. FRAUD-RESIDUAL PLOT FUNCTION
#
# Only the Fraud column is plotted because the target is
# binary and the Non-fraud residual contains the
# complementary direction.
# ============================================================

def create_fraud_residual_plot(
    residual_table,
    feature,
    output_path
):

    fraud_residuals = (
        residual_table[
            "FRAUD"
        ]
    )


    x_values = np.arange(
        len(
            fraud_residuals
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            12,
            7
        )
    )


    ax.bar(

        x_values,

        fraud_residuals
        .to_numpy(
            dtype="float64"
        )
    )


    ax.axhline(
        0,
        linewidth=1
    )


    ax.axhline(

        STANDARDIZED_RESIDUAL_THRESHOLD,

        linestyle="--",

        linewidth=1
    )


    ax.axhline(

        -STANDARDIZED_RESIDUAL_THRESHOLD,

        linestyle="--",

        linewidth=1
    )


    ax.set_xticks(
        x_values
    )


    ax.set_xticklabels(

        residual_table.index
        .astype(
            str
        ),

        rotation=45,

        ha="right"
    )


    ax.set_xlabel(
        feature
    )


    ax.set_ylabel(
        "Fraud adjusted standardized residual"
    )


    ax.set_title(
        f"Fraud residuals by {feature}"
    )


    ax.grid(
        axis="y",
        alpha=0.20
    )


    fig.tight_layout()


    fig.savefig(

        output_path,

        format="png",

        dpi=PNG_DPI,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 26. CREATE SOURCE-FEATURE PLOTS
# ============================================================

for feature in OHEWI_FEATURES:

    result = (
        feature_results[
            feature
        ]
    )


    create_fraud_rate_plot(

        fraud_rate_table=result[
            "fraud_rate_table"
        ],

        feature=feature,

        output_path=FEATURE_FRAUD_RATE_PATHS[
            feature
        ]
    )


    create_fraud_residual_plot(

        residual_table=result[
            "adjusted_residuals_table"
        ],

        feature=feature,

        output_path=FEATURE_RESIDUAL_PATHS[
            feature
        ]
    )


# ============================================================
# 27. CREATE ALL-DUMMY PHI EFFECT PLOT
# ============================================================

dummy_phi_plot_table = (
    dummy_effect_ranking_table
    .sort_values(
        by="PHI",
        ascending=True
    )
    .copy()
)


dummy_phi_plot_table[
    "DUMMY_LABEL"
] = (

    dummy_phi_plot_table[
        "SOURCE_FEATURE"
    ]
    .astype(
        str
    )

    +

    " = "

    +

    dummy_phi_plot_table[
        "CATEGORY"
    ]
    .astype(
        str
    )
)


fig_height = max(
    8,
    len(
        dummy_phi_plot_table
    )
    *
    0.40
)


fig, ax = plt.subplots(
    figsize=(
        12,
        fig_height
    )
)


ax.barh(

    dummy_phi_plot_table[
        "DUMMY_LABEL"
    ],

    dummy_phi_plot_table[
        "PHI"
    ]
)


ax.axvline(
    0,
    linewidth=1
)


ax.set_xlabel(
    "Phi coefficient with TARGET_OMEGA"
)


ax.set_ylabel(
    "Temporary One-Hot dummy"
)


ax.set_title(
    "One-Hot dummy association with fraud"
)


ax.grid(
    axis="x",
    alpha=0.20
)


fig.tight_layout()


fig.savefig(

    DUMMY_PHI_PATH,

    format="png",

    dpi=PNG_DPI,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 28. IMAGE TO BASE64 FUNCTION
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 29. CONVERT IMAGES TO BASE64
# ============================================================

feature_fraud_rate_base64 = {}

feature_residual_base64 = {}


for feature in OHEWI_FEATURES:

    feature_fraud_rate_base64[
        feature
    ] = image_to_base64(
        FEATURE_FRAUD_RATE_PATHS[
            feature
        ]
    )


    feature_residual_base64[
        feature
    ] = image_to_base64(
        FEATURE_RESIDUAL_PATHS[
            feature
        ]
    )


dummy_phi_base64 = (
    image_to_base64(
        DUMMY_PHI_PATH
    )
)


# ============================================================
# 30. PREPARE GENERAL HTML TABLES
# ============================================================

target_overview_html = (
    target_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PERCENTAGE":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


source_overview_html = (
    source_overview_table
    .to_html(
        index=False,
        border=0
    )
)


ohe_feature_structure_html = (
    ohe_feature_structure_table
    .to_html(
        index=False,
        border=0
    )
)


ohe_matrix_structure_html = (
    ohe_matrix_structure_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "VALUE":
                lambda value:
                    (
                        f"{value:.12f}"
                        if isinstance(
                            value,
                            float
                        )
                        else str(
                            value
                        )
                    )
        }
    )
)


association_summary_html = (
    association_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "CHI_SQUARE":
                lambda value:
                    f"{value:.6f}",

            "P_VALUE":
                lambda value:
                    f"{value:.12g}",

            "CRAMERS_V":
                lambda value:
                    f"{value:.6f}",

            "MUTUAL_INFORMATION":
                lambda value:
                    f"{value:.8f}",

            "NORMALIZED_MUTUAL_INFORMATION":
                lambda value:
                    f"{value:.8f}",

            "ABS_CRAMERS_V":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


dummy_effect_html = (
    dummy_effect_ranking_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "FRAUD_RATE":
                lambda value:
                    f"{value:.8f}",

            "FRAUD_PERCENTAGE":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_RATE_LIFT":
                lambda value:
                    f"{value:.6f}",

            "PHI":
                lambda value:
                    (
                        "NaN"
                        if pd.isna(
                            value
                        )
                        else f"{value:.6f}"
                    ),

            "ODDS_RATIO_CATEGORY_VS_REST":
                lambda value:
                    f"{value:.6f}",

            "OR_CI_95_LOWER":
                lambda value:
                    f"{value:.6f}",

            "OR_CI_95_UPPER":
                lambda value:
                    f"{value:.6f}",

            "RISK_RATIO_CATEGORY_VS_REST":
                lambda value:
                    (
                        "NaN"
                        if pd.isna(
                            value
                        )
                        else f"{value:.6f}"
                    ),

            "RISK_DIFFERENCE_PERCENTAGE_POINTS":
                lambda value:
                    f"{value:.6f}",

            "FRAUD_ADJUSTED_RESIDUAL":
                lambda value:
                    f"{value:.6f}",

            "ABS_PHI":
                lambda value:
                    (
                        "NaN"
                        if pd.isna(
                            value
                        )
                        else f"{value:.6f}"
                    )
        }
    )
)


# ============================================================
# 31. PREPARE FEATURE-SPECIFIC HTML SECTIONS
# ============================================================

feature_html_sections = []


for section_number, feature in enumerate(
    OHEWI_FEATURES,
    start=3
):

    result = (
        feature_results[
            feature
        ]
    )


    contingency_html = (
        result[
            "contingency_table"
        ]
        .to_html(
            border=0
        )
    )


    fraud_rate_html = (
        result[
            "fraud_rate_table"
        ]
        .to_html(
            index=False,
            border=0,
            formatters={

                "FRAUD_RATE":
                    lambda value:
                        f"{value:.8f}",

                "FRAUD_PERCENTAGE":
                    lambda value:
                        f"{value:.6f}",

                "FRAUD_RATE_LIFT":
                    lambda value:
                        f"{value:.6f}"
            }
        )
    )


    residual_html = (
        result[
            "adjusted_residuals_table"
        ]
        .to_html(
            border=0,
            float_format=lambda value:
                f"{value:.6f}"
        )
    )


    fraud_rate_image = (
        feature_fraud_rate_base64[
            feature
        ]
    )


    residual_image = (
        feature_residual_base64[
            feature
        ]
    )


    feature_html_sections.append(
        f"""

        <h2>
        {section_number}. {feature} × {TARGET_FEATURE}
        </h2>


        <h3>
        Contingency table
        </h3>


        <div class="table-container">

        {contingency_html}

        </div>


        <h3>
        Fraud rate by category
        </h3>


        <div class="table-container">

        {fraud_rate_html}

        </div>


        <div class="chart">

        <img
            src="data:image/png;base64,{fraud_rate_image}"
            alt="{feature} fraud rate"
        >

        </div>


        <div class="note">

        Fraud-rate lift is defined as:

        <br><br>

        <strong>
        category fraud rate / overall fraud rate
        </strong>

        <br><br>

        Values greater than 1 indicate fraud prevalence
        above the dataset-wide fraud rate.

        </div>


        <h3>
        Global categorical association
        </h3>


        <p class="result">

        Chi-square:
        {result["chi_square_statistic"]:.6f}

        <br>

        Degrees of freedom:
        {result["degrees_of_freedom"]}

        <br>

        p-value:
        {result["chi_square_p_value"]:.12g}

        <br>

        Decision:
        {result["chi_square_decision"]}

        <br><br>

        Cramer's V:
        {result["cramers_v"]:.6f}

        <br>

        Strength:
        {result["cramers_v_strength"]}

        <br><br>

        Mutual Information:
        {result["mutual_information"]:.8f}

        <br>

        Normalized Mutual Information:
        {result["normalized_mutual_information"]:.8f}

        </p>


        <div class="note">

        Minimum expected frequency:

        <strong>
        {result["minimum_expected_frequency"]:.6f}
        </strong>

        <br>

        Cells with expected frequency below 5:

        <strong>
        {result["cells_expected_below_5"]}
        </strong>

        <br>

        Percentage of cells with expected frequency below 5:

        <strong>
        {result["expected_below_5_percentage"]:.6f}%
        </strong>

        <br><br>

        With a very large dataset, very small associations
        can produce very small p-values.

        Cramer's V, fraud-rate differences and individual
        dummy effects should therefore receive more
        interpretive emphasis.

        </div>


        <h3>
        Adjusted standardized residuals
        </h3>


        <div class="table-container">

        {residual_html}

        </div>


        <div class="chart">

        <img
            src="data:image/png;base64,{residual_image}"
            alt="{feature} fraud residuals"
        >

        </div>


        <div class="note">

        Positive fraud residuals indicate categories with
        more fraud observations than expected under
        statistical independence.

        Negative values indicate fewer fraud observations
        than expected.

        <br><br>

        The exploratory reference is:

        <strong>
        |residual| >= {STANDARDIZED_RESIDUAL_THRESHOLD:.1f}
        </strong>.

        This criterion is descriptive only.

        </div>

        """
    )


feature_html_content = "\n".join(
    feature_html_sections
)


# ============================================================
# 32. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
One-Hot Encoding With Ignore - Target Association
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1500px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 13px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 8px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Target Association —
One-Hot Encoding With Ignore
</h1>


<p>

Categorical source features:

</p>


<ul>

<li>{WEEK_FEATURE}</li>
<li>{CATEGORY_FEATURE}</li>

</ul>


<p>

Target:

<strong>
{TARGET_FEATURE}
</strong>

</p>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. TARGET AND SOURCE OVERVIEW
========================================================= -->


<h2>
1. Target and categorical feature overview
</h2>


<div class="table-container">

{target_overview_html}

</div>


<p class="result">

Overall fraud rate:
{fraud_percentage:.6f}%

</p>


<div class="table-container">

{source_overview_html}

</div>


<div class="note">

The original categorical labels are preserved in the
parquet dataset.

TARGET_OMEGA is used only as an outcome variable.

It must not be included in the explanatory feature matrix
used to construct PCA, t-SNE or GMM.

</div>


<!-- ========================================================
     2. TEMPORARY OHE STRUCTURE
========================================================= -->


<h2>
2. Temporary One-Hot Encoding With Ignore structure
</h2>


<div class="table-container">

{ohe_feature_structure_html}

</div>


<div class="table-container">

{ohe_matrix_structure_html}

</div>


<div class="note">

The temporary One-Hot representation generates

<strong>
{generated_dummy_columns}
</strong>

binary columns.

<br><br>

Matrix density:

<strong>
{matrix_density:.12f}
</strong>

<br>

Matrix sparsity:

<strong>
{matrix_sparsity:.12f}
</strong>

<br><br>

Because every complete observation contains one weekday and
one receiver category, approximately two dummy values are
active per observation.

</div>


<div class="note">

The encoder uses:

<br><br>

<strong>
handle_unknown="ignore"
</strong>

<br><br>

No unknown category is expected in this EDA because fitting
and transformation use the same exploratory dataset.

In future or held-out data, an unseen category produces an
all-zero vector within its corresponding dummy group.

</div>


{feature_html_content}


<!-- ========================================================
     5. INDIVIDUAL DUMMY EFFECTS
========================================================= -->


<h2>
5. Individual One-Hot dummy effects
</h2>


<p>

Each original category is temporarily treated as the binary
dummy that One-Hot Encoding would produce:

</p>


<p class="result">

category present = 1

<br>

all other categories = 0

</p>


<div class="table-container">

{dummy_effect_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{dummy_phi_base64}"
    alt="Individual OHE dummy association with target"
>

</div>


<div class="note">

The signed Phi coefficient indicates the direction of
association between each individual temporary dummy and
fraud.

<br><br>

Positive Phi indicates that the presence of the category is
associated with relatively more fraud.

Negative Phi indicates relatively less fraud.

<br><br>

Odds Ratio compares the fraud odds for observations in the
selected category with the fraud odds for all remaining
categories of the same source feature.

Risk Ratio and Risk Difference provide complementary
probability-based interpretations.

</div>


<!-- ========================================================
     6. SOURCE FEATURE COMPARISON
========================================================= -->


<h2>
6. Source-feature target-association comparison
</h2>


<div class="table-container">

{association_summary_html}

</div>


<p class="result">

Strongest source-feature association:

<br>

{strongest_source_feature}

<br>

Cramer's V:
{strongest_source_cramers_v:.6f}

<br>

Strength:
{strongest_source_strength}

</p>


<p class="result">

Strongest individual dummy association:

<br>

{strongest_dummy_source}
=
{strongest_dummy_category}

<br>

Phi:
{strongest_dummy_phi:.6f}

<br>

Strength:
{strongest_dummy_phi_strength}

</p>


<div class="note">

The source-level and dummy-level analyses answer different
questions.

<br><br>

Cramer's V evaluates the overall association between the
complete categorical feature and fraud.

Phi evaluates the association between one specific One-Hot
dummy and fraud.

A globally weak categorical association can still contain
individual categories with comparatively different fraud
rates.

</div>


<!-- ========================================================
     7. MODELING IMPLICATIONS
========================================================= -->


<h2>
7. Potential modeling implications
</h2>


<div class="note">

<strong>Dimensionality:</strong>

<br><br>

The two categorical features generate
<strong>{generated_dummy_columns}</strong>
binary variables.

This expanded dimensionality should be considered before
PCA, t-SNE and GMM.

</div>


<div class="note">

<strong>Sparsity:</strong>

<br><br>

The temporary One-Hot matrix has a sparsity of
<strong>{matrix_sparsity:.12f}</strong>.

Most encoded values are zero.

This can materially affect Euclidean distance structure,
principal components and Gaussian covariance estimation.

</div>


<div class="note">

<strong>Mutually exclusive dummy variables:</strong>

<br><br>

Dummies generated from the same source feature are mutually
exclusive by construction.

A transaction cannot simultaneously be Monday and Tuesday,
and it cannot simultaneously belong to two different
receiver categories.

Therefore, relationships between same-group dummies should
not automatically be interpreted as unexpected redundancy.

</div>


<div class="note">

<strong>Unknown categories:</strong>

<br><br>

With handle_unknown="ignore", an unseen category generates
an all-zero vector inside its feature group.

This geometric behavior should be reviewed before applying
PCA, t-SNE or GMM to held-out or future observations.

</div>


<div class="note">

<strong>Target association:</strong>

<br><br>

Fraud-rate differences among categories do not mean that
TARGET_OMEGA should be used to construct the unsupervised
representation.

The target remains excluded from PCA, t-SNE and GMM.

These results are used for exploratory interpretation and
the final pre-modeling audit.

</div>


<div class="note">

<strong>Large sample size:</strong>

<br><br>

With approximately {analysis_observations:,} observations,
small categorical effects may still produce very small
p-values.

Cramer's V, category-level Phi, fraud-rate lift, Odds Ratio,
Risk Ratio and Risk Difference should therefore receive
greater interpretive emphasis than significance alone.

</div>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary
</h2>


<p class="result">

Categorical source features analyzed:
{len(OHEWI_FEATURES)}

</p>


<p class="result">

Temporary One-Hot dummy columns:
{generated_dummy_columns}

</p>


<p class="result">

Overall fraud rate:
{fraud_percentage:.6f}%

</p>


<p class="result">

Temporary OHE sparsity:
{matrix_sparsity:.12f}

</p>


<p class="result">

Strongest source association:
{strongest_source_feature}

<br>

Cramer's V:
{strongest_source_cramers_v:.6f}

</p>


<p class="result">

Strongest individual dummy:
{strongest_dummy_source}
=
{strongest_dummy_category}

<br>

Phi:
{strongest_dummy_phi:.6f}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The original categorical labels were preserved without
modification.

A temporary sparse One-Hot representation was generated
only to study the transformed feature structure.

<br><br>

TRANS_WEEK_OHEWI and RECEIVE_CATEGORY_OHEWI were evaluated
against fraud using contingency tables, fraud rates,
chi-square tests, Cramer's V, Adjusted Standardized
Residuals and Mutual Information.

<br><br>

Each category was also evaluated as its corresponding
temporary One-Hot dummy using Phi, Odds Ratio, Risk Ratio
and Risk Difference.

<br><br>

No source category or dummy is removed during this
exploratory stage.

Dimensionality, sparsity, category-specific effects,
mutual exclusivity and unknown-category behavior should all
be revisited during the final pre-modeling audit before
PCA, t-SNE and GMM.

</div>


</body>

</html>
"""


# ============================================================
# 33. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 34. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ONE-HOT ENCODING WITH IGNORE - TARGET ASSOCIATION"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 35. DISPLAY TARGET OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TARGET OVERVIEW"
)


print(
    "=" * 100
)


display(
    target_overview_table
)


# ============================================================
# 36. DISPLAY SOURCE FEATURE OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SOURCE FEATURE OVERVIEW"
)


print(
    "=" * 100
)


display(
    source_overview_table
)


# ============================================================
# 37. DISPLAY TEMPORARY OHE STRUCTURE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TEMPORARY ONE-HOT STRUCTURE"
)


print(
    "=" * 100
)


display(
    ohe_feature_structure_table
)


display(
    ohe_matrix_structure_table
)


# ============================================================
# 38. DISPLAY SOURCE-FEATURE ASSOCIATIONS
# ============================================================

for feature in OHEWI_FEATURES:

    result = (
        feature_results[
            feature
        ]
    )


    print(
        "\n"
        + "=" * 100
    )


    print(
        f"{feature} × {TARGET_FEATURE}"
    )


    print(
        "=" * 100
    )


    print(
        "\nContingency table:"
    )


    display(
        result[
            "contingency_table"
        ]
    )


    print(
        "\nFraud rate by category:"
    )


    display(
        result[
            "fraud_rate_table"
        ]
    )


    print(
        "\nChi-square:",
        f'{result["chi_square_statistic"]:.6f}'
    )


    print(
        "Degrees of freedom:",
        result[
            "degrees_of_freedom"
        ]
    )


    print(
        "p-value:",
        f'{result["chi_square_p_value"]:.12g}'
    )


    print(
        "Cramer's V:",
        f'{result["cramers_v"]:.6f}'
    )


    print(
        "Cramer's V strength:",
        result[
            "cramers_v_strength"
        ]
    )


    print(
        "Mutual Information:",
        f'{result["mutual_information"]:.8f}'
    )


    print(
        "Normalized Mutual Information:",
        f'{result["normalized_mutual_information"]:.8f}'
    )


    print(
        "\nAdjusted standardized residuals:"
    )


    display(
        result[
            "adjusted_residuals_table"
        ]
    )


# ============================================================
# 39. DISPLAY ALL DUMMY EFFECTS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "INDIVIDUAL ONE-HOT DUMMY EFFECTS"
)


print(
    "=" * 100
)


display(
    dummy_effect_ranking_table
)


# ============================================================
# 40. DISPLAY SOURCE-FEATURE COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SOURCE-FEATURE TARGET-ASSOCIATION COMPARISON"
)


print(
    "=" * 100
)


display(
    association_summary_table
)


print(
    "\nStrongest source-feature association:"
)


print(
    strongest_source_feature
)


print(
    "Cramer's V:",
    f"{strongest_source_cramers_v:.6f}"
)


print(
    "Strength:",
    strongest_source_strength
)


print(
    "\nStrongest individual dummy association:"
)


print(
    strongest_dummy_source,
    "=",
    strongest_dummy_category
)


print(
    "Phi:",
    f"{strongest_dummy_phi:.6f}"
)


print(
    "Strength:",
    strongest_dummy_phi_strength
)


# ============================================================
# 41. RELEASE MEMORY
# ============================================================

del dataset_ohewi_target

del temporary_ohe_matrix

del active_values_per_observation

gc.collect()


# ============================================================
# 42. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    WEEK_FRAUD_RATE_PATH
)


print(
    CATEGORY_FRAUD_RATE_PATH
)


print(
    WEEK_RESIDUAL_PATH
)


print(
    CATEGORY_RESIDUAL_PATH
)


print(
    DUMMY_PHI_PATH
)


ONE-HOT ENCODING WITH IGNORE - TARGET ASSOCIATION

Total dataset observations: 1852394
Complete observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

TARGET OVERVIEW


,TARGET_CLASS,TARGET_VALUE,COUNT,PERCENTAGE
0,Non-fraud,0,1842743,99.478999
1,Fraud,1,9651,0.521001



SOURCE FEATURE OVERVIEW


,FEATURE,SOURCE_DATA_TYPE,UNIQUE_CATEGORIES,MIN_CATEGORY_COUNT,MAX_CATEGORY_COUNT
0,TRANS_WEEK_OHEWI,category,7,183913,369418
1,RECEIVE_CATEGORY_OHEWI,category,14,57956,188029



TEMPORARY ONE-HOT STRUCTURE


...


INDIVIDUAL ONE-HOT DUMMY EFFECTS


,SOURCE_FEATURE,CATEGORY,TOTAL,FRAUD,FRAUD_RATE,FRAUD_PERCENTAGE,FRAUD_RATE_LIFT,PHI,PHI_STRENGTH,ODDS_RATIO_CATEGORY_VS_REST,OR_CI_95_LOWER,OR_CI_95_UPPER,RISK_RATIO_CATEGORY_VS_REST,RISK_DIFFERENCE_PERCENTAGE_POINTS,FRAUD_ADJUSTED_RESIDUAL,ZERO_CELL_CORRECTION_USED,ABS_PHI
0,RECEIVE_CATEGORY_OHEWI,shopping_net,139322,2219,0.015927,1.592713,3.057023,0.042454,Very weak or negligible,3.714429,3.541294,3.896028,3.671196,1.158873,57.780536,No,0.042454
1,RECEIVE_CATEGORY_OHEWI,grocery_pos,176191,2228,0.012645,1.264537,2.427127,0.033485,Very weak or negligible,2.879240,2.745388,3.019619,2.855477,0.821691,45.573424,No,0.033485
2,RECEIVE_CATEGORY_OHEWI,misc_net,90654,1182,0.013039,1.303859,2.502601,0.024667,Very weak or negligible,2.734937,2.572506,2.907623,2.712315,0.823141,33.572726,No,0.024667
3,RECEIVE_CATEGORY_OHEWI,home,175460,265,0.001510,0.151032,0.289887,-0.016623,Very weak or negligible,0.268734,0.237824,0.303660,0.269838,-0.408680,-22.624499,No,0.016623
4,RECEIVE_CATEGORY_OHEWI,kids_pets,161727,304,0.001880,0.187971,0.360788,-0.014307,Very weak or negligible,0.338755,0.302154,0.379790,0.339998,-0.364888,-19.472764,No,0.014307
5,RECEIVE_CATEGORY_OHEWI,food_dining,130729,205,0.001568,0.156813,0.300984,-0.013940,Very weak or negligible,0.284692,0.247875,0.326977,0.285813,-0.391842,-18.972275,No,0.013940
6,RECEIVE_CATEGORY_OHEWI,health_fitness,122553,185,0.001510,0.150955,0.289740,-0.013681,Very weak or negligible,0.274764,0.237530,0.317835,0.275859,-0.396263,-18.620723,No,0.013681
7,RECEIVE_CATEGORY_OHEWI,entertainment,134118,292,0.002177,0.217719,0.417885,-0.011770,Very weak or negligible,0.398413,0.354561,0.447689,0.399723,-0.326955,-16.018664,No,0.011770
8,RECEIVE_CATEGORY_OHEWI,personal_care,130085,290,0.002229,0.222931,0.427890,-0.011379,Very weak or negligible,0.408848,0.363705,0.459594,0.410166,-0.320583,-15.486679,No,0.011379
9,RECEIVE_CATEGORY_OHEWI,misc_pos,114229,322,0.002819,0.281890,0.541054,-0.008514,Very weak or negligible,0.523871,0.468712,0.585521,0.525213,-0.254826,-11.588436,No,0.008514



SOURCE-FEATURE TARGET-ASSOCIATION COMPARISON


,FEATURE,NUMBER_OF_CATEGORIES,CHI_SQUARE,DEGREES_OF_FREEDOM,P_VALUE,CRAMERS_V,CRAMERS_V_STRENGTH,MUTUAL_INFORMATION,NORMALIZED_MUTUAL_INFORMATION,FLAGGED_RESIDUAL_CELLS,ABS_CRAMERS_V
0,RECEIVE_CATEGORY_OHEWI,14,8329.139946,13,0.000000e+00,0.067055,Very weak or negligible,0.001919,0.001462,28,0.067055
1,TRANS_WEEK_OHEWI,7,290.758047,6,7.809807e-60,0.012529,Very weak or negligible,0.000079,0.000081,14,0.012529



Strongest source-feature association:
RECEIVE_CATEGORY_OHEWI
Cramer's V: 0.067055
Strength: Very weak or negligible

Strongest individual dummy association:
RECEIVE_CATEGORY_OHEWI = shopping_net
Phi: 0.042454
Strength: Very weak or negligible

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/onehot_encoding_with_ignore

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/onehot_encoding_with_ignore/analysis_onehot_encoding_with_ignore_target_association.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/onehot_encoding_with_ignore/ohewi_target_week_fraud_rate.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/02_relationships_with_target/target_association/onehot_encoding_with_ignore/ohewi_targ